In [ ]:

import argparse
import json
import operator
import random
import re
import traceback
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from lifelines import CoxPHFitter, CoxTimeVaryingFitter
from lifelines.utils import concordance_index
from lifelines.statistics import proportional_hazard_test

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler

from deap import base, creator, tools, gp
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)


# ============================================================
# Shared config and data preprocessing
# ============================================================

BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

# One shared train/external pair for both baseline and GP algorithm cells.
# Change these two paths here when switching datasets.
TRAIN_DATA_PATH = r"D:\桌面\所有数据\event_1_death_hospice_eicu.xlsx"
EXTERNAL_VALID_DATA_PATH = r"D:\桌面\所有数据\event_1_death_hospice_mimic.xlsx"

OUT_XLSX_BASELINE =r"D:\桌面\cox_baseline_4fold_catK.xlsx"
RESULT_JSON_BASELINE = r"D:\桌面\cox_baseline_4fold_catK_result.json"

OUT_XLSX_ALGO = r"D:\桌面\diephaseABC.xlsx"
OUT_PNG_TRAJ = r"D:\桌面\diePhaseABC_integrated_trajectories.png"
OUT_PNG_TRAJ_TEMPLATE = r"D:\桌面\diePhaseABC_outer{fold}_trajectory.png"
OUT_PNG_PHASEA_4FOLD = r"D:\桌面\diePhaseA_4fold_VALID_cindex_trajectory.png"

OUT_XLSX = OUT_XLSX_BASELINE
RESULT_JSON = RESULT_JSON_BASELINE

TIME_COL = "time"
EVENT_COL = "event"
MIN_TIME = 0.001
TOP_K = 10

SEED = 42
OUTER_FOLDS = 4

CATEGORICAL_CANDIDATES = ["gender", "ethnicity", "admission_type"]
ID_EXCLUDE_CANDIDATES = ["subject_id", "hadm_id", "stay_id", "subjectid", "hadmid", "stayid", "patientunitstayid"]

COX_PENALIZER = 0.01
COX_L1_RATIO = 0.0

MICE_MAX_ITER = 20
MICE_RANDOM_STATE = 42
MICE_INITIAL_STRATEGY = "median"
MICE_SKIP_COMPLETE = True
WINSOR_Q_LOW = 0.01
WINSOR_Q_HIGH = 0.99
STANDARDIZE_CONTINUOUS = True
STANDARDIZE_EPS = 1e-8

def _default_data_paths(base_dir: Path):
    xlsx_files = [
        p for p in base_dir.glob("*.xlsx")
        if p.name not in {Path(OUT_XLSX).name} and not p.name.startswith("~$")
    ]
    candidates = [p for p in xlsx_files if not p.stem.startswith("cox_baseline")]
    train_candidates = sorted(candidates, key=lambda p: (len(p.stem), p.name))
    external_candidates = sorted(candidates, key=lambda p: (len(p.stem), p.name), reverse=True)
    if not train_candidates or not external_candidates:
        raise FileNotFoundError("Could not identify training/external Excel files in BASE_DIR.")
    train_path = train_candidates[0]
    external_path = external_candidates[0]
    return str(train_path), str(external_path)


def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)


def make_cox():
    return CoxPHFitter(penalizer=float(COX_PENALIZER), l1_ratio=float(COX_L1_RATIO))


def _coerce_time_event(df):
    df = df.copy()
    df[TIME_COL] = pd.to_numeric(df[TIME_COL], errors="coerce").clip(lower=MIN_TIME)
    df[EVENT_COL] = pd.to_numeric(df[EVENT_COL], errors="coerce").fillna(0).astype(int)
    return df


def _is_id_col(c: str) -> bool:
    s = str(c).strip().lower()
    if s in {x.lower() for x in ID_EXCLUDE_CANDIDATES}:
        return True
    return bool(re.search(r"(^id$|^id_|_id$)", s))


def _is_binary_series(s: pd.Series) -> bool:
    vals = pd.to_numeric(s, errors="coerce").dropna().unique()
    if len(vals) == 0:
        return False
    return set(np.asarray(vals, dtype=float)).issubset({0.0, 1.0})


def _fit_mice_on_continuous(df_num: pd.DataFrame):
    if df_num.shape[1] == 0:
        return None, df_num.copy()
    imp = IterativeImputer(
        random_state=MICE_RANDOM_STATE,
        max_iter=MICE_MAX_ITER,
        initial_strategy=MICE_INITIAL_STRATEGY,
        skip_complete=MICE_SKIP_COMPLETE,
        sample_posterior=False,
    )
    arr = imp.fit_transform(df_num)
    return imp, pd.DataFrame(arr, columns=df_num.columns, index=df_num.index)


def _transform_mice_on_continuous(df_num: pd.DataFrame, imp):
    if df_num.shape[1] == 0:
        return df_num.copy()
    if imp is None:
        return df_num.fillna(0.0)
    arr = imp.transform(df_num)
    return pd.DataFrame(arr, columns=df_num.columns, index=df_num.index)


def _fit_winsor_zscore_on_continuous(df_cont: pd.DataFrame):
    stats = {}
    out = df_cont.copy()
    for c in df_cont.columns:
        s = pd.to_numeric(df_cont[c], errors="coerce").astype(float)
        if np.isfinite(s).any():
            q_low = float(np.nanquantile(s, WINSOR_Q_LOW))
            q_high = float(np.nanquantile(s, WINSOR_Q_HIGH))
        else:
            q_low, q_high = 0.0, 0.0
        if q_high < q_low:
            q_low, q_high = q_high, q_low
        sw = s.clip(lower=q_low, upper=q_high)
        if np.isfinite(sw).any():
            mu = float(np.nanmean(sw))
            sd = float(np.nanstd(sw, ddof=0))
        else:
            mu, sd = 0.0, 1.0
        if not np.isfinite(sd) or sd < STANDARDIZE_EPS:
            sd = 1.0
        out[c] = (sw - mu) / sd
        stats[c] = {"q_low": q_low, "q_high": q_high, "mean": mu, "sd": sd}
    return out, stats


def _transform_winsor_zscore_on_continuous(df_cont: pd.DataFrame, stats: dict):
    out = df_cont.copy()
    for c in df_cont.columns:
        s = pd.to_numeric(df_cont[c], errors="coerce").astype(float)
        st = stats.get(c)
        if st is None:
            out[c] = s
            continue
        sd = float(st.get("sd", 1.0))
        if not np.isfinite(sd) or sd < STANDARDIZE_EPS:
            sd = 1.0
        s = s.clip(lower=float(st.get("q_low", np.nan)), upper=float(st.get("q_high", np.nan)))
        out[c] = (s - float(st.get("mean", 0.0))) / sd
    return out


def _apply_cat_fill(df: pd.DataFrame, cat_cols):
    for c in cat_cols:
        df[c] = df[c].astype("object").fillna("UNKNOWN")
    return df


def fit_preprocess(df_raw, cat_candidates=None):
    if cat_candidates is None:
        cat_candidates = CATEGORICAL_CANDIDATES
    df = _coerce_time_event(df_raw).copy()
    cat_cols = [c for c in cat_candidates if c in df.columns]
    id_cols = [c for c in df.columns if _is_id_col(c)]
    base_drop = [TIME_COL, EVENT_COL] + cat_cols + id_cols
    num_cols_all = [c for c in df.columns if c not in base_drop]
    for c in num_cols_all:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    binary_num_cols = [c for c in num_cols_all if _is_binary_series(df[c])]
    cont_cols = [c for c in num_cols_all if c not in binary_num_cols]

    mice_fill_defaults = {}
    for c in cont_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        mice_fill_defaults[c] = float(s.median(skipna=True)) if s.notna().any() else 0.0

    imp, cont_imputed = _fit_mice_on_continuous(df[cont_cols]) if cont_cols else (None, pd.DataFrame(index=df.index))
    cont_standardize_stats = {}
    if cont_cols and STANDARDIZE_CONTINUOUS:
        cont_imputed, cont_standardize_stats = _fit_winsor_zscore_on_continuous(cont_imputed)
    for c in cont_cols:
        df[c] = cont_imputed[c]

    binary_fill = {}
    for c in binary_num_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        mode = s.dropna().mode()
        fill_v = float(mode.iloc[0]) if len(mode) else 0.0
        binary_fill[c] = fill_v
        df[c] = s.fillna(fill_v)

    df = _apply_cat_fill(df, cat_cols)
    df_model = df.drop(columns=id_cols, errors="ignore")
    df_enc = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

    model_cols = [c for c in df_enc.columns if c not in (TIME_COL, EVENT_COL)]
    categorical_dummy_cols = [c for c in model_cols if c not in cont_cols and c not in binary_num_cols]
    prep = {
        "cat_candidates": list(cat_candidates),
        "cat_cols": cat_cols,
        "id_cols": id_cols,
        "cont_cols": cont_cols,
        "binary_num_cols": binary_num_cols,
        "binary_fill": binary_fill,
        "mice_fill_defaults": mice_fill_defaults,
        "mice_imputer": imp,
        "cont_standardize_stats": cont_standardize_stats,
        "standardize_continuous": STANDARDIZE_CONTINUOUS,
        "winsor_q_low": WINSOR_Q_LOW,
        "winsor_q_high": WINSOR_Q_HIGH,
        "dummy_columns": model_cols,
        "categorical_dummy_cols": categorical_dummy_cols,
    }
    return df_enc[[TIME_COL, EVENT_COL] + model_cols], prep


def transform_preprocess(df_raw, prep):
    df = _coerce_time_event(df_raw).copy()
    cat_cols = [c for c in prep["cat_candidates"] if c in df.columns]
    id_cols = [c for c in df.columns if _is_id_col(c)]
    for c in prep["cont_cols"] + prep["binary_num_cols"]:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")

    if prep["cont_cols"]:
        cont_df = df[prep["cont_cols"]].copy()
        try:
            cont_imputed = _transform_mice_on_continuous(cont_df, prep["mice_imputer"])
        except Exception:
            cont_imputed = cont_df.copy()
            for c in prep["cont_cols"]:
                cont_imputed[c] = cont_imputed[c].fillna(prep["mice_fill_defaults"].get(c, 0.0))
        if prep.get("standardize_continuous", False):
            cont_imputed = _transform_winsor_zscore_on_continuous(cont_imputed, prep.get("cont_standardize_stats", {}))
        for c in prep["cont_cols"]:
            df[c] = cont_imputed[c]

    for c in prep["binary_num_cols"]:
        fill_v = prep["binary_fill"].get(c, 0.0)
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(fill_v)

    df = _apply_cat_fill(df, cat_cols)
    df_model = df.drop(columns=id_cols, errors="ignore")
    df_enc = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)
    for col in prep["dummy_columns"]:
        if col not in df_enc.columns:
            df_enc[col] = 0.0
    keep = set(prep["dummy_columns"]) | {TIME_COL, EVENT_COL}
    df_enc = df_enc.drop(columns=[c for c in df_enc.columns if c not in keep])
    return df_enc[[TIME_COL, EVENT_COL] + prep["dummy_columns"]]


# Load once for the downstream baseline and algorithm cells.
if not Path(TRAIN_DATA_PATH).exists() or not Path(EXTERNAL_VALID_DATA_PATH).exists():
    print("Manual TRAIN_DATA_PATH / EXTERNAL_VALID_DATA_PATH not found; falling back to auto-detection.")
    TRAIN_DATA_PATH, EXTERNAL_VALID_DATA_PATH = _default_data_paths(BASE_DIR)

print(f"TRAIN_DATA_PATH = {TRAIN_DATA_PATH}")
print(f"EXTERNAL_VALID_DATA_PATH = {EXTERNAL_VALID_DATA_PATH}")

df_train_raw = _coerce_time_event(pd.read_excel(TRAIN_DATA_PATH))
df_valid_raw = _coerce_time_event(pd.read_excel(EXTERNAL_VALID_DATA_PATH))
df_external_raw = df_valid_raw

train_enc_full, prep_full = fit_preprocess(df_train_raw)
valid_enc_full = transform_preprocess(df_valid_raw, prep_full)

print(f"Train n={len(df_train_raw)} events={int(df_train_raw[EVENT_COL].sum())}")
print(f"External valid n={len(df_valid_raw)} events={int(df_valid_raw[EVENT_COL].sum())}")
print(f"Encoded train shape={train_enc_full.shape}, encoded external shape={valid_enc_full.shape}")


In [ ]:

# ============================================================
# Baseline: Cox continuous + categorical correction
# ============================================================

OUT_XLSX = OUT_XLSX_BASELINE
RESULT_JSON = RESULT_JSON_BASELINE

K_CAT_MAX = 3
K_CAT_GRID = [0, 1, 2, 3]
PHASEC_MIN_SUPPORT_TRAIN = 8
PHASEC_MIN_SUPPORT_VALID = 5
PHASEC_MIN_EVENT_SUPPORT = 4

def _jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj) if np.isfinite(obj) else None
    if isinstance(obj, np.ndarray):
        return _jsonable(obj.tolist())
    if pd.isna(obj) if not isinstance(obj, (list, tuple, dict, np.ndarray)) else False:
        return None
    return obj


def _safe_cindex(T, E, risk):
    T = np.asarray(T, dtype=float)
    E = np.asarray(E, dtype=int)
    risk = np.asarray(risk, dtype=float)
    mask = np.isfinite(T) & np.isfinite(E) & np.isfinite(risk)
    if int(mask.sum()) < 3 or len(np.unique(E[mask])) < 2:
        return np.nan
    try:
        return float(concordance_index(T[mask], -risk[mask], E[mask]))
    except Exception:
        return np.nan


def _fit_cox(df, variables):
    cols = [c for c in variables if c in df.columns]
    if not cols:
        return None
    use = df[[TIME_COL, EVENT_COL] + cols].copy()
    for c in cols:
        use[c] = pd.to_numeric(use[c], errors="coerce").fillna(0.0).astype(float)
    try:
        cph = make_cox()
        cph.fit(use, duration_col=TIME_COL, event_col=EVENT_COL)
        return cph
    except Exception:
        return None


def _predict_risk(cph, df, variables):
    if cph is None:
        return np.full(len(df), np.nan, dtype=float)
    use = df[[TIME_COL, EVENT_COL] + list(variables)].copy()
    for c in variables:
        use[c] = pd.to_numeric(use[c], errors="coerce").fillna(0.0).astype(float)
    try:
        return cph.predict_partial_hazard(use).values.reshape(-1).astype(float)
    except Exception:
        return np.full(len(df), np.nan, dtype=float)


def _cox_coef_table(cph, model_name):
    if cph is None or getattr(cph, "summary", None) is None:
        return pd.DataFrame(columns=["model", "variable", "coefficient", "exp_coef", "se", "p"])
    s = cph.summary.reset_index().rename(columns={"covariate": "variable"})
    out = pd.DataFrame({
        "model": model_name,
        "variable": s["variable"],
        "coefficient": pd.to_numeric(s["coef"], errors="coerce"),
        "exp_coef": pd.to_numeric(s["exp(coef)"], errors="coerce"),
        "se": pd.to_numeric(s["se(coef)"], errors="coerce"),
        "p": pd.to_numeric(s["p"], errors="coerce"),
    })
    return out


def univariate_topk(df_enc, candidate_cols, top_k=TOP_K):
    rows = []
    for col in candidate_cols:
        if col not in df_enc.columns:
            continue
        try:
            cph = make_cox()
            cph.fit(df_enc[[col, TIME_COL, EVENT_COL]], duration_col=TIME_COL, event_col=EVENT_COL)
            s = cph.summary.iloc[0]
            rows.append({"variable": col, "p": float(s["p"])})
        except Exception:
            pass
    if not rows:
        return [], pd.DataFrame()
    uni_df = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
    return uni_df.head(top_k)["variable"].tolist(), uni_df


def _cat_pass_support(col, df_tr, df_va):
    x_tr = pd.to_numeric(df_tr[col], errors="coerce").fillna(0.0).values
    x_va = pd.to_numeric(df_va[col], errors="coerce").fillna(0.0).values
    e_tr = df_tr[EVENT_COL].astype(int).values
    n_train = int((x_tr > 0).sum())
    n_valid = int((x_va > 0).sum())
    n_event = int(((x_tr > 0) & (e_tr == 1)).sum())
    ok = (
        n_train >= int(PHASEC_MIN_SUPPORT_TRAIN)
        and n_valid >= int(PHASEC_MIN_SUPPORT_VALID)
        and n_event >= int(PHASEC_MIN_EVENT_SUPPORT)
    )
    return ok, {"n_train_positive": n_train, "n_valid_positive": n_valid, "n_train_event_positive": n_event}


def rank_categorical_dummies(df_tr, df_va, selected_cont_vars, candidate_cat_cols, base_train_cindex):
    rows = []
    for col in candidate_cat_cols:
        if col not in df_tr.columns or col not in df_va.columns:
            continue
        ok_support, support = _cat_pass_support(col, df_tr, df_va)
        if not ok_support:
            rows.append({"variable": col, "pass_support": 0, "gain": np.nan, "reason": "support_filter", **support})
            continue
        cph = _fit_cox(df_tr, selected_cont_vars + [col])
        if cph is None:
            rows.append({"variable": col, "pass_support": 1, "gain": np.nan, "reason": "cox_fit_failed", **support})
            continue
        risk_tr = _predict_risk(cph, df_tr, selected_cont_vars + [col])
        cidx_tr = _safe_cindex(df_tr[TIME_COL].values, df_tr[EVENT_COL].values, risk_tr)
        gain = cidx_tr - base_train_cindex if np.isfinite(cidx_tr) and np.isfinite(base_train_cindex) else np.nan
        rows.append({"variable": col, "pass_support": 1, "gain": gain, "reason": "", **support})
    df_rank = pd.DataFrame(rows)
    if len(df_rank) == 0:
        return [], df_rank
    df_ok = df_rank[(df_rank["pass_support"] == 1) & np.isfinite(pd.to_numeric(df_rank["gain"], errors="coerce"))].copy()
    df_ok = df_ok.sort_values(["gain", "variable"], ascending=[False, True]).reset_index(drop=True)
    return df_ok["variable"].tolist(), df_rank


def fit_fold_baseline(df_tr_raw, df_va_raw, fold, k_grid):
    print("\n" + "=" * 80)
    print(f"[FOLD {fold}/4] Cox baseline 4-fold internal validation")
    print("=" * 80)
    print(f"  [FOLD {fold}] Training samples = {len(df_tr_raw)}")
    print(f"  [FOLD {fold}] Validation samples = {len(df_va_raw)}")

    df_tr_enc, prep = fit_preprocess(df_tr_raw, cat_candidates=CATEGORICAL_CANDIDATES)
    df_va_enc = transform_preprocess(df_va_raw, prep)

    cont_candidates = [c for c in prep.get("cont_cols", []) if c in df_tr_enc.columns]
    cat_candidates = [c for c in prep.get("categorical_dummy_cols", []) if c in df_tr_enc.columns]

    selected_cont, uni_df = univariate_topk(df_tr_enc, cont_candidates, top_k=TOP_K)
    print(f"  [FOLD {fold}] Selected continuous variables: {len(selected_cont)}")
    print(f"  [FOLD {fold}] Top selected vars first 10: {selected_cont[:10]}")

    cox = _fit_cox(df_tr_enc, selected_cont)
    risk_tr = _predict_risk(cox, df_tr_enc, selected_cont)
    risk_va = _predict_risk(cox, df_va_enc, selected_cont)
    cox_train_cindex = _safe_cindex(df_tr_enc[TIME_COL].values, df_tr_enc[EVENT_COL].values, risk_tr)
    cox_valid_cindex = _safe_cindex(df_va_enc[TIME_COL].values, df_va_enc[EVENT_COL].values, risk_va)
    print(f"  [FOLD {fold}] Cox-continuous FINAL VALID C-index = {cox_valid_cindex:.4f}")

    ranked_cats, cat_rank_df = rank_categorical_dummies(
        df_tr_enc, df_va_enc, selected_cont, cat_candidates, cox_train_cindex
    )
    n_candidate = int(len(cat_candidates))
    n_pass_support = int(cat_rank_df["pass_support"].sum()) if len(cat_rank_df) else 0

    fold_results = {}
    for k in k_grid:
        k = int(k)
        selected_cat = [] if k == 0 else ranked_cats[:k]
        variables = selected_cont + selected_cat
        coxcat = cox if k == 0 else _fit_cox(df_tr_enc, variables)
        risk_cat_va = risk_va.copy() if k == 0 else _predict_risk(coxcat, df_va_enc, variables)
        coxcat_cindex = cox_valid_cindex if k == 0 else _safe_cindex(
            df_va_enc[TIME_COL].values, df_va_enc[EVENT_COL].values, risk_cat_va
        )
        fold_results[k] = {
            "fold": int(fold),
            "cox": cox,
            "coxcat": coxcat,
            "cox_cindex": float(cox_valid_cindex) if np.isfinite(cox_valid_cindex) else np.nan,
            "coxcat_cindex": float(coxcat_cindex) if np.isfinite(coxcat_cindex) else np.nan,
            "delta": float(coxcat_cindex - cox_valid_cindex) if np.isfinite(cox_valid_cindex) and np.isfinite(coxcat_cindex) else np.nan,
            "risk_cox_valid": risk_va,
            "risk_coxcat_valid": risk_cat_va,
            "selected_continuous_vars": selected_cont,
            "selected_cat_vars": selected_cat,
            "n_candidate_cats": n_candidate if k > 0 else 0,
            "n_pass_support_cats": n_pass_support if k > 0 else 0,
            "selected_cat_n": len(selected_cat),
            "cat_rank_df": cat_rank_df,
            "prep": prep,
            "df_tr_enc": df_tr_enc,
            "df_va_enc": df_va_enc,
        }
    return fold_results


def run_internal_4fold_select_catk(df_train_raw, k_grid=(0, 1, 2, 3), n_splits=4):
    df_train_raw = _coerce_time_event(df_train_raw).reset_index(drop=True)
    y = df_train_raw[EVENT_COL].values
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    k_grid = tuple(sorted(set(int(k) for k in k_grid if 0 <= int(k) <= int(K_CAT_MAX))))
    if len(k_grid) == 0:
        k_grid = (0,)

    fold_cache = []
    oof_cox_risk = np.full(len(df_train_raw), np.nan, dtype=float)
    oof_coxcat_risk_by_k = {k: np.full(len(df_train_raw), np.nan, dtype=float) for k in k_grid}

    for fold, (tr_idx, va_idx) in enumerate(cv.split(np.arange(len(y)), y), 1):
        df_tr = df_train_raw.iloc[tr_idx].reset_index(drop=True)
        df_va = df_train_raw.iloc[va_idx].reset_index(drop=True)
        fold_results = fit_fold_baseline(df_tr, df_va, fold, k_grid)
        fold_cache.append({"fold": fold, "tr_idx": tr_idx, "va_idx": va_idx, "results": fold_results})
        if 0 in fold_results:
            oof_cox_risk[va_idx] = np.asarray(fold_results[0]["risk_cox_valid"], dtype=float)
        for k in k_grid:
            if k in fold_results:
                oof_coxcat_risk_by_k[k][va_idx] = np.asarray(fold_results[k]["risk_coxcat_valid"], dtype=float)

    print("\n" + "-" * 80)
    print("Categorical correction K-search based on internal 4-fold validation")
    print("-" * 80)

    search_rows = []
    for k in k_grid:
        print(f"\n[CatKSearch] Start K={k}")
        for fc in fold_cache:
            r = fc["results"][int(k)]
            selected_cats = r["selected_cat_vars"]
            dlt = r["delta"]
            print(
                f"[CatKSearch] fold={fc['fold']} K={k} | "
                f"n_candidate={r['n_candidate_cats']} | n_pass_support={r['n_pass_support_cats']} | "
                f"selected_cat_n={r['selected_cat_n']} | selected_cats={selected_cats} | "
                f"Cox={r['cox_cindex']:.4f} | Cox+C={r['coxcat_cindex']:.4f} | delta={dlt:.4f}"
            )
            search_rows.append({
                "K_cat": int(k),
                "fold": int(fc["fold"]),
                "cox_cindex": r["cox_cindex"],
                "coxcat_cindex": r["coxcat_cindex"],
                "delta": dlt,
                "n_candidate_cats": r["n_candidate_cats"],
                "n_pass_support_cats": r["n_pass_support_cats"],
                "selected_cat_n": r["selected_cat_n"],
                "selected_cat_names": json.dumps(selected_cats, ensure_ascii=False),
            })

    df_search = pd.DataFrame(search_rows)
    summary_rows = []
    for k in k_grid:
        dfk = df_search[df_search["K_cat"] == int(k)].copy()
        cox_vals = pd.to_numeric(dfk["cox_cindex"], errors="coerce").values
        coxcat_vals = pd.to_numeric(dfk["coxcat_cindex"], errors="coerce").values
        dlt_vals = pd.to_numeric(dfk["delta"], errors="coerce").values
        selected_vals = pd.to_numeric(dfk["selected_cat_n"], errors="coerce").values
        mean_cox = float(np.nanmean(cox_vals)) if np.isfinite(cox_vals).any() else np.nan
        mean_coxcat = float(np.nanmean(coxcat_vals)) if np.isfinite(coxcat_vals).any() else np.nan
        mean_delta = float(np.nanmean(dlt_vals)) if np.isfinite(dlt_vals).any() else np.nan
        sd_coxcat = float(np.nanstd(coxcat_vals, ddof=0)) if np.isfinite(coxcat_vals).any() else np.nan
        positive_folds = int(np.sum(dlt_vals > 0)) if len(dlt_vals) else 0
        mean_selected = float(np.nanmean(selected_vals)) if np.isfinite(selected_vals).any() else 0.0
        eligible = bool(int(k) > 0 and np.isfinite(mean_delta) and mean_delta > float(PHASEC_MEAN_DELTA_MIN) and positive_folds >= int(PHASEC_POSITIVE_FOLDS_MIN))
        summary_rows.append({
            "K_cat": int(k),
            "mean_cox": mean_cox,
            "mean_coxcat": mean_coxcat,
            "mean_delta": mean_delta,
            "sd_coxcat": sd_coxcat,
            "positive_folds": positive_folds,
            "mean_selected_cat_n": mean_selected,
            "eligible": int(eligible),
        })
    df_summary = pd.DataFrame(summary_rows).sort_values("K_cat").reset_index(drop=True)
    eligible_df = df_summary[df_summary["eligible"] == 1].copy()
    selected_k = 0 if len(eligible_df) == 0 else int(
        eligible_df.sort_values(["mean_coxcat", "mean_delta", "K_cat"], ascending=[False, False, True]).iloc[0]["K_cat"]
    )
    df_summary["selected"] = (df_summary["K_cat"] == selected_k).astype(int)
    for _, r in df_summary.iterrows():
        print(
            f"[CatKSearch] Summary K={int(r['K_cat'])} | "
            f"mean_cox={float(r['mean_cox']):.4f} | mean_coxcat={float(r['mean_coxcat']):.4f} | "
            f"mean_delta={float(r['mean_delta']):.4f} | sd_coxcat={float(r['sd_coxcat']):.4f} | "
            f"positive_folds={int(r['positive_folds'])} | mean_selected_cat_n={float(r['mean_selected_cat_n']):.2f} | "
            f"eligible={int(r['eligible'])} | selected={int(r['selected'])}"
        )
    print(f"[CatKSearch] Selected K = {selected_k}")

    oof_coxcat_risk = oof_coxcat_risk_by_k.get(selected_k, oof_cox_risk.copy())
    if int(selected_k) == 0:
        oof_coxcat_risk = oof_cox_risk.copy()

    T_all = df_train_raw[TIME_COL].astype(float).values
    E_all = df_train_raw[EVENT_COL].astype(int).values
    cox_oof = _safe_cindex(T_all, E_all, oof_cox_risk)
    coxcat_oof = _safe_cindex(T_all, E_all, oof_coxcat_risk)

    cox_fold = []
    coxcat_fold = []
    fold_summary_rows = []
    for fc in fold_cache:
        r0 = fc["results"][0]
        rk = fc["results"][selected_k]
        cox_fold.append(r0["cox_cindex"])
        coxcat_fold.append(rk["coxcat_cindex"])
        print(f"  [FOLD {fc['fold']}] Selected categorical correction variables for K={selected_k}: {rk['selected_cat_vars']}")
        print(f"  [FOLD {fc['fold']}] Cox+C FINAL VALID C-index = {rk['coxcat_cindex']:.4f}")
        fold_summary_rows.append({
            "fold": int(fc["fold"]),
            "cox_cindex": r0["cox_cindex"],
            "coxcat_cindex": rk["coxcat_cindex"],
            "delta": rk["delta"],
            "selected_k": int(selected_k),
            "selected_continuous_vars": json.dumps(rk["selected_continuous_vars"], ensure_ascii=False),
            "selected_cat_vars": json.dumps(rk["selected_cat_vars"], ensure_ascii=False),
        })

    cox_mean = float(np.nanmean(cox_fold)) if np.isfinite(np.asarray(cox_fold, dtype=float)).any() else np.nan
    coxcat_mean = float(np.nanmean(coxcat_fold)) if np.isfinite(np.asarray(coxcat_fold, dtype=float)).any() else np.nan
    return {
        "selected_k": int(selected_k),
        "fold_summary": pd.DataFrame(fold_summary_rows),
        "catk_search": df_search,
        "catk_summary": df_summary,
        "oof_cox_risk": oof_cox_risk,
        "oof_coxcat_risk": oof_coxcat_risk,
        "cox_fold_cindexes": [float(x) if np.isfinite(x) else np.nan for x in cox_fold],
        "coxcat_fold_cindexes": [float(x) if np.isfinite(x) else np.nan for x in coxcat_fold],
        "cox_mean_cindex": cox_mean,
        "coxcat_mean_cindex": coxcat_mean,
        "cox_oof_cindex": cox_oof,
        "coxcat_oof_cindex": coxcat_oof,
        "fold_cache": fold_cache,
    }


def fit_final_on_full_train_and_external(df_train_raw, df_external_raw, selected_k):
    df_tr_enc, prep = fit_preprocess(df_train_raw, cat_candidates=CATEGORICAL_CANDIDATES)
    df_ext_enc = transform_preprocess(df_external_raw, prep)
    cont_candidates = [c for c in prep.get("cont_cols", []) if c in df_tr_enc.columns]
    cat_candidates = [c for c in prep.get("categorical_dummy_cols", []) if c in df_tr_enc.columns]
    selected_cont, uni_df = univariate_topk(df_tr_enc, cont_candidates, top_k=TOP_K)

    cox = _fit_cox(df_tr_enc, selected_cont)
    risk_cox_ext = _predict_risk(cox, df_ext_enc, selected_cont)
    external_cox_cindex = _safe_cindex(df_ext_enc[TIME_COL].values, df_ext_enc[EVENT_COL].values, risk_cox_ext)

    risk_cox_tr = _predict_risk(cox, df_tr_enc, selected_cont)
    cox_train_cindex = _safe_cindex(df_tr_enc[TIME_COL].values, df_tr_enc[EVENT_COL].values, risk_cox_tr)
    ranked_cats, cat_rank_df = rank_categorical_dummies(df_tr_enc, df_ext_enc, selected_cont, cat_candidates, cox_train_cindex)
    selected_cat = [] if int(selected_k) == 0 else ranked_cats[:int(selected_k)]
    coxcat_vars = selected_cont + selected_cat
    coxcat = cox if int(selected_k) == 0 else _fit_cox(df_tr_enc, coxcat_vars)
    risk_coxcat_ext = risk_cox_ext.copy() if int(selected_k) == 0 else _predict_risk(coxcat, df_ext_enc, coxcat_vars)
    external_coxcat_cindex = external_cox_cindex if int(selected_k) == 0 else _safe_cindex(
        df_ext_enc[TIME_COL].values, df_ext_enc[EVENT_COL].values, risk_coxcat_ext
    )
    return {
        "selected_continuous_vars": selected_cont,
        "selected_categorical_vars": selected_cat,
        "cox": cox,
        "coxcat": coxcat,
        "external_cox_cindex": external_cox_cindex,
        "external_coxcat_cindex": external_coxcat_cindex,
        "external_delta": external_coxcat_cindex - external_cox_cindex if np.isfinite(external_cox_cindex) and np.isfinite(external_coxcat_cindex) else np.nan,
        "external_cox_risk": risk_cox_ext,
        "external_coxcat_risk": risk_coxcat_ext,
        "cat_rank_df": cat_rank_df,
        "prep": prep,
    }


def run_once(overrides=None):
    if overrides:
        globals().update(overrides)
    set_all_seeds(SEED)

    print("=" * 80)
    print("Load dataset")
    print("=" * 80)
    df_train_raw_local = globals().get("df_train_raw")
    df_external_raw_local = globals().get("df_external_raw", globals().get("df_valid_raw"))
    if df_train_raw_local is None or df_external_raw_local is None:
        df_train_raw_local = _coerce_time_event(pd.read_excel(TRAIN_DATA_PATH))
        df_external_raw_local = _coerce_time_event(pd.read_excel(EXTERNAL_VALID_DATA_PATH))
    df_train_raw = df_train_raw_local.copy()
    df_external_raw = df_external_raw_local.copy()
    print(f"Train n={len(df_train_raw)} events={int(df_train_raw[EVENT_COL].sum())}")
    print(f"External valid n={len(df_external_raw)} events={int(df_external_raw[EVENT_COL].sum())}")

    cv_res = run_internal_4fold_select_catk(df_train_raw, k_grid=K_CAT_GRID, n_splits=OUTER_FOLDS)
    selected_k = int(cv_res["selected_k"])
    final_res = fit_final_on_full_train_and_external(df_train_raw, df_external_raw, selected_k)

    cox_fold = cv_res["cox_fold_cindexes"]
    coxcat_fold = cv_res["coxcat_fold_cindexes"]
    cox_mean = float(cv_res["cox_mean_cindex"])
    coxcat_mean = float(cv_res["coxcat_mean_cindex"])
    cox_oof = float(cv_res["cox_oof_cindex"])
    coxcat_oof = float(cv_res["coxcat_oof_cindex"])
    mean_delta = coxcat_mean - cox_mean if np.isfinite(cox_mean) and np.isfinite(coxcat_mean) else np.nan
    oof_delta = coxcat_oof - cox_oof if np.isfinite(cox_oof) and np.isfinite(coxcat_oof) else np.nan

    print("\n" + "=" * 80)
    print("Internal 4-fold Cox-continuous baseline")
    print(f"Cox fold C-indexes = {cox_fold}")
    print(f"Cox mean C-index   = {cox_mean:.4f}")
    print(f"Cox OOF C-index    = {cox_oof:.4f}")
    print("=" * 80)

    print("\n" + "=" * 80)
    print("Internal 4-fold Cox + categorical correction baseline")
    print(f"Selected categorical K = {selected_k}")
    print(f"Cox+C fold C-indexes = {coxcat_fold}")
    print(f"Cox+C mean C-index   = {coxcat_mean:.4f}")
    print(f"Cox+C OOF C-index    = {coxcat_oof:.4f}")
    print(f"Cox+C vs Cox mean delta = {mean_delta:.4f}")
    print(f"Cox+C vs Cox OOF delta  = {oof_delta:.4f}")
    print("=" * 80)

    external_cox = float(final_res["external_cox_cindex"])
    external_coxcat = float(final_res["external_coxcat_cindex"])
    external_delta = float(final_res["external_delta"]) if np.isfinite(final_res["external_delta"]) else np.nan
    print("\n" + "=" * 80)
    print("External validation after fitting final baseline models on full training set")
    print(f"External Cox C-index   = {external_cox:.4f}")
    print(f"External Cox+C C-index = {external_coxcat:.4f}")
    print(f"External delta         = {external_delta:.4f}")
    print("=" * 80)

    df_coefs = pd.concat([
        _cox_coef_table(final_res["cox"], "Cox"),
        _cox_coef_table(final_res["coxcat"], "Cox+C"),
    ], axis=0, ignore_index=True)
    final_vars = []
    coxcat_coef_map = {}
    if final_res["coxcat"] is not None:
        coxcat_coef_map = {str(k): float(v) for k, v in final_res["coxcat"].params_.to_dict().items()}
    for v in final_res["selected_continuous_vars"]:
        final_vars.append({
            "variable": v,
            "variable_type": "continuous",
            "selected_in_final_model": 1,
            "coefficient": coxcat_coef_map.get(v, np.nan),
        })
    for v in final_res["selected_categorical_vars"]:
        final_vars.append({
            "variable": v,
            "variable_type": "categorical_dummy",
            "selected_in_final_model": 1,
            "coefficient": coxcat_coef_map.get(v, np.nan),
        })
    df_final_vars = pd.DataFrame(final_vars, columns=["variable", "variable_type", "selected_in_final_model", "coefficient"])

    df_internal_oof = pd.DataFrame({
        "row_id": np.arange(len(df_train_raw)),
        TIME_COL: df_train_raw[TIME_COL].astype(float).values,
        EVENT_COL: df_train_raw[EVENT_COL].astype(int).values,
        "oof_cox_risk": cv_res["oof_cox_risk"],
        "oof_coxcat_risk": cv_res["oof_coxcat_risk"],
    })
    df_external_risk = pd.DataFrame({
        "row_id": np.arange(len(df_external_raw)),
        TIME_COL: df_external_raw[TIME_COL].astype(float).values,
        EVENT_COL: df_external_raw[EVENT_COL].astype(int).values,
        "external_cox_risk": final_res["external_cox_risk"],
        "external_coxcat_risk": final_res["external_coxcat_risk"],
    })
    df_external_summary = pd.DataFrame([{
        "external_cox_cindex": external_cox,
        "external_coxcat_cindex": external_coxcat,
        "external_delta": external_delta,
        "selected_k_cat": selected_k,
        "selected_continuous_vars": json.dumps(final_res["selected_continuous_vars"], ensure_ascii=False),
        "selected_cat_vars": json.dumps(final_res["selected_categorical_vars"], ensure_ascii=False),
    }])

    Path(OUT_XLSX).parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
        cv_res["fold_summary"].to_excel(writer, sheet_name="Fold_Summary", index=False)
        cv_res["catk_search"].to_excel(writer, sheet_name="CatK_Search", index=False)
        cv_res["catk_summary"].to_excel(writer, sheet_name="CatK_Summary", index=False)
        df_final_vars.to_excel(writer, sheet_name="Final_Selected_Variables", index=False)
        df_coefs.to_excel(writer, sheet_name="Cox_Coefficients", index=False)
        df_internal_oof.to_excel(writer, sheet_name="Internal4Fold_OOF_Risk", index=False)
        df_external_summary.to_excel(writer, sheet_name="External_Validation", index=False)
        df_external_risk.to_excel(writer, sheet_name="External_Validation_Risk", index=False)

    cox_coef = {} if final_res["cox"] is None else {str(k): float(v) for k, v in final_res["cox"].params_.to_dict().items()}
    coxcat_coef = {} if final_res["coxcat"] is None else {str(k): float(v) for k, v in final_res["coxcat"].params_.to_dict().items()}
    result = {
        "baseline_internal_cv_summary": {
            "cox_fold_cindexes": cox_fold,
            "cox_mean_cindex": cox_mean,
            "cox_oof_cindex": cox_oof,
            "coxcat_fold_cindexes": coxcat_fold,
            "coxcat_mean_cindex": coxcat_mean,
            "coxcat_oof_cindex": coxcat_oof,
            "coxcat_vs_cox_mean_delta": mean_delta,
            "coxcat_vs_cox_oof_delta": oof_delta,
            "selected_k_cat": selected_k,
        },
        "baseline_external_summary": {
            "external_cox_cindex": external_cox,
            "external_coxcat_cindex": external_coxcat,
            "external_delta": external_delta,
        },
        "final_model_diag": {
            "selected_continuous_vars": final_res["selected_continuous_vars"],
            "selected_categorical_vars": final_res["selected_categorical_vars"],
            "cox_coefficients": cox_coef,
            "coxcat_coefficients": coxcat_coef,
            "preprocessing": "same as original GP pipeline",
        },
        "catk_summary": cv_res["catk_summary"].to_dict(orient="records"),
    }
    payload = {"status": "ok", "error": "", "result": result}
    Path(RESULT_JSON).write_text(json.dumps(_jsonable(payload), ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved Excel: {OUT_XLSX}")
    print(f"Saved JSON:  {RESULT_JSON}")
    return result


baseline_result = run_once()
baseline_result


In [ ]:
class SurvivalAwareKFold:
    """Use stratification when possible and ordinary shuffled K-fold otherwise."""

    def __init__(self, n_splits=4, shuffle=True, random_state=None):
        self.n_splits = int(n_splits)
        self.shuffle = bool(shuffle)
        self.random_state = random_state

    def split(self, X, y=None, groups=None):
        from sklearn.model_selection import KFold, StratifiedKFold
        y_array = np.asarray(y) if y is not None else np.asarray([])
        if y_array.size and len(np.unique(y_array)) >= 2:
            splitter = StratifiedKFold(
                n_splits=self.n_splits,
                shuffle=self.shuffle,
                random_state=self.random_state,
            )
            return splitter.split(X, y_array, groups)
        splitter = KFold(
            n_splits=self.n_splits,
            shuffle=self.shuffle,
            random_state=self.random_state,
        )
        return splitter.split(X, y_array if y_array.size else None, groups)



# ============================================================
# New Cox + GP-AB Phase-ABC algorithm experiment
# ============================================================


# ============================================================
# GP algorithm config
# ============================================================

OUT_XLSX = str(BASE_DIR / "newcox_gpab.xlsx")
RESULT_JSON = str(BASE_DIR / "newcox_gpab_result.json")

TOP_K = 10
N_TREES = 3

FEATURE_SET_MODE = "auto"
BASELINE_STABLE_MIN_STRICT_N = 4
BASELINE_STABLE_VARS = []
BASELINE_STABLE_VAR_FREQUENCY = []
BASELINE_STABLE_MIN_FREQ_USED = OUTER_FOLDS
FINAL_FEATURE_SET_MODE = None
FEATURE_SET_DECISION = ""
USE_MAIN_EFFECTS_FOR_THIS_RUN = None
MAIN_EFFECT_VARS = []
GP_INPUT_VARS = []
REFERENCE_TOP10_VARS = []
REFERENCE_TOP10_SOURCE = "full_train_cell2_style_top10"

GP_PARAM_PROFILE = "auto"
FINAL_FULL_TRAIN_N_GEN_X = 30
MAIN_EFFECT_GP_PARAMS = {
    "N_TREES": 3,
    "MAX_TREE_HEIGHT_X": 2,
    "N_GEN_X": 8,
    "POP_SIZE_X": 24,
    "COMPLEXITY_LAMBDA_X": 0.006,
    "DUPLICATE_TERMINAL_LAMBDA_X": 0.002,
}
GP_ONLY_PARAMS = {
    "N_TREES": 5,
    "MAX_TREE_HEIGHT_X": 3,
    "N_GEN_X": 10,
    "POP_SIZE_X": 32,
    "COMPLEXITY_LAMBDA_X": 0.003,
    "DUPLICATE_TERMINAL_LAMBDA_X": 0.001,
}
ACTUAL_GP_PARAM_PROFILE = None
ACTUAL_GP_PARAMS = {}

FINAL_PHASEB_GATE = True
INNER_FOLDS = 3
FINAL_HPO_FOLDS = 3
UNKNOWN_TOKENS = {"unknown", "unk", "na", "nan", "none", ""}

POP_SIZE_X = 32
N_GEN_X = 10
CX_PB_X = 0.70
MUT_PB_X = 0.10
TOURN_SIZE = 5
ELITE_K = 6
MAX_TREE_HEIGHT_X = 3
FIT_FOLDS_X = 3

POP_SIZE_T = 24
N_GEN_T = 8
MAX_TREE_HEIGHT_T = 3
FIT_FOLDS_T = 3
CX_PB_T = 0.70
MUT_PB_T = 0.10

PH_P_THRESHOLD = 0.05
PHASEB_MIN_DELTA = 0.005
PHASEB_MODE = "time_basis_pool"  # options: original, time_basis_pool, off
PHASEB_USE_PH_VIOLATING_MAIN_EFFECTS = True
PHASEB_USE_PH_VIOLATING_GP_FEATURES = True
PHASEB_ENABLE_EXPLORATORY_STABLE_MAIN = False
PHASEB_ENABLE_EXPLORATORY_ALL_GP = False
PHASEB_TIME_BASIS_NAMES = [
    "zlog", "zlog2", "zlog3",
    "early3", "early7", "late7",
    "hinge_after3", "hinge_after7",
    "exp_decay3", "exp_decay7",
    "inv_decay3", "inv_decay7",
    "t_scaled", "t_scaled2", "t_scaled3",
]
PHASEB_BASIS_PRIORITY = {
    "zlog": 1, "zlog2": 1, "zlog3": 1,
    "early3": 2, "early7": 2, "late7": 2,
    "hinge_after3": 2, "hinge_after7": 2,
    "exp_decay3": 2, "exp_decay7": 2,
    "inv_decay3": 2, "inv_decay7": 2,
    "t_scaled": 3, "t_scaled2": 3, "t_scaled3": 3,
}
PHASEB_MAX_BASIS_PER_FEATURE = 1
PHASEB_INTERVAL_CUTS = [3.0, 7.0, 14.0]
PHASEB_T_WINSOR_Q = 0.95
PHASEB_PENALIZER = 0.05
PHASEB_GATE_MIN_MEAN_DELTA = 0.0
PHASEB_GATE_MIN_POSITIVE_FOLDS = 2
# Profile-style Phase-B selection aliases. Defaults preserve the legacy gate:
# mean_delta > 0 and positive_folds >= 2.
PHASEB_MEAN_DELTA_MIN = PHASEB_GATE_MIN_MEAN_DELTA
PHASEB_POSITIVE_FOLDS_MIN = PHASEB_GATE_MIN_POSITIVE_FOLDS
PHASEB_COMPLEXITY_LAMBDA = 0.0
TIMEBASIS_COMPLEXITY_LAMBDA = 0.0
PHASEB_SELECTION_MODE = "legacy_mean_delta_positive_folds"
PHASEB_MIN_COL_STD = 1e-8
PHASEB_TIE_LOGLIK_EPS = 1e-4
PHASEB_TIE_CINDEX_EPS = 0.002
COMPLEXITY_LAMBDA_X = 0.003
COMPLEXITY_LAMBDA_T = 0.003
DUPLICATE_TERMINAL_LAMBDA_X = 0.001
GP_MIN_FEATURE_STD = 1e-6
ENABLE_GP_POST_FILTER = True
GP_CORR_FILTER_THRESHOLD = 0.90
GP_MIN_ACTIVE_FEATURES = 2

K_INTERACTION_MAX = 3
K_INTERACTION = 0
POP_SIZE_C = 24
N_GEN_C = 8
CX_PB_C = 0.70
MUT_PB_C = 0.20
MAX_TREE_HEIGHT_C = 3
TOURN_SIZE_C = 5
ELITE_K_C = 6
COMPLEXITY_LAMBDA_C = 0.003
PHASEC_MIN_SUPPORT_TRAIN = 8
PHASEC_MIN_SUPPORT_VALID = 5
PHASEC_MIN_EVENT_SUPPORT = 4
PHASEC_PENALIZER = 0.05
PHASEC_L1_RATIO = 0.5
PHASEC_MODE = "stgp_and"  # options: original, stgp_and, off
PHASEC_CAT_PREFIXES = ["gender_", "admission_type_", "ethnicity_"]
PHASEC_EXCLUDE_KEYWORDS = ["UNKNOWN", "Unknown", "unknown", "Other/Unknown", "nan", "NaN"]
PHASEC_BINARY_COLS = [
    "Mechanical.ventilation",
    "glucocorticoid",
    "Morphine.Sulfate",
    "cefazolin",
    "Kcl",
    "Dopamine",
    "Dobutamine",
    "Epinephrine",
    "Norepinephrine",
    "Paralysis",
]
PHASEC_MIN_MAIN_N = 20
PHASEC_MIN_MAIN_EVENTS = 4
PHASEC_MIN_INTER_N = 10
PHASEC_MIN_INTER_EVENTS = 3
PHASEC_POP_SIZE = 64
PHASEC_N_GEN = 20
PHASEC_MAX_TREE_HEIGHT = 2
PHASEC_K_PROFILE = "legacy_k_0_1_2_3"
PHASEC_K_VALUES = [0, 1, 2, 3]
PHASEC_MEAN_DELTA_MIN = 0.0
PHASEC_POSITIVE_FOLDS_MIN = 2
PHASEC_SELECTION_MODE = "legacy_mean_delta_positive_folds"
PHASEC_SELECTION_COMPLEXITY_LAMBDA = 0.0
# Existing algorithmic STGP fitness complexity; preserved to avoid changing model logic.
PHASEC_COMPLEXITY_LAMBDA = 0.005
PHASEC_MAX_TERMS_PER_INDIVIDUAL = 1
PHASEC_MAX_INTERACTION_ORDER = 2
PHASEC_ELITE_SIZE = 3
PHASEC_HOF_SIZE = 20
PHASEC_CROSSOVER_RATE = 0.5
PHASEC_MUTATION_RATE = 0.3
PHASEC_RANDOM_INJECTION_RATE = 0.1
PHASEC_STGP_USE_AND_ONLY = True
PHASEC_STGP_ALLOW_OR = False
PHASEC_STGP_ALLOW_NOT = False
PHASEC_STGP_ALLOW_CAT_CAT_AND = False
PHASEC_STGP_MAX_TERMINALS_PER_RULE = 2
PHASEC_STGP_HEREDITY = True


VERBOSE_TRIAL_LOG = True
PRINT_MODEL_SUMMARY_TO_TERMINAL = False
PRINT_GEN_LOG = True

OVERRIDABLE_KEYS = {
    "TRAIN_DATA_PATH",
    "EXTERNAL_VALID_DATA_PATH",
    "OUT_XLSX",
    "OUT_PNG_TRAJ",
    "OUT_PNG_TRAJ_TEMPLATE",
    "OUT_PNG_PHASEA_4FOLD",
    "RESULT_JSON",
    "TOP_K",
    "FEATURE_SET_MODE",
    "BASELINE_STABLE_MIN_STRICT_N",
    "BASELINE_STABLE_VARS",
    "BASELINE_STABLE_VAR_FREQUENCY",
    "BASELINE_STABLE_MIN_FREQ_USED",
    "FINAL_FEATURE_SET_MODE",
    "FEATURE_SET_DECISION",
    "USE_MAIN_EFFECTS_FOR_THIS_RUN",
    "MAIN_EFFECT_VARS",
    "GP_INPUT_VARS",
    "REFERENCE_TOP10_VARS",
    "REFERENCE_TOP10_SOURCE",
    "GP_PARAM_PROFILE",
    "MAIN_EFFECT_GP_PARAMS",
    "GP_ONLY_PARAMS",
    "ACTUAL_GP_PARAM_PROFILE",
    "ACTUAL_GP_PARAMS",
    "PHASEB_MIN_DELTA",
    "FINAL_PHASEB_GATE",
    "PHASEB_MODE",
    "PHASEB_USE_PH_VIOLATING_MAIN_EFFECTS",
    "PHASEB_USE_PH_VIOLATING_GP_FEATURES",
    "PHASEB_ENABLE_EXPLORATORY_STABLE_MAIN",
    "PHASEB_ENABLE_EXPLORATORY_ALL_GP",
    "PHASEB_TIME_BASIS_NAMES",
    "PHASEB_BASIS_PRIORITY",
    "PHASEB_MAX_BASIS_PER_FEATURE",
    "PHASEB_INTERVAL_CUTS",
    "PHASEB_T_WINSOR_Q",
    "PHASEB_PENALIZER",
    "PHASEB_GATE_MIN_MEAN_DELTA",
    "PHASEB_GATE_MIN_POSITIVE_FOLDS",
    "PHASEB_MEAN_DELTA_MIN",
    "PHASEB_POSITIVE_FOLDS_MIN",
    "PHASEB_COMPLEXITY_LAMBDA",
    "TIMEBASIS_COMPLEXITY_LAMBDA",
    "PHASEB_SELECTION_MODE",
    "SEED",
    "OUTER_FOLDS",
    "INNER_FOLDS",
    "FINAL_HPO_FOLDS",
    "N_TREES",
    "POP_SIZE_X",
    "N_GEN_X",
    "CX_PB_X",
    "MUT_PB_X",
    "TOURN_SIZE",
    "ELITE_K",
    "MAX_TREE_HEIGHT_X",
    "FIT_FOLDS_X",
    "POP_SIZE_T",
    "N_GEN_T",
    "CX_PB_T",
    "MUT_PB_T",
    "MAX_TREE_HEIGHT_T",
    "FIT_FOLDS_T",
    "COX_PENALIZER",
    "COX_L1_RATIO",
    "PH_P_THRESHOLD",
    "COMPLEXITY_LAMBDA_X",
    "COMPLEXITY_LAMBDA_T",
    "MICE_MAX_ITER",
    "MICE_RANDOM_STATE",
    "MICE_INITIAL_STRATEGY",
    "MICE_SKIP_COMPLETE",
    "WINSOR_Q_LOW",
    "WINSOR_Q_HIGH",
    "STANDARDIZE_CONTINUOUS",
    "STANDARDIZE_EPS",
    "DUPLICATE_TERMINAL_LAMBDA_X",
    "GP_MIN_FEATURE_STD",
    "ENABLE_GP_POST_FILTER",
    "GP_CORR_FILTER_THRESHOLD",
    "GP_MIN_ACTIVE_FEATURES",
    "K_INTERACTION_MAX",
    "K_INTERACTION",
    "POP_SIZE_C",
    "N_GEN_C",
    "CX_PB_C",
    "MUT_PB_C",
    "MAX_TREE_HEIGHT_C",
    "TOURN_SIZE_C",
    "ELITE_K_C",
    "COMPLEXITY_LAMBDA_C",
    "PHASEC_PENALIZER",
    "PHASEC_L1_RATIO",
    "PHASEC_MODE",
    "PHASEC_CAT_PREFIXES",
    "PHASEC_EXCLUDE_KEYWORDS",
    "PHASEC_BINARY_COLS",
    "PHASEC_MIN_MAIN_N",
    "PHASEC_MIN_MAIN_EVENTS",
    "PHASEC_MIN_INTER_N",
    "PHASEC_MIN_INTER_EVENTS",
    "PHASEC_POP_SIZE",
    "PHASEC_N_GEN",
    "PHASEC_MAX_TREE_HEIGHT",
    "PHASEC_K_PROFILE",
    "PHASEC_K_VALUES",
    "PHASEC_MEAN_DELTA_MIN",
    "PHASEC_POSITIVE_FOLDS_MIN",
    "PHASEC_SELECTION_MODE",
    "PHASEC_SELECTION_COMPLEXITY_LAMBDA",
    "PHASEC_COMPLEXITY_LAMBDA",
    "PHASEC_MAX_TERMS_PER_INDIVIDUAL",
    "PHASEC_MAX_INTERACTION_ORDER",
    "PHASEC_ELITE_SIZE",
    "PHASEC_HOF_SIZE",
    "PHASEC_CROSSOVER_RATE",
    "PHASEC_MUTATION_RATE",
    "PHASEC_RANDOM_INJECTION_RATE",
    "PHASEC_STGP_USE_AND_ONLY",
    "PHASEC_STGP_ALLOW_OR",
    "PHASEC_STGP_ALLOW_NOT",
    "PHASEC_STGP_ALLOW_CAT_CAT_AND",
    "PHASEC_STGP_MAX_TERMINALS_PER_RULE",
    "PHASEC_STGP_HEREDITY",
    "PHASEC_MIN_SUPPORT_TRAIN",
    "PHASEC_MIN_SUPPORT_VALID",
    "PHASEC_MIN_EVENT_SUPPORT",
    "VERBOSE_TRIAL_LOG",
    "PRINT_GEN_LOG",
}


def resolve_phasec_k_values(profile, explicit_values=None):
    """Resolve profile-style Phase-C K choices without changing legacy defaults."""
    if explicit_values is not None:
        return [int(k) for k in list(explicit_values)]
    mapping = {
        "legacy_k_0_1_2_3": [0, 1, 2, 3],
        "k_0_1_2": [0, 1, 2],
        "k_0_1_2_3": [0, 1, 2, 3],
        "k_1_2_3": [1, 2, 3],
        "k_0_1_2_3_4": [0, 1, 2, 3, 4],
    }
    if profile not in mapping:
        raise ValueError(f"Unknown PHASEC_K_PROFILE: {profile}")
    return list(mapping[profile])


def sync_phaseB_phaseC_profile_style_parameters():
    """Keep legacy variable names and profile-style names equivalent for defaults/patches."""
    global PHASEB_MEAN_DELTA_MIN, PHASEB_POSITIVE_FOLDS_MIN
    global PHASEB_GATE_MIN_MEAN_DELTA, PHASEB_GATE_MIN_POSITIVE_FOLDS
    global PHASEC_K_VALUES
    PHASEB_MEAN_DELTA_MIN = float(globals().get("PHASEB_MEAN_DELTA_MIN", globals().get("PHASEB_GATE_MIN_MEAN_DELTA", 0.0)))
    PHASEB_POSITIVE_FOLDS_MIN = int(globals().get("PHASEB_POSITIVE_FOLDS_MIN", globals().get("PHASEB_GATE_MIN_POSITIVE_FOLDS", 2)))
    PHASEB_GATE_MIN_MEAN_DELTA = PHASEB_MEAN_DELTA_MIN
    PHASEB_GATE_MIN_POSITIVE_FOLDS = PHASEB_POSITIVE_FOLDS_MIN
    PHASEC_K_VALUES = resolve_phasec_k_values(globals().get("PHASEC_K_PROFILE", "legacy_k_0_1_2_3"), globals().get("PHASEC_K_VALUES", None))


sync_phaseB_phaseC_profile_style_parameters()


def safe_div(a, b):
    return a / (b + 1e-9)


def safe_log(x):
    return np.log(np.abs(x) + 1e-9)


def safe_sqrt(x):
    return np.sqrt(np.abs(x) + 1e-9)


def clip(x, lo=-10, hi=10):
    return np.clip(x, lo, hi)


def _tree_complexity_x(individual):
    return float(sum(len(t) for t in individual))


def _tree_complexity_t(tree):
    return float(len(tree))


def tree_to_named_expr(tree, feature_names):
    expr = str(tree)

    def repl(m):
        idx = int(m.group(1))
        if 0 <= idx < len(feature_names):
            return feature_names[idx]
        return m.group(0)

    return re.sub(r"\bx(\d+)\b", repl, expr)


def fit_preprocess_continuous_only(df_raw):
    df = _coerce_time_event(df_raw).copy()
    cat_cols = [c for c in CATEGORICAL_CANDIDATES if c in df.columns]
    id_cols = [c for c in df.columns if _is_id_col(c)]
    drop_ = [TIME_COL, EVENT_COL] + cat_cols + id_cols
    num_cols_all = [c for c in df.columns if c not in drop_]
    for c in num_cols_all:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    cont_cols = [c for c in num_cols_all if not _is_binary_series(df[c])]
    cont_df = df[cont_cols].copy()
    fill_defaults = {}
    for c in cont_cols:
        fill_defaults[c] = float(cont_df[c].median(skipna=True)) if cont_df[c].notna().any() else 0.0
    imp, cont_imputed = _fit_mice_on_continuous(cont_df) if cont_cols else (None, pd.DataFrame(index=df.index))
    cont_standardize_stats = {}
    if cont_cols and STANDARDIZE_CONTINUOUS:
        cont_imputed, cont_standardize_stats = _fit_winsor_zscore_on_continuous(cont_imputed)
    out = df[[TIME_COL, EVENT_COL]].copy()
    for c in cont_cols:
        out[c] = cont_imputed[c]
    prep = {
        "cont_cols": cont_cols,
        "mice_imputer": imp,
        "fill_defaults": fill_defaults,
        "cont_standardize_stats": cont_standardize_stats,
        "standardize_continuous": STANDARDIZE_CONTINUOUS,
        "winsor_q_low": WINSOR_Q_LOW,
        "winsor_q_high": WINSOR_Q_HIGH,
    }
    return out, prep


def univariate_topk(df_enc, top_k=TOP_K):
    rows = []
    for col in df_enc.columns:
        if col in (TIME_COL, EVENT_COL):
            continue
        try:
            cph = make_cox()
            cph.fit(
                df_enc[[col, TIME_COL, EVENT_COL]],
                duration_col=TIME_COL,
                event_col=EVENT_COL
            )
            s = cph.summary.iloc[0]
            rows.append({"variable": col, "p": float(s["p"])})
        except Exception:
            pass

    if not rows:
        return [], pd.DataFrame()

    uni_df = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
    return uni_df.head(top_k)["variable"].tolist(), uni_df


def nested_variable_selection(df_raw, top_k=TOP_K, inner_folds=INNER_FOLDS, random_state=SEED):
    df_raw = _coerce_time_event(df_raw).reset_index(drop=True)

    y = df_raw[EVENT_COL].values
    inner_cv = SurvivalAwareKFold(
        n_splits=inner_folds,
        shuffle=True,
        random_state=random_state
    )

    selected_vars_all = []
    inner_records = []

    for inner_i, (inner_tr_idx, inner_val_idx) in enumerate(inner_cv.split(df_raw, y), 1):
        df_inner_train = df_raw.iloc[inner_tr_idx].reset_index(drop=True)

        df_inner_cont, _ = fit_preprocess_continuous_only(df_inner_train)
        top_vars, uni_df = univariate_topk(df_inner_cont, top_k=top_k)

        selected_vars_all.extend(top_vars)

        tmp = pd.DataFrame({
            "inner_fold": inner_i,
            "variable": top_vars,
            "rank_in_fold": np.arange(1, len(top_vars) + 1)
        })
        inner_records.append(tmp)

    if len(selected_vars_all) == 0:
        return [], pd.DataFrame(), pd.DataFrame()

    counter = Counter(selected_vars_all)
    final_vars = [v for v, _ in counter.most_common(top_k)]

    freq_df = pd.DataFrame(counter.items(), columns=["variable", "frequency"])
    freq_df = freq_df.sort_values(["frequency", "variable"], ascending=[False, True]).reset_index(drop=True)

    inner_selected_df = pd.concat(inner_records, axis=0, ignore_index=True) if inner_records else pd.DataFrame()

    return final_vars, freq_df, inner_selected_df


def build_gp_toolbox_x(n_features: int):
    pset = gp.PrimitiveSetTyped("MAIN_X", [np.ndarray] * n_features, np.ndarray)

    pset.addPrimitive(operator.add, [np.ndarray, np.ndarray], np.ndarray)
    pset.addPrimitive(operator.sub, [np.ndarray, np.ndarray], np.ndarray)
    pset.addPrimitive(operator.mul, [np.ndarray, np.ndarray], np.ndarray)
    pset.addPrimitive(safe_div,    [np.ndarray, np.ndarray], np.ndarray)
    pset.addPrimitive(safe_log,    [np.ndarray], np.ndarray)
    pset.addPrimitive(safe_sqrt,   [np.ndarray], np.ndarray)
    pset.addPrimitive(clip,        [np.ndarray], np.ndarray)

    pset.addEphemeralConstant(
        "rand_x",
        lambda: np.array(random.uniform(-2.0, 2.0), dtype=float),
        np.ndarray
    )

    for i in range(n_features):
        pset.renameArguments(**{f"ARG{i}": f"x{i}"})

    if "FitnessMax" not in creator.__dict__:
        creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    if "IndividualX" not in creator.__dict__:
        creator.create("IndividualX", list, fitness=creator.FitnessMax)

    n_trees = int(N_TREES)

    tb = base.Toolbox()
    tb.register(
        "expr_x",
        gp.genHalfAndHalf,
        pset=pset,
        min_=1,
        max_=max(1, int(MAX_TREE_HEIGHT_X))
    )
    tb.register("tree_x", tools.initIterate, gp.PrimitiveTree, tb.expr_x)

    def init_ind():
        return creator.IndividualX([tb.tree_x() for _ in range(n_trees)])

    tb.register("individual", init_ind)
    tb.register("population", tools.initRepeat, list, tb.individual)
    tb.register("compile",    gp.compile, pset=pset)

    def _limit(tree):
        return tb.tree_x() if tree.height > MAX_TREE_HEIGHT_X else tree

    def mate_x(i1, i2):
        for k in range(len(i1)):
            if random.random() < 0.5:
                gp.cxOnePoint(i1[k], i2[k])
            i1[k] = _limit(i1[k]); i2[k] = _limit(i2[k])
        return i1, i2

    def mutate_x(ind):
        if len(ind) == 0:
            return (ind,)
        k = random.randrange(len(ind))
        gp.mutUniform(ind[k], expr=tb.expr_x, pset=pset)
        ind[k] = _limit(ind[k])
        return (ind,)

    tb.register("mate",   mate_x)
    tb.register("mutate", mutate_x)
    tb.register("select", tools.selTournament, tournsize=TOURN_SIZE)
    return tb, pset


def compute_F(individual, toolbox_x, X: np.ndarray):
    funcs = [toolbox_x.compile(expr=tree) for tree in individual]
    cols  = [X[:, j] for j in range(X.shape[1])]
    try:
        F = np.column_stack([f(*cols) for f in funcs]).astype(float)
    except Exception:
        return None
    return np.nan_to_num(F, nan=0.0, posinf=0.0, neginf=0.0)


def terminal_duplicate_penalty(tree):
    toks = []
    for node in tree:
        name = getattr(node, "name", str(node))
        if re.fullmatch(r"x\d+", name):
            toks.append(name)

    if not toks:
        return 0.0

    counts = Counter(toks)
    return float(sum(max(0, c - 1) for c in counts.values()))


def _main_effect_vars_from_final(final_vars):
    # Legacy compatibility helper; current feature modes decide main effects explicitly.
    return []


def _main_effect_indices_from_final(final_vars, main_effect_vars):
    return [int(final_vars.index(v)) for v in main_effect_vars if v in final_vars]


def _main_matrix_from_X(X, main_effect_indices):
    if X is None or not main_effect_indices:
        return None
    return np.asarray(X[:, main_effect_indices], dtype=float)


def _fit_predict_cox_from_matrix(mat_train, names, T_train, E_train, mat_apply=None):
    if mat_train is None or len(names) == 0:
        return None, None, np.nan
    mat_train = np.asarray(mat_train, dtype=float)
    names = list(names)
    df_tr = pd.DataFrame(mat_train, columns=names)
    df_tr[TIME_COL] = np.asarray(T_train, dtype=float)
    df_tr[EVENT_COL] = np.asarray(E_train, dtype=int)
    try:
        cph = make_cox()
        cph.fit(df_tr, duration_col=TIME_COL, event_col=EVENT_COL)
        if mat_apply is None:
            risk = cph.predict_partial_hazard(df_tr).values.reshape(-1).astype(float)
            cidx = float(concordance_index(df_tr[TIME_COL].values, -risk, df_tr[EVENT_COL].values))
            return cph, risk, cidx
        df_ap = pd.DataFrame(np.asarray(mat_apply, dtype=float), columns=names)
        df_ap[TIME_COL] = 1.0
        df_ap[EVENT_COL] = 0
        risk = cph.predict_partial_hazard(df_ap).values.reshape(-1).astype(float)
        return cph, risk, np.nan
    except Exception:
        return None, None, np.nan


def score_main_only_on_test(M_dev, main_effect_names, T_dev, E_dev, M_te, T_te, E_te):
    if M_dev is None or M_te is None or not main_effect_names:
        return np.nan
    cph, risk, _ = _fit_predict_cox_from_matrix(M_dev, main_effect_names, T_dev, E_dev, M_te)
    if risk is None:
        return np.nan
    try:
        return float(concordance_index(T_te, -risk, E_te))
    except Exception:
        return np.nan


def predict_main_only_risk_on_test(M_dev, main_effect_names, T_dev, E_dev, M_te):
    if M_dev is None or M_te is None or not main_effect_names:
        return None
    _, risk, _ = _fit_predict_cox_from_matrix(M_dev, main_effect_names, T_dev, E_dev, M_te)
    return risk

def evaluate_individual_x(individual, toolbox_x, X, T, E, M=None, main_effect_names=None):
    F = compute_F(individual, toolbox_x, X)
    if F is None:
        return (0.0,)
    stds = np.nanstd(F, axis=0)
    if np.any(~np.isfinite(stds)) or np.any(stds < GP_MIN_FEATURE_STD):
        return (0.0,)

    main_effect_names = list(main_effect_names or [])
    use_main = M is not None and len(main_effect_names) > 0
    if use_main:
        M = np.asarray(M, dtype=float)

    kf = KFold(n_splits=FIT_FOLDS_X, shuffle=True, random_state=SEED)
    cidxs = []
    for tr, te in kf.split(F):
        sc = StandardScaler()
        Fz_tr = sc.fit_transform(F[tr])
        Fz_te = sc.transform(F[te])

        gp_cols = [f"gp{i+1}" for i in range(Fz_tr.shape[1])]
        if use_main:
            cols = main_effect_names + gp_cols
            mat_tr = np.column_stack([M[tr], Fz_tr])
            mat_te = np.column_stack([M[te], Fz_te])
        else:
            cols = gp_cols
            mat_tr = Fz_tr
            mat_te = Fz_te

        df_tr = pd.DataFrame(mat_tr, columns=cols)
        df_tr[TIME_COL] = T[tr]
        df_tr[EVENT_COL] = E[tr]

        df_te = pd.DataFrame(mat_te, columns=cols)
        df_te[TIME_COL] = T[te]
        df_te[EVENT_COL] = E[te]

        try:
            cph = make_cox()
            cph.fit(df_tr, duration_col=TIME_COL, event_col=EVENT_COL)
            risk = cph.predict_partial_hazard(df_te).values.reshape(-1)
            cidxs.append(float(concordance_index(df_te[TIME_COL].values, -risk, df_te[EVENT_COL].values)))
        except Exception:
            return (0.0,)

    base = float(np.mean(cidxs))
    complexity_pen = COMPLEXITY_LAMBDA_X * _tree_complexity_x(individual)
    dup_pen = DUPLICATE_TERMINAL_LAMBDA_X * sum(
        terminal_duplicate_penalty(tree) for tree in individual
    )
    return (base - complexity_pen - dup_pen,)


def _single_gp_cindex(feature, T, E):
    try:
        f = np.asarray(feature, dtype=float).reshape(-1, 1)

        if np.nanstd(f) < GP_MIN_FEATURE_STD:
            return -np.inf

        sc = StandardScaler()
        fz = sc.fit_transform(f)

        df = pd.DataFrame({
            "gp": fz.reshape(-1),
            TIME_COL: T,
            EVENT_COL: E
        })

        cph = make_cox()
        cph.fit(df, duration_col=TIME_COL, event_col=EVENT_COL)

        risk = cph.predict_partial_hazard(df).values.reshape(-1)
        return float(concordance_index(T, -risk, E))

    except Exception:
        return -np.inf


def filter_gp_trees_after_evolution(best_ind_x, toolbox_x, X, T, E):
    if not ENABLE_GP_POST_FILTER:
        F = compute_F(best_ind_x, toolbox_x, X)
        return best_ind_x, F, {"enabled": False}

    F = compute_F(best_ind_x, toolbox_x, X)

    if F is None:
        return best_ind_x, F, {
            "enabled": True,
            "reason": "F_none"
        }

    stds = np.nanstd(F, axis=0)
    valid_idx = [
        i for i, s in enumerate(stds)
        if np.isfinite(s) and s >= GP_MIN_FEATURE_STD
    ]

    if len(valid_idx) == 0:
        return best_ind_x, F, {
            "enabled": True,
            "reason": "no_valid_gp",
            "original_n": int(len(best_ind_x)),
            "kept_n": int(len(best_ind_x)),
        }

    scores = {
        i: _single_gp_cindex(F[:, i], T, E)
        for i in valid_idx
    }

    ranked = sorted(
        valid_idx,
        key=lambda i: scores.get(i, -np.inf),
        reverse=True
    )

    selected = []

    for i in ranked:
        if len(selected) == 0:
            selected.append(i)
            continue

        keep = True

        for j in selected:
            try:
                r = np.corrcoef(F[:, i], F[:, j])[0, 1]
                if np.isfinite(r) and abs(r) >= GP_CORR_FILTER_THRESHOLD:
                    keep = False
                    break
            except Exception:
                pass

        if keep:
            selected.append(i)

    if len(selected) < GP_MIN_ACTIVE_FEATURES:
        selected = ranked[:min(GP_MIN_ACTIVE_FEATURES, len(ranked))]

    filtered_ind = creator.IndividualX([best_ind_x[i] for i in selected])
    F_filtered = compute_F(filtered_ind, toolbox_x, X)

    diag = {
        "enabled": True,
        "original_n": int(len(best_ind_x)),
        "kept_n": int(len(filtered_ind)),
        "selected_indices": [int(i) for i in selected],
        "removed_indices": [
            int(i) for i in range(len(best_ind_x))
            if i not in selected
        ],
        "single_gp_cindex": {
            str(int(k)): float(v)
            for k, v in scores.items()
            if np.isfinite(v)
        },
        "stds": [
            float(x) if np.isfinite(x) else np.nan
            for x in stds
        ],
    }

    return filtered_ind, F_filtered, diag


def build_gp_toolbox_t():
    pset = gp.PrimitiveSetTyped("MAIN_T", [np.ndarray], np.ndarray)

    pset.addPrimitive(operator.add, [np.ndarray, np.ndarray], np.ndarray)
    pset.addPrimitive(operator.sub, [np.ndarray, np.ndarray], np.ndarray)
    pset.addPrimitive(operator.mul, [np.ndarray, np.ndarray], np.ndarray)
    pset.addPrimitive(safe_div,    [np.ndarray, np.ndarray], np.ndarray)
    pset.addPrimitive(safe_log,    [np.ndarray], np.ndarray)
    pset.addPrimitive(safe_sqrt,   [np.ndarray], np.ndarray)
    pset.addPrimitive(clip,        [np.ndarray], np.ndarray)

    pset.addEphemeralConstant(
        "rand_t",
        lambda: np.array(random.uniform(-2.0, 2.0), dtype=float),
        np.ndarray
    )

    pset.renameArguments(ARG0="logt")

    if "IndividualT" not in creator.__dict__:
        creator.create("IndividualT", gp.PrimitiveTree, fitness=creator.FitnessMax)

    tb = base.Toolbox()
    tb.register(
        "expr_t",
        gp.genHalfAndHalf,
        pset=pset,
        min_=1,
        max_=max(1, int(MAX_TREE_HEIGHT_T))
    )
    tb.register("individual", tools.initIterate, creator.IndividualT, tb.expr_t)
    tb.register("population", tools.initRepeat, list, tb.individual)
    tb.register("compile",    gp.compile, pset=pset)

    def _limit_t(tree):
        if tree.height > MAX_TREE_HEIGHT_T:
            new = gp.PrimitiveTree(tb.expr_t())
            return creator.IndividualT(new)
        return tree

    def mate_t(i1, i2):
        gp.cxOnePoint(i1, i2)
        return _limit_t(i1), _limit_t(i2)

    def mutate_t(ind):
        gp.mutUniform(ind, expr=tb.expr_t, pset=pset)
        return (_limit_t(ind),)

    tb.register("mate",   mate_t)
    tb.register("mutate", mutate_t)
    tb.register("select", tools.selTournament, tournsize=TOURN_SIZE)
    return tb, pset


def ph_test_on_train_full_model(M_tr, main_effect_names, F_tr, T_tr, E_tr):
    main_effect_names = list(main_effect_names or [])
    sc = StandardScaler()
    Fz = sc.fit_transform(F_tr)
    n_gp = int(Fz.shape[1])
    gp_cols = [f"gp{i+1}" for i in range(n_gp)]

    if M_tr is not None and len(main_effect_names) > 0:
        mat = np.column_stack([np.asarray(M_tr, dtype=float), Fz])
        cols = main_effect_names + gp_cols
    else:
        mat = Fz
        cols = gp_cols

    df = pd.DataFrame(mat, columns=cols)
    df[TIME_COL] = T_tr
    df[EVENT_COL] = E_tr

    main_violating = []
    gp_violating = []
    try:
        cph = make_cox()
        cph.fit(df, duration_col=TIME_COL, event_col=EVENT_COL)
        res = proportional_hazard_test(cph, df, time_transform="rank")
        ph_sum = res.summary
        for name in main_effect_names:
            if name in ph_sum.index and float(ph_sum.loc[name, "p"]) < PH_P_THRESHOLD:
                main_violating.append(name)
        for i, col in enumerate(gp_cols):
            if col in ph_sum.index and float(ph_sum.loc[col, "p"]) < PH_P_THRESHOLD:
                gp_violating.append(i)
    except Exception:
        main_violating = []
        gp_violating = list(range(F_tr.shape[1]))
    return main_violating, gp_violating


def compute_g_centered(tree, toolbox_t, logt_train, logt_apply):
    func = toolbox_t.compile(expr=tree)
    try:
        g_tr = np.asarray(func(logt_train), dtype=float).reshape(-1)
        g_ap = np.asarray(func(logt_apply), dtype=float).reshape(-1)
    except Exception:
        g_tr = np.zeros(len(logt_train), dtype=float)
        g_ap = np.zeros(len(logt_apply), dtype=float)

    g_tr = np.nan_to_num(g_tr, nan=0.0, posinf=0.0, neginf=0.0)
    g_ap = np.nan_to_num(g_ap, nan=0.0, posinf=0.0, neginf=0.0)

    mu = float(g_tr.mean())
    return g_ap - mu, mu


def build_cox_matrix(F, T,
                     violating_indices,
                     g_trees_dict,
                     toolbox_t,
                     M=None,
                     main_effect_names=None,
                     sc_F=None, g_mus=None, sc_I_list=None,
                     logt_train_ref=None, logt_apply=None):
    fit_mode = (sc_F is None)
    if logt_apply is None:
        logt_apply = np.log(np.clip(T, MIN_TIME, None))

    if fit_mode:
        sc_F = StandardScaler()
        Fz = sc_F.fit_transform(F)
        g_mus = []
        sc_I_list = []
    else:
        Fz = sc_F.transform(F)

    n_gp = int(Fz.shape[1])
    gp_col_names = [f"gp{i+1}" for i in range(n_gp)]
    main_effect_names = list(main_effect_names or [])
    if M is not None and len(main_effect_names) > 0:
        mat = np.column_stack([np.asarray(M, dtype=float), Fz])
        col_names = main_effect_names + gp_col_names
    else:
        mat = Fz.copy()
        col_names = gp_col_names

    for j in violating_indices:
        tree = g_trees_dict[j]
        if fit_mode:
            g_c, mu = compute_g_centered(tree, toolbox_t, logt_train_ref, logt_apply)
            I = Fz[:, j] * g_c
            sc_I = StandardScaler()
            I_z = sc_I.fit_transform(I.reshape(-1, 1)).reshape(-1)
            g_mus.append((j, mu))
            sc_I_list.append((j, sc_I))
        else:
            mu = dict(g_mus)[j]
            sc_I = dict(sc_I_list)[j]
            func = toolbox_t.compile(expr=tree)
            try:
                g_raw = np.asarray(func(logt_apply), dtype=float).reshape(-1)
            except Exception:
                g_raw = np.zeros(len(logt_apply), dtype=float)
            g_raw = np.nan_to_num(g_raw, nan=0.0, posinf=0.0, neginf=0.0)
            I = Fz[:, j] * (g_raw - mu)
            I_z = sc_I.transform(I.reshape(-1, 1)).reshape(-1)

        mat = np.column_stack([mat, I_z])
        col_names.append(f"gp{j+1}_x_gt")

    if fit_mode:
        return mat, col_names, sc_F, g_mus, sc_I_list
    return mat, col_names


def evaluate_individual_t(tree, toolbox_t,
                          F_train_raw, T_train, E_train,
                          target_j):
    logt_all = np.log(np.clip(T_train, MIN_TIME, None))

    kf = KFold(n_splits=FIT_FOLDS_T, shuffle=True, random_state=SEED)
    cidxs = []

    for tr, te in kf.split(F_train_raw):
        F_tr_raw = F_train_raw[tr]
        F_te_raw = F_train_raw[te]
        T_tr = T_train[tr]
        T_te = T_train[te]
        E_tr = E_train[tr]
        E_te = E_train[te]

        logt_tr = logt_all[tr]
        logt_te = logt_all[te]

        scF = StandardScaler()
        Fz_tr = scF.fit_transform(F_tr_raw)
        Fz_te = scF.transform(F_te_raw)

        g_te_c, _ = compute_g_centered(tree, toolbox_t, logt_tr, logt_te)
        g_tr_c, _ = compute_g_centered(tree, toolbox_t, logt_tr, logt_tr)

        I_tr = Fz_tr[:, target_j] * g_tr_c
        I_te = Fz_te[:, target_j] * g_te_c

        scI = StandardScaler()
        I_tr_z = scI.fit_transform(I_tr.reshape(-1, 1)).reshape(-1)
        I_te_z = scI.transform(I_te.reshape(-1, 1)).reshape(-1)

        mat_tr = np.column_stack([Fz_tr, I_tr_z])
        mat_te = np.column_stack([Fz_te, I_te_z])

        cols = [f"gp{i+1}" for i in range(Fz_tr.shape[1])] + [f"gp{target_j+1}_x_gt"]
        df_tr = pd.DataFrame(mat_tr, columns=cols)
        df_tr[TIME_COL] = T_tr
        df_tr[EVENT_COL] = E_tr
        df_te = pd.DataFrame(mat_te, columns=cols)
        df_te[TIME_COL] = T_te
        df_te[EVENT_COL] = E_te

        try:
            cph = make_cox()
            cph.fit(df_tr, duration_col=TIME_COL, event_col=EVENT_COL)
            risk = cph.predict_partial_hazard(df_te).values.reshape(-1)
            cidxs.append(float(concordance_index(T_te, -risk, E_te)))
        except Exception:
            return (0.0,)

    base = float(np.mean(cidxs))
    pen = COMPLEXITY_LAMBDA_T * _tree_complexity_t(tree)
    return (base - pen,)


def predict_ab_risk_on_test(F_dev, T_dev, E_dev,
                            F_te, T_te, E_te,
                            g_trees_dict, toolbox_t,
                            M_dev=None, M_te=None, main_effect_names=None):
    if g_trees_dict:
        return predict_full_model_risk_on_test(
            F_dev, T_dev, E_dev, F_te, T_te, E_te, g_trees_dict, toolbox_t,
            M_dev=M_dev, M_te=M_te, main_effect_names=main_effect_names
        )
    return predict_phase_a_risk_on_test(
        F_dev, T_dev, E_dev, F_te, T_te, E_te,
        M_dev=M_dev, M_te=M_te, main_effect_names=main_effect_names
    )


def score_phase_a_on_test(F_dev, T_dev, E_dev, F_te, T_te, E_te,
                          M_dev=None, M_te=None, main_effect_names=None):
    if F_dev is None or F_te is None:
        return np.nan
    sc = StandardScaler()
    Fz_dev = sc.fit_transform(F_dev)
    Fz_te = sc.transform(F_te)

    gp_cols = [f"gp{i+1}" for i in range(Fz_dev.shape[1])]
    main_effect_names = list(main_effect_names or [])
    if M_dev is not None and M_te is not None and len(main_effect_names) > 0:
        cols = main_effect_names + gp_cols
        mat_dev = np.column_stack([np.asarray(M_dev, dtype=float), Fz_dev])
        mat_te = np.column_stack([np.asarray(M_te, dtype=float), Fz_te])
    else:
        cols = gp_cols
        mat_dev = Fz_dev
        mat_te = Fz_te

    df_dev = pd.DataFrame(mat_dev, columns=cols)
    df_dev[TIME_COL] = T_dev
    df_dev[EVENT_COL] = E_dev
    df_te = pd.DataFrame(mat_te, columns=cols)
    df_te[TIME_COL] = T_te
    df_te[EVENT_COL] = E_te
    try:
        cph = make_cox()
        cph.fit(df_dev, duration_col=TIME_COL, event_col=EVENT_COL)
        risk = cph.predict_partial_hazard(df_te).values.reshape(-1)
        return float(concordance_index(T_te, -risk, E_te))
    except Exception:
        return np.nan


def predict_phase_a_risk_on_test(F_dev, T_dev, E_dev, F_te, T_te, E_te,
                                 M_dev=None, M_te=None, main_effect_names=None):
    if F_dev is None or F_te is None:
        return None
    sc = StandardScaler()
    Fz_dev = sc.fit_transform(F_dev)
    Fz_te = sc.transform(F_te)

    gp_cols = [f"gp{i+1}" for i in range(Fz_dev.shape[1])]
    main_effect_names = list(main_effect_names or [])
    if M_dev is not None and M_te is not None and len(main_effect_names) > 0:
        cols = main_effect_names + gp_cols
        mat_dev = np.column_stack([np.asarray(M_dev, dtype=float), Fz_dev])
        mat_te = np.column_stack([np.asarray(M_te, dtype=float), Fz_te])
    else:
        cols = gp_cols
        mat_dev = Fz_dev
        mat_te = Fz_te

    df_dev = pd.DataFrame(mat_dev, columns=cols)
    df_dev[TIME_COL] = T_dev
    df_dev[EVENT_COL] = E_dev
    df_te = pd.DataFrame(mat_te, columns=cols)
    df_te[TIME_COL] = T_te
    df_te[EVENT_COL] = E_te
    try:
        cph = make_cox()
        cph.fit(df_dev, duration_col=TIME_COL, event_col=EVENT_COL)
        return cph.predict_partial_hazard(df_te).values.reshape(-1).astype(float)
    except Exception:
        return None


def predict_full_model_risk_on_test(F_dev, T_dev, E_dev,
                                    F_te, T_te, E_te,
                                    g_trees_dict, toolbox_t,
                                    M_dev=None, M_te=None, main_effect_names=None):
    if F_dev is None or F_te is None:
        return None

    vi = sorted(g_trees_dict.keys())
    logt_dev = np.log(np.clip(T_dev, MIN_TIME, None))
    logt_te = np.log(np.clip(T_te, MIN_TIME, None))

    try:
        mat_dev, cols_dev, scF, g_mus, scI = build_cox_matrix(
            F_dev, T_dev, vi, g_trees_dict, toolbox_t,
            M=M_dev, main_effect_names=main_effect_names,
            sc_F=None, g_mus=None, sc_I_list=None,
            logt_train_ref=logt_dev, logt_apply=logt_dev
        )
        mat_te, cols_te = build_cox_matrix(
            F_te, T_te, vi, g_trees_dict, toolbox_t,
            M=M_te, main_effect_names=main_effect_names,
            sc_F=scF, g_mus=g_mus, sc_I_list=scI,
            logt_train_ref=None, logt_apply=logt_te
        )

        df_dev = pd.DataFrame(mat_dev, columns=cols_dev)
        df_dev[TIME_COL] = T_dev
        df_dev[EVENT_COL] = E_dev
        df_te = pd.DataFrame(mat_te, columns=cols_te)
        df_te[TIME_COL] = T_te
        df_te[EVENT_COL] = E_te
        cph = make_cox()
        cph.fit(df_dev, duration_col=TIME_COL, event_col=EVENT_COL)
        return cph.predict_partial_hazard(df_te).values.reshape(-1).astype(float)
    except Exception:
        return None


def run_evolution(toolbox, pop_size, n_gen, cx_pb, mut_pb, label="", on_generation_end=None):
    global PHASEA_CHECKPOINT_COUNTER
    phasea_checkpoint_index = None
    is_phasea = "Phase-A" in str(label) or "PhaseA" in str(label)
    if is_phasea:
        PHASEA_CHECKPOINT_COUNTER = int(globals().get("PHASEA_CHECKPOINT_COUNTER", 0)) + 1
        phasea_checkpoint_index = PHASEA_CHECKPOINT_COUNTER
    pop = toolbox.population(n=pop_size)
    hof = tools.HallOfFame(ELITE_K)

    stat = tools.Statistics(lambda ind: ind.fitness.values[0])
    stat.register("avg", np.mean)
    stat.register("max", np.max)

    for ind in [i for i in pop if not i.fitness.valid]:
        ind.fitness.values = toolbox.evaluate(ind)
    hof.update(pop)

    if on_generation_end is not None:
        on_generation_end(gen=0, pop=pop, hof=hof)

    for gen in range(1, n_gen + 1):
        offspring = list(map(toolbox.clone, toolbox.select(pop, len(pop))))

        for c1, c2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < cx_pb:
                toolbox.mate(c1, c2)
                del c1.fitness.values
                del c2.fitness.values

        for mut in offspring:
            if random.random() < mut_pb:
                toolbox.mutate(mut)
                del mut.fitness.values

        for ind in [i for i in offspring if not i.fitness.valid]:
            ind.fitness.values = toolbox.evaluate(ind)

        pop[:] = offspring
        hof.update(pop)

        rec = stat.compile(pop)
        if PRINT_GEN_LOG:
            print(f"  [{label}] Gen {gen:02d} | avg={rec['avg']:.4f} max={rec['max']:.4f} hofFit={hof[0].fitness.values[0]:.4f}")

        if on_generation_end is not None:
            on_generation_end(gen=gen, pop=pop, hof=hof)

    if is_phasea:
        import pickle
        checkpoint_dir = Path(OUT_XLSX).parent / "phaseA_checkpoints"
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        checkpoint_name = f"phaseA_call_{int(phasea_checkpoint_index):02d}_generation_{int(n_gen):02d}"
        checkpoint_path = checkpoint_dir / f"{checkpoint_name}.pkl"
        payload = {
            "schema_version": "phaseA_deap_checkpoint_v1",
            "endpoint": globals().get("ENDPOINT", globals().get("TARGET_NAME", "")),
            "target_data_path": str(TRAIN_DATA_PATH),
            "phaseA_call_index": int(phasea_checkpoint_index),
            "generation": int(n_gen),
            "population_expressions": [[str(tree) for tree in ind] for ind in pop],
            "population_fitness": [tuple(float(v) for v in ind.fitness.values) for ind in pop],
            "hall_of_fame_expressions": [[str(tree) for tree in ind] for ind in hof],
            "hall_of_fame_fitness": [tuple(float(v) for v in ind.fitness.values) for ind in hof],
            "python_random_state": random.getstate(),
            "numpy_random_state": np.random.get_state(),
            "population_size": int(pop_size),
            "cx_probability": float(cx_pb),
            "mutation_probability": float(mut_pb),
            "n_trees": int(N_TREES),
            "max_tree_height": int(MAX_TREE_HEIGHT_X),
            "gp_input_vars": list(GP_INPUT_VARS),
            "main_effect_vars": list(MAIN_EFFECT_VARS),
            "run_config": _build_run_config_snapshot(),
        }
        with checkpoint_path.open("wb") as handle:
            pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)
        checkpoint_audit = {
            "schema_version": payload["schema_version"],
            "checkpoint_path": str(checkpoint_path.resolve()),
            "phaseA_call_index": payload["phaseA_call_index"],
            "generation": payload["generation"],
            "population_n": len(payload["population_expressions"]),
            "hall_of_fame_n": len(payload["hall_of_fame_expressions"]),
            "population_fitness_valid_n": int(sum(bool(ind.fitness.valid) for ind in pop)),
            "python_rng_saved": True,
            "numpy_rng_saved": True,
            "gp_input_vars": payload["gp_input_vars"],
            "main_effect_vars": payload["main_effect_vars"],
        }
        (checkpoint_dir / f"{checkpoint_name}.json").write_text(
            json.dumps(checkpoint_audit, ensure_ascii=False, indent=2), encoding="utf-8"
        )

    return hof


def plot_final_fulltrain_trajectory(traj, out_png):
    if not isinstance(traj, dict):
        return
    a = list(traj.get("phaseA", []))
    b = list(traj.get("phaseB", []))
    c = traj.get("phaseC", np.nan)
    ys = a + b + ([c] if np.isfinite(c) else [])
    if len(ys) == 0:
        return
    xs = np.arange(len(ys))
    y_finite = [v for v in ys if np.isfinite(v)]
    y_top = max(y_finite) if y_finite else 1.0
    b_start = len(a)
    c_start = len(a) + len(b)

    plt.figure(figsize=(9, 5.2))
    plt.plot(xs, ys, marker="o", linewidth=2.0, color="tab:blue")
    plt.axvline(x=0, color="gray", linestyle="--", linewidth=0.9, alpha=0.45)
    plt.text(0, y_top, "Phase A start", fontsize=9, va="bottom", ha="left")
    if b_start < len(ys):
        plt.axvline(x=b_start, color="gray", linestyle="--", linewidth=0.9, alpha=0.45)
        plt.text(b_start, y_top, "Phase B start", fontsize=9, va="bottom", ha="left")
    if c_start < len(ys):
        plt.axvline(x=c_start, color="gray", linestyle="--", linewidth=0.9, alpha=0.45)
        plt.text(c_start, y_top, "Phase C start", fontsize=9, va="bottom", ha="left")
    plt.xlabel("Cumulative evolution generation")
    plt.ylabel("Tracked score")
    plt.title("Final full-train Phase A/B/C trajectory")
    plt.grid(True, alpha=0.25)
    Path(out_png).parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(out_png, dpi=220)
    plt.close()


def plot_phasea_4fold_valid_cindex_trajectory(df_phasea_traj, out_png):
    if df_phasea_traj is None or len(df_phasea_traj) == 0:
        return
    dfa = df_phasea_traj.copy()
    if "phase" in dfa.columns:
        dfa = dfa[dfa["phase"] == "A"].copy()
    if len(dfa) == 0:
        return
    plt.figure(figsize=(9.2, 5.2))
    for fold in sorted(pd.to_numeric(dfa["fold"], errors="coerce").dropna().astype(int).unique()):
        ds = dfa[pd.to_numeric(dfa["fold"], errors="coerce").astype("Int64") == int(fold)].copy()
        ds = ds.sort_values("generation")
        xs = pd.to_numeric(ds["generation"], errors="coerce").values
        ys = pd.to_numeric(ds["valid_hof_cindex"], errors="coerce").values
        plt.plot(xs, ys, marker="o", linewidth=1.8, label=f"Fold {fold}")
    plt.xlabel("Generation")
    plt.ylabel("Validation C-index")
    plt.title("Phase-A 4-fold validation C-index trajectories")
    plt.grid(True, alpha=0.25)
    plt.legend()
    Path(out_png).parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(out_png, dpi=220)
    plt.close()


def _apply_overrides(overrides):
    if not overrides:
        return
    g = globals()
    for k, v in overrides.items():
        if k in OVERRIDABLE_KEYS:
            g[k] = v


def _build_run_config_snapshot():
    g = globals()
    cfg = {}
    for k in sorted(OVERRIDABLE_KEYS):
        cfg[k] = g.get(k)
    return cfg


def get_fixed_ab_params():
    return {
        "TOP_K": TOP_K,
        "FEATURE_SET_MODE": FEATURE_SET_MODE,
        "BASELINE_STABLE_MIN_STRICT_N": BASELINE_STABLE_MIN_STRICT_N,
        "BASELINE_STABLE_VARS": list(BASELINE_STABLE_VARS),
        "BASELINE_STABLE_VAR_FREQUENCY": list(BASELINE_STABLE_VAR_FREQUENCY),
        "BASELINE_STABLE_MIN_FREQ_USED": BASELINE_STABLE_MIN_FREQ_USED,
        "FINAL_FEATURE_SET_MODE": FINAL_FEATURE_SET_MODE,
        "FEATURE_SET_DECISION": FEATURE_SET_DECISION,
        "USE_MAIN_EFFECTS_FOR_THIS_RUN": USE_MAIN_EFFECTS_FOR_THIS_RUN,
        "MAIN_EFFECT_VARS": list(MAIN_EFFECT_VARS),
        "GP_INPUT_VARS": list(GP_INPUT_VARS),
        "REFERENCE_TOP10_VARS": list(REFERENCE_TOP10_VARS),
        "REFERENCE_TOP10_SOURCE": REFERENCE_TOP10_SOURCE,
        "GP_PARAM_PROFILE": GP_PARAM_PROFILE,
        "ACTUAL_GP_PARAM_PROFILE": ACTUAL_GP_PARAM_PROFILE,
        "ACTUAL_GP_PARAMS": dict(ACTUAL_GP_PARAMS),
        "FINAL_PHASEB_GATE": FINAL_PHASEB_GATE,
        "N_TREES": N_TREES,
        "POP_SIZE_X": POP_SIZE_X,
        "N_GEN_X": N_GEN_X,
        "CX_PB_X": CX_PB_X,
        "MUT_PB_X": MUT_PB_X,
        "TOURN_SIZE": TOURN_SIZE,
        "ELITE_K": ELITE_K,
        "MAX_TREE_HEIGHT_X": MAX_TREE_HEIGHT_X,
        "FIT_FOLDS_X": FIT_FOLDS_X,
        "POP_SIZE_T": POP_SIZE_T,
        "N_GEN_T": N_GEN_T,
        "MAX_TREE_HEIGHT_T": MAX_TREE_HEIGHT_T,
        "FIT_FOLDS_T": FIT_FOLDS_T,
        "CX_PB_T": CX_PB_T,
        "MUT_PB_T": MUT_PB_T,
        "COX_PENALIZER": COX_PENALIZER,
        "COX_L1_RATIO": COX_L1_RATIO,
        "PH_P_THRESHOLD": PH_P_THRESHOLD,
        "PHASEB_MIN_DELTA": PHASEB_MIN_DELTA,
        "COMPLEXITY_LAMBDA_X": COMPLEXITY_LAMBDA_X,
        "COMPLEXITY_LAMBDA_T": COMPLEXITY_LAMBDA_T,
    }


def get_fixed_phasec_params(k_interaction=0):
    # K_interaction means the max number of Phase-C interaction expressions (trees).
    # One expression counts as one K; inherited main effects do not count toward K.
    # K is an upper bound, not necessarily equal to final selected tree count.
    return {
        "K_interaction": int(k_interaction),
        "POP_SIZE_C": POP_SIZE_C,
        "N_GEN_C": N_GEN_C,
        "CX_PB_C": CX_PB_C,
        "MUT_PB_C": MUT_PB_C,
        "MAX_TREE_HEIGHT_C": MAX_TREE_HEIGHT_C,
        "TOURN_SIZE_C": TOURN_SIZE_C,
        "ELITE_K_C": ELITE_K_C,
        "COMPLEXITY_LAMBDA_C": COMPLEXITY_LAMBDA_C,
        "PHASEC_MIN_SUPPORT_TRAIN": PHASEC_MIN_SUPPORT_TRAIN,
        "PHASEC_MIN_SUPPORT_VALID": PHASEC_MIN_SUPPORT_VALID,
        "PHASEC_MIN_EVENT_SUPPORT": PHASEC_MIN_EVENT_SUPPORT,
        "PHASEC_PENALIZER": PHASEC_PENALIZER,
        "PHASEC_L1_RATIO": PHASEC_L1_RATIO,
    }


def _temporary_set_params(params):
    g = globals()
    old = {}
    for k, v in params.items():
        old[k] = g.get(k)
        g[k] = v
    return old


def _restore_params(old):
    g = globals()
    for k, v in old.items():
        g[k] = v


def _safe_float_cindex(T, E, risk):
    T = np.asarray(T, dtype=float)
    E = np.asarray(E, dtype=int)
    risk = np.asarray(risk, dtype=float)
    mask = np.isfinite(T) & np.isfinite(E) & np.isfinite(risk)
    if int(mask.sum()) < 3:
        return np.nan
    try:
        return float(concordance_index(T[mask], -risk[mask], E[mask]))
    except Exception:
        return np.nan


def _parse_selected_vars(raw):
    if isinstance(raw, list):
        return [str(x) for x in raw]
    if pd.isna(raw):
        return []
    if isinstance(raw, str):
        s = raw.strip()
        if not s:
            return []
        try:
            v = json.loads(s)
            if isinstance(v, list):
                return [str(x) for x in v]
        except Exception:
            pass
        return [x.strip() for x in s.split(",") if x.strip()]
    return []


def extract_baseline_stable_vars_from_cell2_results(df_train_raw=None, top_k=TOP_K):
    fold_rows = []
    baseline_xlsx = globals().get("OUT_XLSX_BASELINE", "")
    if baseline_xlsx and Path(str(baseline_xlsx)).exists():
        try:
            df_fold = pd.read_excel(str(baseline_xlsx), sheet_name="Fold_Summary")
            if {"fold", "selected_continuous_vars"}.issubset(df_fold.columns):
                for _, r in df_fold.iterrows():
                    fold_rows.append({"fold": int(r["fold"]), "selected_vars": _parse_selected_vars(r["selected_continuous_vars"])[:int(top_k)]})
        except Exception as exc:
            print(f"[BaselineStableVars] Could not read Cell2 Fold_Summary from {baseline_xlsx}: {exc}")
    if not fold_rows:
        if df_train_raw is None:
            raise RuntimeError("Cell2 baseline Fold_Summary is unavailable and df_train_raw was not provided.")
        df_train_raw = _coerce_time_event(df_train_raw).reset_index(drop=True)
        y = df_train_raw[EVENT_COL].astype(int).values
        cv = SurvivalAwareKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=SEED)
        for fold, (tr_idx, _) in enumerate(cv.split(np.arange(len(y)), y), 1):
            df_tr = df_train_raw.iloc[tr_idx].reset_index(drop=True)
            enc_tr, prep = fit_preprocess(df_tr, cat_candidates=CATEGORICAL_CANDIDATES)
            cont_cols = [c for c in prep.get("cont_cols", []) if c in enc_tr.columns]
            enc_cont = enc_tr[[TIME_COL, EVENT_COL] + cont_cols].copy()
            top_vars, _ = univariate_topk(enc_cont, top_k=int(top_k))
            fold_rows.append({"fold": int(fold), "selected_vars": top_vars[:int(top_k)]})
    fold_rows = sorted(fold_rows, key=lambda x: int(x["fold"]))[:int(OUTER_FOLDS)]
    for fr in fold_rows:
        print(f"[BaselineStableVars] Fold {int(fr['fold'])} top{int(top_k)} = {fr['selected_vars']}")
    selected_folds = {}
    for fr in fold_rows:
        fold = int(fr["fold"])
        for v in list(dict.fromkeys(fr["selected_vars"][:int(top_k)])):
            selected_folds.setdefault(str(v), set()).add(fold)
    freq_rows = []
    for v, folds_set in selected_folds.items():
        folds = sorted(int(x) for x in folds_set)
        frequency = int(len(folds))
        freq_rows.append({"variable": v, "frequency": frequency, "selected_folds": json.dumps(folds, ensure_ascii=False), "included_in_strict_stable_vars": int(frequency == int(OUTER_FOLDS))})
    freq_df = pd.DataFrame(freq_rows, columns=["variable", "frequency", "selected_folds", "included_in_strict_stable_vars"])
    if len(freq_df):
        freq_df = freq_df.sort_values(["frequency", "variable"], ascending=[False, True]).reset_index(drop=True)
    strict_stable_vars = freq_df.loc[freq_df["included_in_strict_stable_vars"] == 1, "variable"].tolist() if len(freq_df) else []
    print(f"[BaselineStableVars] strict {int(OUTER_FOLDS)}/{int(OUTER_FOLDS)} vars = {strict_stable_vars}")
    print(f"[BaselineStableVars] strict_n = {len(strict_stable_vars)}")
    print(f"[BaselineStableVars] BASELINE_STABLE_MIN_STRICT_N = {int(BASELINE_STABLE_MIN_STRICT_N)}")
    return strict_stable_vars, freq_df, int(OUTER_FOLDS)


def decide_feature_set_from_baseline_stable(strict_stable_vars, reference_top10_vars):
    strict_stable_vars = list(strict_stable_vars or [])
    reference_top10_vars = list(reference_top10_vars or [])[:int(TOP_K)]
    strict_n = int(len(strict_stable_vars))
    mode = str(FEATURE_SET_MODE).lower()
    final_mode = ("baseline_stable_main_gp" if strict_n >= int(BASELINE_STABLE_MIN_STRICT_N) else "top10_gp_only") if mode == "auto" else mode
    if final_mode not in {"baseline_stable_main_gp", "top10_gp_only"}:
        raise ValueError("FEATURE_SET_MODE must be 'auto', 'baseline_stable_main_gp', or 'top10_gp_only'.")
    use_main = final_mode == "baseline_stable_main_gp"
    main_effect_vars = list(strict_stable_vars) if use_main else []
    gp_input_vars = list(strict_stable_vars) if use_main else list(reference_top10_vars)
    source = "baseline strict 4/4 stable vars" if use_main else "full-train Cell2-style top10 for final external validation; training-fold top10 inside internal CV"
    print(f"[FeatureSetDecision] strict_n = {strict_n}")
    print(f"[FeatureSetDecision] final_feature_set_mode = {final_mode}")
    print(f"[FeatureSetDecision] use_main_effects_for_this_run = {bool(use_main)}")
    print(f"[FeatureSetDecision] main_effect_vars = {main_effect_vars}")
    print(f"[FeatureSetDecision] gp_input_vars = {gp_input_vars}")
    print(f"[FeatureSetDecision] gp_input_vars_source = {source}")
    return {"feature_set_decision": source, "final_feature_set_mode": final_mode, "use_main_effects_for_this_run": bool(use_main), "main_effect_vars": main_effect_vars, "gp_input_vars": gp_input_vars, "gp_input_vars_source": source}


def apply_gp_param_profile(ab_params, final_feature_set_mode):
    profile_mode = str(GP_PARAM_PROFILE).lower()
    profile_name = ("main_effect_gp" if final_feature_set_mode == "baseline_stable_main_gp" else "gp_only") if profile_mode == "auto" else profile_mode
    if profile_name == "main_effect_gp":
        overrides = dict(MAIN_EFFECT_GP_PARAMS)
    elif profile_name == "gp_only":
        overrides = dict(GP_ONLY_PARAMS)
    else:
        raise ValueError("GP_PARAM_PROFILE must be 'auto', 'main_effect_gp', or 'gp_only'.")
    out = dict(ab_params)
    out.update(overrides)
    out["ACTUAL_GP_PARAM_PROFILE"] = profile_name
    out["ACTUAL_GP_PARAMS"] = dict(overrides)
    globals()["ACTUAL_GP_PARAM_PROFILE"] = profile_name
    globals()["ACTUAL_GP_PARAMS"] = dict(overrides)
    print(f"[GPParamProfile] selected profile = {profile_name}")
    print(f"[GPParamProfile] params = {overrides}")
    return out


def _top10_continuous_vars_from_training(df_train_raw, top_k=TOP_K):
    enc_tr, prep = fit_preprocess(df_train_raw, cat_candidates=CATEGORICAL_CANDIDATES)
    candidate_cols = [c for c in prep.get("cont_cols", []) if c in enc_tr.columns]
    enc_cont = enc_tr[[TIME_COL, EVENT_COL] + candidate_cols].copy()
    top_vars, uni_df = univariate_topk(enc_cont, top_k=int(top_k))
    return top_vars[:int(top_k)], uni_df


def _select_ab_feature_sets(df_train_raw, seed_offset=0, inner_folds_for_varsel=INNER_FOLDS):
    final_mode = str(globals().get("FINAL_FEATURE_SET_MODE") or globals().get("FEATURE_SET_MODE", "auto")).lower()
    if final_mode == "auto":
        strict_n = len(globals().get("BASELINE_STABLE_VARS", []) or [])
        final_mode = "baseline_stable_main_gp" if strict_n >= int(BASELINE_STABLE_MIN_STRICT_N) else "top10_gp_only"
    stable_freq_df = pd.DataFrame(globals().get("BASELINE_STABLE_VAR_FREQUENCY", []) or [])
    freq_df = pd.DataFrame()
    inner_selected_df = pd.DataFrame()
    stable_vars = list(globals().get("BASELINE_STABLE_VARS", []) or [])
    if final_mode == "baseline_stable_main_gp":
        if len(stable_vars) < int(BASELINE_STABLE_MIN_STRICT_N):
            raise RuntimeError("Not enough strict 4/4 baseline stable vars for baseline_stable_main_gp mode.")
        gp_input_vars = list(stable_vars)
        main_effect_vars = list(stable_vars)
        gp_source = "baseline strict 4/4 stable vars"
    elif final_mode == "top10_gp_only":
        gp_input_vars, uni_df = _top10_continuous_vars_from_training(df_train_raw, top_k=TOP_K)
        main_effect_vars = []
        freq_df = uni_df
        gp_source = "training-fold top10" if seed_offset != 909 else "full-train Cell2-style top10"
    else:
        raise ValueError(f"Unknown final feature set mode: {final_mode!r}")
    main_effect_indices = _main_effect_indices_from_final(gp_input_vars, main_effect_vars)
    return {"feature_set_mode": final_mode, "final_feature_set_mode": final_mode, "feature_set_decision": globals().get("FEATURE_SET_DECISION", ""), "use_main_effects_for_this_run": bool(len(main_effect_vars)), "final_vars": list(gp_input_vars), "gp_input_vars": list(gp_input_vars), "gp_input_vars_source": gp_source, "main_effect_vars": list(main_effect_vars), "main_effect_indices": main_effect_indices, "stable_vars": list(stable_vars), "stable_freq_df": stable_freq_df, "stable_min_freq_used": int(OUTER_FOLDS), "stable_fold_df": pd.DataFrame(), "freq_df": freq_df, "inner_selected_df": inner_selected_df}

def reproduce_cell2_topk_main_only(df_train_raw, df_external_raw, top_k=TOP_K):
    train_enc, prep = fit_preprocess(df_train_raw, cat_candidates=CATEGORICAL_CANDIDATES)
    external_enc = transform_preprocess(df_external_raw, prep)
    candidate_cols = [c for c in prep.get("cont_cols", []) if c in train_enc.columns]
    enc_cont = train_enc[[TIME_COL, EVENT_COL] + candidate_cols].copy()
    selected_topk_vars, uni_df = univariate_topk(enc_cont, top_k=int(top_k))
    for col in selected_topk_vars:
        if col not in external_enc.columns:
            external_enc[col] = 0.0

    if len(selected_topk_vars) == 0:
        return {
            "selected_topk_vars": [],
            "train_cindex": np.nan,
            "external_cindex": np.nan,
            "cox_coefficients": {},
        }

    df_fit = train_enc[[TIME_COL, EVENT_COL] + selected_topk_vars].copy()
    for c in selected_topk_vars:
        df_fit[c] = pd.to_numeric(df_fit[c], errors="coerce").fillna(0.0).astype(float)
    try:
        cph = make_cox()
        cph.fit(df_fit, duration_col=TIME_COL, event_col=EVENT_COL)
        train_risk = cph.predict_partial_hazard(df_fit).values.reshape(-1)
        train_cindex = _safe_float_cindex(df_fit[TIME_COL].values, df_fit[EVENT_COL].values, train_risk)
        df_ext = external_enc[[TIME_COL, EVENT_COL] + selected_topk_vars].copy()
        for c in selected_topk_vars:
            df_ext[c] = pd.to_numeric(df_ext[c], errors="coerce").fillna(0.0).astype(float)
        external_risk = cph.predict_partial_hazard(df_ext).values.reshape(-1)
        external_cindex = _safe_float_cindex(df_ext[TIME_COL].values, df_ext[EVENT_COL].values, external_risk)
        coefs = {str(k): float(v) for k, v in cph.params_.to_dict().items()}
    except Exception:
        train_cindex = np.nan
        external_cindex = np.nan
        coefs = {}

    return {
        "selected_topk_vars": selected_topk_vars,
        "train_cindex": train_cindex,
        "external_cindex": external_cindex,
        "cox_coefficients": coefs,
    }

def _phaseb_safe_name(x):
    return re.sub(r"[^0-9A-Za-z_]+", "_", str(x)).strip("_") or "x"


def _phaseb_interaction_col(feature, basis):
    return f"PHASEB__{_phaseb_safe_name(feature)}__X__{_phaseb_safe_name(basis)}"


def phaseb_make_time_stats(T_train):
    t = np.asarray(T_train, dtype=float)
    t = np.clip(t[np.isfinite(t)], MIN_TIME, None)
    if len(t) == 0:
        t = np.asarray([MIN_TIME], dtype=float)
    logt = np.log1p(t)
    mean_logt = float(np.mean(logt))
    sd_logt = float(np.std(logt, ddof=0))
    if not np.isfinite(sd_logt) or sd_logt < STANDARDIZE_EPS:
        sd_logt = 1.0
    q = float(np.nanquantile(t, float(PHASEB_T_WINSOR_Q)))
    if not np.isfinite(q) or q <= 0:
        q = float(np.nanmax(t)) if len(t) else MIN_TIME
    tw = np.minimum(t, q)
    mean_tw = float(np.mean(tw))
    sd_tw = float(np.std(tw, ddof=0))
    if not np.isfinite(sd_tw) or sd_tw < STANDARDIZE_EPS:
        sd_tw = 1.0
    return {"mean_logt": mean_logt, "sd_logt": sd_logt, "t_winsor_q": q, "mean_t_winsor": mean_tw, "sd_t_winsor": sd_tw}


def phaseb_eval_time_basis(t, basis, time_stats):
    t = np.asarray(t, dtype=float)
    t = np.clip(t, MIN_TIME, None)
    logt = np.log1p(t)
    zlogt = (logt - float(time_stats["mean_logt"])) / float(time_stats["sd_logt"])
    tw = np.minimum(t, float(time_stats["t_winsor_q"]))
    t_scaled = (tw - float(time_stats["mean_t_winsor"])) / float(time_stats["sd_t_winsor"])
    if basis == "zlog":
        return zlogt
    if basis == "zlog2":
        return zlogt ** 2
    if basis == "zlog3":
        return zlogt ** 3
    if basis == "early3":
        return (t <= 3.0).astype(float)
    if basis == "early7":
        return (t <= 7.0).astype(float)
    if basis == "late7":
        return (t > 7.0).astype(float)
    if basis == "hinge_after3":
        z3 = (np.log1p(3.0) - float(time_stats["mean_logt"])) / float(time_stats["sd_logt"])
        return np.maximum(0.0, zlogt - z3)
    if basis == "hinge_after7":
        z7 = (np.log1p(7.0) - float(time_stats["mean_logt"])) / float(time_stats["sd_logt"])
        return np.maximum(0.0, zlogt - z7)
    if basis == "exp_decay3":
        return np.exp(-t / 3.0)
    if basis == "exp_decay7":
        return np.exp(-t / 7.0)
    if basis == "inv_decay3":
        return 1.0 / (1.0 + t / 3.0)
    if basis == "inv_decay7":
        return 1.0 / (1.0 + t / 7.0)
    if basis == "t_scaled":
        return t_scaled
    if basis == "t_scaled2":
        return t_scaled ** 2
    if basis == "t_scaled3":
        return t_scaled ** 3
    raise ValueError(f"Unknown Phase-B time basis: {basis}")


def _phaseb_infer_n(*arrays, fallback=0):
    for arr in arrays:
        if arr is None:
            continue
        try:
            return int(len(arr))
        except TypeError:
            continue
    return int(fallback)


def _phaseb_as_2d(arr, n_rows):
    if arr is None:
        return None
    a = np.asarray(arr, dtype=float)
    if a.ndim == 1:
        a = a.reshape(-1, 1)
    if a.shape[0] != int(n_rows):
        return None
    return a


def phaseb_feature_frames(M_tr, M_va, main_effect_names, F_tr, F_va, n_tr=None, n_va=None):
    main_effect_names = list(main_effect_names or [])
    n_tr = _phaseb_infer_n(M_tr, F_tr, fallback=0 if n_tr is None else n_tr)
    n_va = _phaseb_infer_n(M_va, F_va, fallback=n_tr if n_va is None else n_va)
    tr = pd.DataFrame(index=np.arange(int(n_tr)))
    va = pd.DataFrame(index=np.arange(int(n_va)))
    M_tr_arr = _phaseb_as_2d(M_tr, n_tr)
    M_va_arr = _phaseb_as_2d(M_va, n_va)
    if M_tr_arr is not None and len(main_effect_names):
        for j, name in enumerate(main_effect_names):
            if j < M_tr_arr.shape[1]:
                tr[name] = M_tr_arr[:, j]
            if M_va_arr is not None and j < M_va_arr.shape[1]:
                va[name] = M_va_arr[:, j]
    gp_names = []
    F_tr_arr = _phaseb_as_2d(F_tr, n_tr)
    F_va_arr = _phaseb_as_2d(F_va, n_va)
    if F_tr_arr is not None:
        for j in range(F_tr_arr.shape[1]):
            name = f"gp{j+1}"
            gp_names.append(name)
            tr[name] = F_tr_arr[:, j]
            if F_va_arr is not None and j < F_va_arr.shape[1]:
                va[name] = F_va_arr[:, j]
    for c in tr.columns:
        if c not in va.columns:
            va[c] = 0.0
    return tr, va.reindex(columns=tr.columns, fill_value=0.0).copy(), gp_names, {c: c for c in tr.columns}


def phaseb_feature_pool(main_violating, gp_violating, main_effect_names, gp_names):
    pool = []
    main_violating = list(main_violating or [])
    gp_violating = list(gp_violating or [])
    if bool(PHASEB_USE_PH_VIOLATING_MAIN_EFFECTS):
        pool.extend([x for x in main_violating if x in set(main_effect_names or [])])
    if bool(PHASEB_USE_PH_VIOLATING_GP_FEATURES):
        pool.extend([x for x in gp_violating if x in set(gp_names or [])])
    if bool(PHASEB_ENABLE_EXPLORATORY_STABLE_MAIN):
        pool.extend(list(main_effect_names or []))
    if bool(PHASEB_ENABLE_EXPLORATORY_ALL_GP):
        pool.extend(list(gp_names or []))
    out = []
    for x in pool:
        if x not in out:
            out.append(x)
    return out


def phaseb_build_candidates(feature_pool):
    rows = []
    for feature in list(feature_pool or []):
        for basis in list(PHASEB_TIME_BASIS_NAMES):
            rows.append({
                "name": f"PHASEB::{feature} * {basis}",
                "feature": feature,
                "basis": basis,
                "interaction_col": _phaseb_interaction_col(feature, basis),
                "type": "time_basis_interaction",
                "expr": f"{feature} * {basis}",
                "priority": int(PHASEB_BASIS_PRIORITY.get(basis, 99)),
            })
    return rows


def phaseb_build_long_df(T, E, eta_ab, feature_df, selected_candidates, time_stats):
    T = np.asarray(T, dtype=float)
    E = np.asarray(E, dtype=int)
    eta_ab = np.asarray(eta_ab, dtype=float)
    feature_df = feature_df.reset_index(drop=True).copy()
    cuts = sorted(float(x) for x in PHASEB_INTERVAL_CUTS if float(x) > 0)
    rows = []
    for i, ti in enumerate(T):
        ti = float(np.clip(ti, MIN_TIME, None)) if np.isfinite(ti) else MIN_TIME
        start = 0.0
        stops = [x for x in cuts if x < ti] + [ti]
        for stop in stops:
            stop = float(stop)
            if stop <= start:
                continue
            row = {"id": int(i), "start": float(start), "stop": stop, "event": int(E[i] == 1 and abs(stop - ti) < 1e-12), "eta_ab": float(eta_ab[i]) if np.isfinite(eta_ab[i]) else 0.0}
            for cand in selected_candidates or []:
                feat = cand["feature"]
                x = float(feature_df.loc[i, feat]) if feat in feature_df.columns and np.isfinite(feature_df.loc[i, feat]) else 0.0
                g = float(phaseb_eval_time_basis([stop], cand["basis"], time_stats)[0])
                row[cand["interaction_col"]] = x * g
            rows.append(row)
            start = stop
    return pd.DataFrame(rows)


def fit_phaseb_ctv_model(long_df, cols, penalizer=None):
    cols = [c for c in list(cols or []) if c in long_df.columns]
    keep_cols = ["id", "start", "stop", "event"] + cols
    if len(long_df) == 0 or not cols:
        return None, "empty_long_or_no_cols", []
    df = long_df[keep_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(df) < 5 or int(df["event"].sum()) < 2:
        return None, "insufficient_long_events", []
    usable_cols = []
    for c in cols:
        sd = float(pd.to_numeric(df[c], errors="coerce").std(ddof=0))
        if np.isfinite(sd) and sd > float(PHASEB_MIN_COL_STD):
            usable_cols.append(c)
    if not usable_cols:
        return None, "all_candidate_columns_low_variance", []
    try:
        ctv = CoxTimeVaryingFitter(penalizer=float(PHASEB_PENALIZER if penalizer is None else penalizer))
        ctv.fit(df[["id", "start", "stop", "event"] + usable_cols], id_col="id", start_col="start", stop_col="stop", event_col="event")
        return ctv, "ok", usable_cols
    except Exception as e:
        return None, f"fit_failed:{type(e).__name__}:{str(e)[:120]}", usable_cols


def score_phaseb_timebasis_candidates(candidates, T, E, eta_ab, feature_df, time_stats):
    rows = []
    base_long = phaseb_build_long_df(T, E, eta_ab, feature_df, [], time_stats)
    base_model, _, _ = fit_phaseb_ctv_model(base_long, ["eta_ab"], PHASEB_PENALIZER)
    base_ll = float(getattr(base_model, "log_likelihood_", np.nan)) if base_model is not None else np.nan
    for cand in candidates:
        long_df = phaseb_build_long_df(T, E, eta_ab, feature_df, [cand], time_stats)
        model, reason, _ = fit_phaseb_ctv_model(long_df, ["eta_ab", cand["interaction_col"]], PHASEB_PENALIZER)
        cand_ll = float(getattr(model, "log_likelihood_", np.nan)) if model is not None else np.nan
        coef = np.nan
        if model is not None and cand["interaction_col"] in getattr(model, "params_", pd.Series(dtype=float)).index:
            coef = float(model.params_.loc[cand["interaction_col"]])
        row = dict(cand)
        row.update({
            "train_loglik_base": base_ll,
            "train_loglik_candidate": cand_ll,
            "train_loglik_improvement": cand_ll - base_ll if np.isfinite(cand_ll) and np.isfinite(base_ll) else np.nan,
            "coefficient": coef,
            "converged": bool(model is not None),
            "reason": reason if model is None else "ok",
            "selected": False,
        })
        rows.append(row)
    return pd.DataFrame(rows)


def select_phaseb_timebasis_candidates(score_df):
    if score_df is None or len(score_df) == 0:
        return []
    df = score_df.copy()
    df = df[df["converged"].astype(bool) & np.isfinite(pd.to_numeric(df["train_loglik_improvement"], errors="coerce"))].copy()
    if len(df) == 0:
        return []
    selected = []
    for _, dfg in df.groupby("feature", sort=False):
        dfg = dfg.sort_values(["train_loglik_improvement", "priority"], ascending=[False, True]).copy()
        best_imp = float(dfg.iloc[0]["train_loglik_improvement"])
        near = dfg[np.abs(pd.to_numeric(dfg["train_loglik_improvement"], errors="coerce") - best_imp) < float(PHASEB_TIE_LOGLIK_EPS)].copy()
        if len(near):
            dfg = near.sort_values(["priority", "train_loglik_improvement"], ascending=[True, False])
        selected.extend(dfg.head(int(PHASEB_MAX_BASIS_PER_FEATURE)).to_dict(orient="records"))
    return selected


def time_dependent_cindex_phaseb(T, E, eta_ab, feature_df, selected_candidates=None, coefficients=None, time_stats=None):
    T = np.asarray(T, dtype=float)
    E = np.asarray(E, dtype=int)
    eta_ab = np.asarray(eta_ab, dtype=float)
    feature_df = feature_df.reset_index(drop=True).copy()
    coefficients = coefficients or {"eta_ab": 1.0}
    selected_candidates = selected_candidates or []
    event_idx = [i for i in range(len(T)) if E[i] == 1 and np.isfinite(T[i])]
    conc = ties = perm = 0.0
    for i in event_idx:
        ti = float(T[i])
        comparable = np.where(T > ti)[0]
        if len(comparable) == 0:
            continue
        risks = float(coefficients.get("eta_ab", 1.0)) * eta_ab.copy()
        for cand in selected_candidates:
            feat = cand.get("feature")
            if feat not in feature_df.columns:
                continue
            g = float(phaseb_eval_time_basis([ti], cand.get("basis"), time_stats)[0])
            risks = risks + float(coefficients.get(cand.get("interaction_col"), 0.0)) * pd.to_numeric(feature_df[feat], errors="coerce").fillna(0.0).values * g
        ri = risks[i]
        rj = risks[comparable]
        conc += float(np.sum(ri > rj))
        ties += float(np.sum(np.isclose(ri, rj)))
        perm += float(len(comparable))
    return float((conc + 0.5 * ties) / perm) if perm > 0 else np.nan


def phaseb_static_eta_from_ab(F_tr, T_tr, E_tr, F_apply, T_apply, E_apply, toolbox_t, M_tr=None, M_apply=None, main_effect_names=None):
    risk = predict_ab_risk_on_test(F_tr, T_tr, E_tr, F_apply, T_apply, E_apply, {}, toolbox_t, M_dev=M_tr, M_te=M_apply, main_effect_names=main_effect_names)
    risk = np.asarray(risk, dtype=float) if risk is not None else np.full(len(T_apply), np.nan)
    return np.log(np.clip(risk, MIN_TIME, None))


def run_phaseb_timebasis_pool_for_train_valid(M_tr, M_va, main_effect_names, F_tr, F_va, T_tr, E_tr, T_va, E_va, eta_tr, eta_va, main_violating, gp_violating, fold_label=""):
    feature_tr, feature_va, gp_names, _ = phaseb_feature_frames(M_tr, M_va, main_effect_names, F_tr, F_va, n_tr=len(T_tr), n_va=len(T_va))
    feature_pool = phaseb_feature_pool(main_violating, gp_violating, main_effect_names, gp_names)
    candidates = phaseb_build_candidates(feature_pool)
    candidates_labeled = [dict(c, fold=fold_label) for c in candidates]
    time_stats = phaseb_make_time_stats(T_tr)
    score_df = score_phaseb_timebasis_candidates(candidates, T_tr, E_tr, eta_tr, feature_tr, time_stats)
    selected = select_phaseb_timebasis_candidates(score_df)
    selected_cols = [c["interaction_col"] for c in selected]
    train_long = phaseb_build_long_df(T_tr, E_tr, eta_tr, feature_tr, selected, time_stats)
    final_model, final_reason, _ = fit_phaseb_ctv_model(train_long, ["eta_ab"] + selected_cols, PHASEB_PENALIZER)
    coefs = {"eta_ab": 1.0}
    if final_model is not None:
        coefs.update({str(k): float(v) for k, v in final_model.params_.to_dict().items()})
    c_ab = time_dependent_cindex_phaseb(T_va, E_va, eta_va, feature_va, [], {"eta_ab": 1.0}, time_stats)
    c_pb = time_dependent_cindex_phaseb(T_va, E_va, eta_va, feature_va, selected, coefs, time_stats) if selected and final_model is not None else c_ab
    delta = c_pb - c_ab if np.isfinite(c_pb) and np.isfinite(c_ab) else np.nan
    if len(score_df):
        score_df["fold"] = fold_label
        score_df["valid_ab_cindex"] = c_ab
        score_df["valid_phaseb_cindex"] = c_pb
        score_df["delta"] = delta
        score_df["phaseb_A_cindex"] = c_ab
        score_df["phaseb_AB_cindex"] = c_pb
        score_df["phaseb_delta_AB_minus_A"] = delta
        score_df["selected"] = score_df["interaction_col"].isin(selected_cols)
    selected_df = pd.DataFrame(selected)
    if len(selected_df):
        selected_df["fold"] = fold_label
        selected_df["valid_ab_cindex"] = c_ab
        selected_df["valid_phaseb_cindex"] = c_pb
        selected_df["delta"] = delta
        selected_df["phaseb_A_cindex"] = c_ab
        selected_df["phaseb_AB_cindex"] = c_pb
        selected_df["phaseb_delta_AB_minus_A"] = delta
    return {
        "mode": "time_basis_pool",
        "feature_pool": feature_pool,
        "raw_candidates": candidates_labeled,
        "candidate_scores": score_df,
        "selected_candidates": selected,
        "selected_candidates_df": selected_df,
        "time_stats": time_stats,
        "ctv_model": final_model,
        "ctv_fit_reason": final_reason,
        "coefficients": coefs,
        "valid_ab_cindex": c_ab,
        "valid_phaseb_cindex": c_pb,
        "delta": delta,
        "fold_summary": pd.DataFrame([{
            "fold": fold_label,
            "feature_pool": json.dumps(feature_pool, ensure_ascii=False),
            "raw_candidates": int(len(candidates)),
            "selected_candidates": json.dumps([c.get("name", "") for c in selected], ensure_ascii=False),
            "valid_ab_cindex": c_ab,
            "valid_phaseb_cindex": c_pb,
            "delta": delta,
            "phaseb_A_cindex": c_ab,
            "phaseb_AB_cindex": c_pb,
            "phaseb_delta_AB_minus_A": delta,
            "reason": final_reason,
        }]),
    }
def _fit_ab_backbone_on_train(df_train_raw, ab_params, seed_offset=0, inner_folds_for_varsel=INNER_FOLDS, collect_trajectory=False, allow_phaseB=True):
    old = _temporary_set_params(ab_params)
    try:
        set_all_seeds(SEED + seed_offset)
        feature_info = _select_ab_feature_sets(df_train_raw, seed_offset=seed_offset, inner_folds_for_varsel=inner_folds_for_varsel)
        final_vars = feature_info["final_vars"]
        if len(final_vars) < 1:
            return None
        gp_input_vars = feature_info["gp_input_vars"]
        main_effect_vars = feature_info["main_effect_vars"]
        main_effect_indices = feature_info["main_effect_indices"]

        train_enc, prep = fit_preprocess(df_train_raw, cat_candidates=CATEGORICAL_CANDIDATES)
        for col in gp_input_vars:
            if col not in train_enc.columns:
                train_enc[col] = 0.0
        X = train_enc[gp_input_vars].astype(float).values
        M = _main_matrix_from_X(X, main_effect_indices)
        T = train_enc[TIME_COL].astype(float).values
        E = train_enc[EVENT_COL].astype(int).values

        phaseA_history = []
        phaseB_history = []
        toolbox_x, _ = build_gp_toolbox_x(X.shape[1])
        toolbox_x.register("evaluate", evaluate_individual_x, toolbox_x=toolbox_x, X=X, T=T, E=E, M=M, main_effect_names=main_effect_vars)

        def _cb_a(gen, pop, hof):
            if collect_trajectory and len(hof):
                phaseA_history.append(float(hof[0].fitness.values[0]))

        hof_x = run_evolution(toolbox_x, POP_SIZE_X, N_GEN_X, CX_PB_X, MUT_PB_X, label="AB-PhaseA", on_generation_end=_cb_a)
        del toolbox_x.__dict__["evaluate"]
        best_ind_x = hof_x[0]

        best_ind_x, F, gp_filter_diag = filter_gp_trees_after_evolution(best_ind_x, toolbox_x, X, T, E)
        if F is None:
            return None
        main_violating, violating = ph_test_on_train_full_model(M, main_effect_vars, F, T, E)
        toolbox_t, _ = build_gp_toolbox_t()
        g_fixed = {}
        base_main_only = score_main_only_on_test(M, main_effect_vars, T, E, M, T, E)
        main_gp_c = _score_ab_with_gdict_on_valid(F, T, E, F, T, E, {}, toolbox_t, M_tr=M, M_va=M, main_effect_names=main_effect_vars)
        current_best = main_gp_c

        phaseb_timebasis = {}
        mode = str(globals().get("PHASEB_MODE", "time_basis_pool")).lower()
        if mode == "off" or not allow_phaseB:
            print("[Final-PhaseB-TimeBasis] disabled by internal gate" if mode == "time_basis_pool" and not allow_phaseB else "[Phase-B] off; skip Phase-B.")
        elif mode == "time_basis_pool":
            gp_names_viol = [f"gp{j+1}" for j in violating]
            eta_tr = phaseb_static_eta_from_ab(F, T, E, F, T, E, toolbox_t, M_tr=M, M_apply=M, main_effect_names=main_effect_vars)
            phaseb_timebasis = run_phaseb_timebasis_pool_for_train_valid(
                M, M, main_effect_vars, F, F, T, E, T, E, eta_tr, eta_tr,
                main_violating, gp_names_viol, fold_label="final_full_train"
            )
            current_best = float(phaseb_timebasis.get("valid_phaseb_cindex", main_gp_c))
            if collect_trajectory and np.isfinite(current_best):
                phaseB_history.append(current_best)
            print(f"[Final-PhaseB-TimeBasis] PH-violating features = {phaseb_timebasis.get('feature_pool', [])}")
            print(f"[Final-PhaseB-TimeBasis] selected_candidates = {[c.get('name', '') for c in phaseb_timebasis.get('selected_candidates', [])]}")
            print("[Final-PhaseB-TimeBasis] Phase C uses static main+GP risk; time-basis Phase B is evaluated separately.")
        else:
            for j in violating:
                local_hist = []

                def _cb_b(gen, pop, hof, _store=local_hist):
                    if collect_trajectory and len(hof):
                        g_tmp = dict(g_fixed)
                        g_tmp[j] = hof[0]
                        _store.append(_score_ab_with_gdict_on_valid(F, T, E, F, T, E, g_tmp, toolbox_t, M_tr=M, M_va=M, main_effect_names=main_effect_vars))

                toolbox_t.register("evaluate", evaluate_individual_t, toolbox_t=toolbox_t, F_train_raw=F, T_train=T, E_train=E, target_j=j)
                hof_t = run_evolution(toolbox_t, POP_SIZE_T, N_GEN_T, CX_PB_T, MUT_PB_T, label=f"AB-PhaseB-gp{j+1}", on_generation_end=_cb_b)
                del toolbox_t.__dict__["evaluate"]
                cand = dict(g_fixed)
                cand[j] = hof_t[0]
                cand_c = _score_ab_with_gdict_on_valid(F, T, E, F, T, E, cand, toolbox_t, M_tr=M, M_va=M, main_effect_names=main_effect_vars)
                delta = cand_c - current_best if np.isfinite(cand_c) and np.isfinite(current_best) else np.nan
                if np.isfinite(delta) and delta >= float(PHASEB_MIN_DELTA):
                    g_fixed[j] = hof_t[0]
                    current_best = cand_c
                    if collect_trajectory:
                        phaseB_history.extend(local_hist)
        return {
            "prep": prep,
            "feature_set_mode": feature_info["feature_set_mode"],
            "final_feature_set_mode": feature_info.get("final_feature_set_mode", feature_info["feature_set_mode"]),
            "feature_set_decision": feature_info.get("feature_set_decision", ""),
            "gp_input_vars_source": feature_info.get("gp_input_vars_source", ""),
            "use_main_effects_for_this_run": bool(feature_info.get("use_main_effects_for_this_run", False)),
            "stable_selection_mode": "not_applicable",
            "baseline_stable_vars": list(globals().get("BASELINE_STABLE_VARS", []) or []),
            "stable_vars": feature_info["stable_vars"],
            "baseline_stable_var_frequency": list(globals().get("BASELINE_STABLE_VAR_FREQUENCY", []) or []),
            "stable_var_frequency": feature_info["stable_freq_df"].to_dict(orient="records"),
            "baseline_stable_min_freq_used": globals().get("BASELINE_STABLE_MIN_FREQ_USED", np.nan),
            "stable_min_freq_used": feature_info["stable_min_freq_used"],
            "stable_fold_topk": feature_info["stable_fold_df"].to_dict(orient="records"),
            "baseline_stable_min_strict_n": int(BASELINE_STABLE_MIN_STRICT_N),
            "stable_fallback_freq": "not_applicable",
            "final_vars": final_vars,
            "gp_input_vars": gp_input_vars,
            "main_effect_vars": main_effect_vars,
            "main_effect_indices": main_effect_indices,
            "use_main_effects_for_this_run": bool(feature_info.get("use_main_effects_for_this_run", False)),
            "main_effect_k": "not_applicable",
            "phaseB_allowed_final": bool(allow_phaseB),
            "phaseB_mode": mode,
            "toolbox_x": toolbox_x,
            "best_ind_x": best_ind_x,
            "toolbox_t": toolbox_t,
            "g_fixed": g_fixed,
            "train_enc": train_enc,
            "X_train": X,
            "M_train": M,
            "F_train": F,
            "T_train": T,
            "E_train": E,
            "gp_exprs": [tree_to_named_expr(t, gp_input_vars) for t in best_ind_x],
            "gp_filter_diag": gp_filter_diag,
            "actual_n_gp": int(len(best_ind_x)),
            "actual_tree_heights": [int(t.height) for t in best_ind_x],
            "actual_tree_lengths": [int(len(t)) for t in best_ind_x],
            "phaseA_history": phaseA_history,
            "phaseB_history": phaseB_history,
            "phaseb_timebasis": phaseb_timebasis,
            "ph_violating_main_effects": main_violating,
            "ph_violating_gp_names": [f"gp{j+1}" for j in violating],
            "kept_phaseB_interactions": [c.get("interaction_col", "") for c in phaseb_timebasis.get("selected_candidates", [])] if mode == "time_basis_pool" else [f"gp{j+1}_x_gt" for j in sorted(g_fixed.keys())],
            "main_only_cindex": base_main_only,
            "main_gp_cindex": main_gp_c,
            "main_gp_phaseB_cindex": current_best,
        }
    finally:
        _restore_params(old)


def _score_phasea_individual_on_valid(individual, toolbox_x, X_tr, T_tr, E_tr, X_va, T_va, E_va,
                                      M_tr=None, M_va=None, main_effect_names=None):
    try:
        F_tr = compute_F(individual, toolbox_x, X_tr)
        F_va = compute_F(individual, toolbox_x, X_va)
        if F_tr is None or F_va is None:
            return np.nan
        return float(score_phase_a_on_test(
            F_tr, T_tr, E_tr, F_va, T_va, E_va,
            M_dev=M_tr, M_te=M_va, main_effect_names=main_effect_names
        ))
    except Exception:
        return np.nan


def _score_ab_with_gdict_on_valid(F_tr, T_tr, E_tr, F_va, T_va, E_va, g_dict, toolbox_t,
                                  M_tr=None, M_va=None, main_effect_names=None):
    try:
        risk = predict_ab_risk_on_test(
            F_tr, T_tr, E_tr, F_va, T_va, E_va, g_dict, toolbox_t,
            M_dev=M_tr, M_te=M_va, main_effect_names=main_effect_names
        )
        if risk is None:
            return np.nan
        return float(concordance_index(T_va, -np.asarray(risk, dtype=float), E_va))
    except Exception:
        return np.nan


def _fit_ab_backbone_on_train_with_valid_logging(
    df_train_raw,
    df_valid_raw,
    ab_params,
    fold_idx,
    inner_folds_for_varsel=4,
):
    old = _temporary_set_params(ab_params)
    old_print_gen = globals().get("PRINT_GEN_LOG", False)
    globals()["PRINT_GEN_LOG"] = True
    try:
        set_all_seeds(SEED + 500 + int(fold_idx))
        feature_info = _select_ab_feature_sets(df_train_raw, seed_offset=500 + int(fold_idx), inner_folds_for_varsel=inner_folds_for_varsel)
        final_vars = feature_info["final_vars"]
        if len(final_vars) < 1:
            return None, [], []

        gp_input_vars = feature_info["gp_input_vars"]
        main_effect_vars = feature_info["main_effect_vars"]
        main_effect_indices = feature_info["main_effect_indices"]

        print("\n" + "=" * 80)
        print(f"[FOLD {fold_idx}/4] Stable-vars GP-AB internal validation")
        print("=" * 80)
        print(f"  [FOLD {fold_idx}] Training samples = {len(df_train_raw)}")
        print(f"  [FOLD {fold_idx}] Validation samples = {len(df_valid_raw)}")
        print(f"  [FOLD {fold_idx}] final_feature_set_mode = {feature_info.get('final_feature_set_mode', feature_info.get('feature_set_mode'))}")
        print(f"  [FOLD {fold_idx}] use_main_effects_for_this_run = {bool(main_effect_vars)}")
        print(f"  [FOLD {fold_idx}] Main effect variables: {main_effect_vars}")
        print(f"  [FOLD {fold_idx}] GP input variables: {gp_input_vars}")
        print(f"  [FOLD {fold_idx}] GP input source: {feature_info.get('gp_input_vars_source', '')}")

        train_enc, prep = fit_preprocess(df_train_raw, cat_candidates=CATEGORICAL_CANDIDATES)
        valid_enc = transform_preprocess(df_valid_raw, prep)
        for col in gp_input_vars:
            if col not in train_enc.columns:
                train_enc[col] = 0.0
            if col not in valid_enc.columns:
                valid_enc[col] = 0.0

        X = train_enc[gp_input_vars].astype(float).values
        T = train_enc[TIME_COL].astype(float).values
        E = train_enc[EVENT_COL].astype(int).values
        Xv = valid_enc[gp_input_vars].astype(float).values
        Tv = valid_enc[TIME_COL].astype(float).values
        Ev = valid_enc[EVENT_COL].astype(int).values
        M = _main_matrix_from_X(X, main_effect_indices)
        Mv = _main_matrix_from_X(Xv, main_effect_indices)

        ab_gen_traj = []
        phasea_hof_traj = []

        toolbox_x, _ = build_gp_toolbox_x(X.shape[1])
        toolbox_x.register("evaluate", evaluate_individual_x, toolbox_x=toolbox_x, X=X, T=T, E=E, M=M, main_effect_names=main_effect_vars)

        def _cb_a(gen, pop, hof):
            fits = [float(ind.fitness.values[0]) for ind in pop if ind.fitness.valid]
            train_avg = float(np.mean(fits)) if len(fits) else np.nan
            train_max = float(np.max(fits)) if len(fits) else np.nan
            hof_fit = float(hof[0].fitness.values[0]) if len(hof) else np.nan
            best_gen_ind = max(pop, key=lambda ind: ind.fitness.values[0]) if len(pop) else None
            c_best = _score_phasea_individual_on_valid(best_gen_ind, toolbox_x, X, T, E, Xv, Tv, Ev, M_tr=M, M_va=Mv, main_effect_names=main_effect_vars) if best_gen_ind is not None else np.nan
            c_hof = _score_phasea_individual_on_valid(hof[0], toolbox_x, X, T, E, Xv, Tv, Ev, M_tr=M, M_va=Mv, main_effect_names=main_effect_vars) if len(hof) else np.nan
            phasea_hof_traj.append({"fold": int(fold_idx), "generation": int(gen), "valid_hof_cindex": c_hof})
            ab_gen_traj.append({
                "fold": int(fold_idx),
                "phase": "A",
                "gp_name": "",
                "generation": int(gen),
                "train_avg_fit": train_avg,
                "train_max_fit": train_max,
                "hof_fit": hof_fit,
                "valid_bestgen_cindex": c_best,
                "valid_hof_cindex": c_hof,
            })
            print(f"  [FOLD {fold_idx}] Phase-A Gen {int(gen):02d} VALID C-index | bestGen={c_best:.4f} | hof={c_hof:.4f}")

        hof_x = run_evolution(
            toolbox_x,
            POP_SIZE_X,
            N_GEN_X,
            CX_PB_X,
            MUT_PB_X,
            label=f"Phase-A(fold{fold_idx})",
            on_generation_end=_cb_a
        )
        del toolbox_x.__dict__["evaluate"]
        best_ind_x = hof_x[0]

        best_ind_x, F, gp_filter_diag = filter_gp_trees_after_evolution(best_ind_x, toolbox_x, X, T, E)
        if F is None:
            return None, ab_gen_traj, phasea_hof_traj

        Fv = compute_F(best_ind_x, toolbox_x, Xv)
        print(f"[FOLD {fold_idx}] actual_n_gp after GP post-filter: {int(len(best_ind_x))}")
        main_violating, violating = ph_test_on_train_full_model(M, main_effect_vars, F, T, E)
        gp_names = [f"gp{j+1}" for j in violating]
        print(f"[FOLD {fold_idx}] PH violating main effects: {main_violating}")
        print(f"[FOLD {fold_idx}] PH violating GP names: {gp_names}")
        print(f"[FOLD {fold_idx}] Will try Phase-B interactions: {[f'gp{j+1}_x_gt' for j in violating]}")

        toolbox_t, _ = build_gp_toolbox_t()
        g_fixed = {}
        phaseB_history = []
        main_only_c = score_main_only_on_test(M, main_effect_vars, T, E, Mv, Tv, Ev)
        main_gp_c = _score_ab_with_gdict_on_valid(F, T, E, Fv, Tv, Ev, {}, toolbox_t, M_tr=M, M_va=Mv, main_effect_names=main_effect_vars)
        current_best = main_gp_c

        phaseb_timebasis = {}
        mode = str(globals().get("PHASEB_MODE", "time_basis_pool")).lower()
        if mode == "off":
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] PHASEB_MODE=off; skip Phase-B.")
        elif mode == "time_basis_pool":
            eta_tr = phaseb_static_eta_from_ab(F, T, E, F, T, E, toolbox_t, M_tr=M, M_apply=M, main_effect_names=main_effect_vars)
            eta_va = phaseb_static_eta_from_ab(F, T, E, Fv, Tv, Ev, toolbox_t, M_tr=M, M_apply=Mv, main_effect_names=main_effect_vars)
            phaseb_timebasis = run_phaseb_timebasis_pool_for_train_valid(
                M, Mv, main_effect_vars, F, Fv, T, E, Tv, Ev, eta_tr, eta_va,
                main_violating, gp_names, fold_label=int(fold_idx)
            )
            current_best = float(phaseb_timebasis.get("valid_phaseb_cindex", main_gp_c))
            phaseB_history.append(current_best)
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] PH-violating main effects = {main_violating}")
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] PH-violating GP features = {gp_names}")
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] feature_pool = {phaseb_timebasis.get('feature_pool', [])}")
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] raw_candidates = {len(phaseb_timebasis.get('raw_candidates', []))}")
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] selected_candidates = {[c.get('name', '') for c in phaseb_timebasis.get('selected_candidates', [])]}")
            for cand in phaseb_timebasis.get("selected_candidates", []):
                print(f"  [PhaseB-TimeBasis][Fold {fold_idx}] feature={cand.get('feature')} | basis={cand.get('basis')} | expr={cand.get('expr')} | interaction_col={cand.get('interaction_col')} | priority={cand.get('priority')} | train_loglik_improvement={float(cand.get('train_loglik_improvement', np.nan)):.4f} | coefficient={float(cand.get('coefficient', np.nan)):.4f}")
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] A valid C-index = {phaseb_timebasis.get('valid_ab_cindex', np.nan):.4f}")
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] AB valid C-index = {phaseb_timebasis.get('valid_phaseb_cindex', np.nan):.4f}")
            print(f"[PhaseB-TimeBasis][Fold {fold_idx}] delta_AB_minus_A = {phaseb_timebasis.get('delta', np.nan):.4f}")
        else:
            for j in violating:
                print("\n" + "-" * 80)
                print(f"[FOLD {fold_idx}] Phase-B evolve time function for gp{j+1}, interaction gp{j+1}_x_gt")
                print("-" * 80)

                def _cb_b(gen, pop, hof):
                    fits = [float(ind.fitness.values[0]) for ind in pop if ind.fitness.valid]
                    train_avg = float(np.mean(fits)) if len(fits) else np.nan
                    train_max = float(np.max(fits)) if len(fits) else np.nan
                    hof_fit = float(hof[0].fitness.values[0]) if len(hof) else np.nan
                    best_gen_ind = max(pop, key=lambda ind: ind.fitness.values[0]) if len(pop) else None
                    g_best = dict(g_fixed)
                    if best_gen_ind is not None:
                        g_best[j] = best_gen_ind
                    g_hof = dict(g_fixed)
                    if len(hof):
                        g_hof[j] = hof[0]
                    c_best = _score_ab_with_gdict_on_valid(F, T, E, Fv, Tv, Ev, g_best, toolbox_t, M_tr=M, M_va=Mv, main_effect_names=main_effect_vars)
                    c_hof = _score_ab_with_gdict_on_valid(F, T, E, Fv, Tv, Ev, g_hof, toolbox_t, M_tr=M, M_va=Mv, main_effect_names=main_effect_vars)
                    phaseB_history.append(c_hof)
                    ab_gen_traj.append({
                        "fold": int(fold_idx),
                        "phase": "B",
                        "gp_name": f"gp{j+1}",
                        "generation": int(gen),
                        "train_avg_fit": train_avg,
                        "train_max_fit": train_max,
                        "hof_fit": hof_fit,
                        "valid_bestgen_cindex": c_best,
                        "valid_hof_cindex": c_hof,
                    })
                    print(f"  [FOLD {fold_idx}] Phase-AB(gp{j+1}) Gen {int(gen):02d} VALID C-index | bestGen={c_best:.4f} | hof={c_hof:.4f}")

                toolbox_t.register("evaluate", evaluate_individual_t, toolbox_t=toolbox_t, F_train_raw=F, T_train=T, E_train=E, target_j=j)
                hof_t = run_evolution(
                    toolbox_t,
                    POP_SIZE_T,
                    N_GEN_T,
                    CX_PB_T,
                    MUT_PB_T,
                    label=f"Phase-B(fold{fold_idx},gp{j+1})",
                    on_generation_end=_cb_b
                )
                del toolbox_t.__dict__["evaluate"]
                candidate_g = dict(g_fixed)
                candidate_g[j] = hof_t[0]
                candidate_c = _score_ab_with_gdict_on_valid(F, T, E, Fv, Tv, Ev, candidate_g, toolbox_t, M_tr=M, M_va=Mv, main_effect_names=main_effect_vars)
                delta = candidate_c - current_best if np.isfinite(candidate_c) and np.isfinite(current_best) else np.nan
                if np.isfinite(delta) and delta >= float(PHASEB_MIN_DELTA):
                    g_fixed[j] = hof_t[0]
                    current_best = candidate_c
                    print(f"[Phase-B] Keep gp{j+1}_x_gt because delta={delta:.4f}")
                else:
                    print(f"[Phase-B] Skip gp{j+1}_x_gt because delta={delta if np.isfinite(delta) else np.nan:.4f} < PHASEB_MIN_DELTA={PHASEB_MIN_DELTA}")
        if mode == "time_basis_pool":
            c_ab_final = current_best
            kept = [c.get("interaction_col", "") for c in phaseb_timebasis.get("selected_candidates", [])]
        else:
            c_ab_final = _score_ab_with_gdict_on_valid(F, T, E, Fv, Tv, Ev, g_fixed, toolbox_t, M_tr=M, M_va=Mv, main_effect_names=main_effect_vars)
            kept = [f"gp{j+1}_x_gt" for j in sorted(g_fixed.keys())]
        print(f"[FOLD {fold_idx}] Kept Phase-B interactions: {kept}")
        print(f"[FOLD {fold_idx}] main-only VALID C-index = {main_only_c:.4f}")
        print(f"[FOLD {fold_idx}] A/main+GP VALID C-index = {main_gp_c:.4f}")
        print(f"[Phase A][Fold {fold_idx}] A_static valid C-index = {main_gp_c:.4f}")
        print(f"[FOLD {fold_idx}] AB/main+GP+PhaseB VALID C-index = {c_ab_final:.4f}")
        print(f"[PhaseB-TimeBasis][Fold {fold_idx}] AB_timevarying valid C-index = {c_ab_final:.4f}")
        print("\n" + "=" * 80)
        print(f"[FOLD {fold_idx}] AB diagnostic VALID C-index = {c_ab_final:.4f}")
        print("=" * 80)

        model = {
            "prep": prep,
            "feature_set_mode": feature_info["feature_set_mode"],
            "final_feature_set_mode": feature_info.get("final_feature_set_mode", feature_info["feature_set_mode"]),
            "feature_set_decision": feature_info.get("feature_set_decision", ""),
            "gp_input_vars_source": feature_info.get("gp_input_vars_source", ""),
            "use_main_effects_for_this_run": bool(feature_info.get("use_main_effects_for_this_run", False)),
            "stable_selection_mode": "not_applicable",
            "baseline_stable_vars": list(globals().get("BASELINE_STABLE_VARS", []) or []),
            "stable_vars": feature_info["stable_vars"],
            "baseline_stable_var_frequency": list(globals().get("BASELINE_STABLE_VAR_FREQUENCY", []) or []),
            "stable_var_frequency": feature_info["stable_freq_df"].to_dict(orient="records"),
            "baseline_stable_min_freq_used": globals().get("BASELINE_STABLE_MIN_FREQ_USED", np.nan),
            "stable_min_freq_used": feature_info["stable_min_freq_used"],
            "stable_fold_topk": feature_info["stable_fold_df"].to_dict(orient="records"),
            "baseline_stable_min_strict_n": int(BASELINE_STABLE_MIN_STRICT_N),
            "stable_fallback_freq": "not_applicable",
            "final_vars": final_vars,
            "gp_input_vars": gp_input_vars,
            "main_effect_vars": main_effect_vars,
            "main_effect_indices": main_effect_indices,
            "use_main_effects_for_this_run": bool(feature_info.get("use_main_effects_for_this_run", False)),
            "main_effect_k": "not_applicable",
            "phaseB_mode": mode,
            "toolbox_x": toolbox_x,
            "best_ind_x": best_ind_x,
            "toolbox_t": toolbox_t,
            "g_fixed": g_fixed,
            "train_enc": train_enc,
            "X_train": X,
            "M_train": M,
            "F_train": F,
            "T_train": T,
            "E_train": E,
            "gp_exprs": [tree_to_named_expr(t, gp_input_vars) for t in best_ind_x],
            "gp_filter_diag": gp_filter_diag,
            "actual_n_gp": int(len(best_ind_x)),
            "actual_tree_heights": [int(t.height) for t in best_ind_x],
            "actual_tree_lengths": [int(len(t)) for t in best_ind_x],
            "phaseA_history": [x["valid_hof_cindex"] for x in phasea_hof_traj],
            "phaseB_history": list(phaseB_history),
            "phaseb_timebasis": phaseb_timebasis,
            "ab_valid_cindex": c_ab_final,
            "main_only_cindex": main_only_c,
            "main_gp_cindex": main_gp_c,
            "main_gp_phaseB_cindex": c_ab_final,
            "ph_violating_main_effects": main_violating,
            "ph_violating_gp_names": gp_names,
            "kept_phaseB_interactions": kept,
            "n_time_interactions": int(len(kept)),
        }
        return model, ab_gen_traj, phasea_hof_traj
    finally:
        globals()["PRINT_GEN_LOG"] = old_print_gen
        _restore_params(old)


def _ab_prepare_apply(model_ab, df_raw):
    prep = model_ab["prep"]
    df_enc = transform_preprocess(df_raw, prep)
    for col in model_ab["final_vars"]:
        if col not in df_enc.columns:
            df_enc[col] = 0.0
    X = df_enc[model_ab["final_vars"]].astype(float).values
    T = df_enc[TIME_COL].astype(float).values
    E = df_enc[EVENT_COL].astype(int).values
    M = _main_matrix_from_X(X, model_ab.get("main_effect_indices", []))
    F = compute_F(model_ab["best_ind_x"], model_ab["toolbox_x"], X)
    return {"df_enc": df_enc, "X": X, "M": M, "F": F, "T": T, "E": E}


def _score_ab_on_validation(model_ab, df_valid_raw):
    va = _ab_prepare_apply(model_ab, df_valid_raw)
    if va["F"] is None:
        return np.nan, va, None
    risk = predict_ab_risk_on_test(
        model_ab["F_train"], model_ab["T_train"], model_ab["E_train"],
        va["F"], va["T"], va["E"],
        model_ab["g_fixed"], model_ab["toolbox_t"],
        M_dev=model_ab.get("M_train"), M_te=va.get("M"), main_effect_names=model_ab.get("main_effect_vars", [])
    )
    if risk is None:
        return np.nan, va, None
    cidx = float(concordance_index(va["T"], -risk, va["E"]))
    return cidx, va, risk


def fit_ab_cox_coefficients_for_reporting(model_ab):
    try:
        F = model_ab.get("F_train")
        M = model_ab.get("M_train")
        T = model_ab.get("T_train")
        E = model_ab.get("E_train")
        g_fixed = model_ab.get("g_fixed", {})
        toolbox_t = model_ab.get("toolbox_t")
        main_effect_names = model_ab.get("main_effect_vars", [])

        if F is None:
            return {}

        vi = sorted(g_fixed.keys())
        logt = np.log(np.clip(T, MIN_TIME, None))
        mat, cols, _, _, _ = build_cox_matrix(
            F,
            T,
            vi,
            g_fixed,
            toolbox_t,
            M=M,
            main_effect_names=main_effect_names,
            sc_F=None,
            g_mus=None,
            sc_I_list=None,
            logt_train_ref=logt,
            logt_apply=logt
        )

        df = pd.DataFrame(mat, columns=cols)
        df[TIME_COL] = T
        df[EVENT_COL] = E

        cph = make_cox()
        cph.fit(df, duration_col=TIME_COL, event_col=EVENT_COL)

        return {
            str(k): float(v)
            for k, v in cph.params_.to_dict().items()
        }

    except Exception:
        return {}


def _phasec_seeded_safe_name(x):
    return re.sub(r"[^0-9A-Za-z_]+", "_", str(x)).strip("_") or "x"


def _phasec_seeded_interaction_col(a, b):
    return f"PHASEC__{_phasec_seeded_safe_name(a)}__X__{_phasec_seeded_safe_name(b)}"


def _phasec_is_binary_col(df, col):
    if col not in df.columns:
        return False
    vals = pd.to_numeric(df[col], errors="coerce").dropna().unique()
    return len(vals) > 0 and set(np.asarray(vals, dtype=float)).issubset({0.0, 1.0})

def _phasec_mode_is_stgp_and():
    return str(globals().get("PHASEC_MODE", "stgp_and")).lower() == "stgp_and"


def phasec_stgp_and_log_policy_once():
    print("[PhaseC-STGP-AND] primitives = AND only")
    print("[PhaseC-STGP-AND] OR disabled")
    print("[PhaseC-STGP-AND] NOT disabled")
    print("[PhaseC-STGP-AND] heredity = True")
    print("[PhaseC-STGP-AND] initialization = 100% random legal")
    print("[PhaseC-STGP-AND] allowed_rules = cat_binary_AND, binary_binary_AND")
    print("[PhaseC-STGP-AND] disallowed_rules = cat_cat_AND, same_factor_AND, three_way_AND")


def build_phasec_term_metadata(df_enc, prep=None):
    term_meta = {}
    for c in df_enc.columns:
        if c in (TIME_COL, EVENT_COL):
            continue
        s = str(c)
        if any(str(k) in s for k in PHASEC_EXCLUDE_KEYWORDS):
            continue
        if not _phasec_is_binary_col(df_enc, c):
            continue
        if s.startswith("gender_"):
            term_meta[c] = {"kind": "cat_dummy", "factor": "gender", "level": s[len("gender_"):]}
        elif s.startswith("admission_type_"):
            term_meta[c] = {"kind": "cat_dummy", "factor": "admission_type", "level": s[len("admission_type_"):]}
        elif s.startswith("ethnicity_"):
            term_meta[c] = {"kind": "cat_dummy", "factor": "ethnicity", "level": s[len("ethnicity_"):]}
    for c in PHASEC_BINARY_COLS:
        if c in df_enc.columns and _phasec_is_binary_col(df_enc, c):
            term_meta[c] = {"kind": "binary", "factor": str(c), "level": "1"}
    return term_meta



def phasec_stgp_and_source_columns(df_enc, prep=None):
    term_meta = build_phasec_term_metadata(df_enc, prep)
    cat_cols = [c for c, m in term_meta.items() if m.get("kind") == "cat_dummy"]
    bin_cols = [c for c, m in term_meta.items() if m.get("kind") == "binary"]
    return sorted(dict.fromkeys(cat_cols)), sorted(dict.fromkeys(bin_cols))


def phasec_seeded_source_columns(df_enc, prep=None):
    # legacy only; not used by the STGP-AND main path
    if not _phasec_mode_is_stgp_and():
        return _phasec_legacy_seeded_source_columns(df_enc, prep)
    return phasec_stgp_and_source_columns(df_enc, prep)



def phasec_is_semantically_valid_and(term_a, term_b, term_meta):
    if term_a == term_b:
        return False, "semantic_invalid::self_and"
    if term_a not in term_meta or term_b not in term_meta:
        return False, "semantic_invalid::missing_term_meta"
    ma, mb = term_meta[term_a], term_meta[term_b]
    ka, kb = ma.get("kind"), mb.get("kind")
    if ka == "cat_dummy" and kb == "cat_dummy":
        if ma.get("factor") == mb.get("factor"):
            return False, "semantic_invalid::same_factor"
        if not bool(PHASEC_STGP_ALLOW_CAT_CAT_AND):
            return False, "semantic_invalid::cat_cat_disallowed"
    if {ka, kb} == {"cat_dummy", "binary"}:
        return True, "ok"
    if ka == "binary" and kb == "binary":
        return True, "ok"
    return False, "semantic_invalid::unsupported_kind"


def _phasec_stgp_and_col(a, b):
    return f"PHASEC_AND__{_phasec_seeded_safe_name(a)}__AND__{_phasec_seeded_safe_name(b)}"


def _phasec_order_pair(a, b):
    return tuple(sorted([str(a), str(b)]))


def _phasec_make_stgp_main_candidate(a, term_meta=None):
    return {
        "name": f"MAIN::{a}", "candidate_name": f"MAIN::{a}", "type": "main",
        "operator": "main", "main_effects": [a], "interaction": None,
        "interaction_pair": None, "cols_to_add": [a], "expr": str(a),
        "complexity": 1, "leaf_terms": [a], "semantic_valid": True,
        "semantic_reason": "ok",
    }


def _phasec_make_stgp_and_candidate(a, b, term_meta):
    a, b = _phasec_order_pair(a, b)
    ok, reason = phasec_is_semantically_valid_and(a, b, term_meta)
    inter = _phasec_stgp_and_col(a, b)
    return {
        "name": f"AND::{a} AND {b}",
        "candidate_name": f"AND::{a} AND {b}",
        "type": "and_rule", "operator": "AND",
        "main_effects": [a, b], "interaction": inter,
        "interaction_pair": [a, b], "cols_to_add": [a, b, inter],
        "expr": f"AND({a}, {b})", "complexity": 3,
        "leaf_terms": [a, b], "semantic_valid": bool(ok),
        "semantic_reason": reason,
    }


def phasec_stgp_and_build_rule_universe(df_enc, prep=None):
    term_meta = build_phasec_term_metadata(df_enc, prep)
    cat_cols, bin_cols = phasec_stgp_and_source_columns(df_enc, prep)
    rows = [_phasec_make_stgp_main_candidate(c, term_meta) for c in (cat_cols + bin_cols)]
    seen = {phasec_candidate_signature(c) for c in rows}
    for cat in cat_cols:
        for b in bin_cols:
            cand = _phasec_make_stgp_and_candidate(cat, b, term_meta)
            sig = phasec_candidate_signature(cand)
            if sig not in seen:
                seen.add(sig)
                rows.append(cand)
    for i, a in enumerate(bin_cols):
        for b in bin_cols[i + 1:]:
            cand = _phasec_make_stgp_and_candidate(a, b, term_meta)
            sig = phasec_candidate_signature(cand)
            if sig not in seen:
                seen.add(sig)
                rows.append(cand)
    return rows


def phasec_seeded_build_candidate_pool(df_enc, prep=None):
    # legacy only; not used by the STGP-AND main path
    if not _phasec_mode_is_stgp_and():
        return _phasec_legacy_seeded_build_candidate_pool(df_enc, prep)
    return phasec_stgp_and_build_rule_universe(df_enc, prep)



def _phasec_support_for_and(df_enc, E, a, b):
    if a not in df_enc.columns or b not in df_enc.columns:
        return 0, 0
    xa = pd.to_numeric(df_enc[a], errors="coerce").fillna(0.0).astype(float).values
    xb = pd.to_numeric(df_enc[b], errors="coerce").fillna(0.0).astype(float).values
    mask = (xa > 0) & (xb > 0)
    return int(mask.sum()), int(np.asarray(E, dtype=int)[mask].sum())


def phasec_stgp_precompute_support(df_enc, E, term_meta, candidates=None):
    E_arr = np.asarray(E, dtype=int)
    term_values = {}
    main_support = {}
    for col in term_meta:
        if col not in df_enc.columns:
            continue
        x = pd.to_numeric(df_enc[col], errors="coerce").fillna(0.0).astype(float).values
        xb = x > 0
        term_values[col] = xb
        main_support[col] = {"n": int(xb.sum()), "events": int(E_arr[xb].sum())}
    and_support = {}
    pair_set = set()
    for cand in candidates or []:
        if cand.get("operator") == "AND" and cand.get("interaction_pair"):
            pair = tuple(_phasec_order_pair(*list(cand.get("interaction_pair"))[:2]))
            pair_set.add(pair)
    for a, b in sorted(pair_set):
        xa = term_values.get(a)
        xb = term_values.get(b)
        if xa is None or xb is None:
            and_support[(a, b)] = {"n": 0, "events": 0}
            continue
        mask = xa & xb
        and_support[(a, b)] = {"n": int(mask.sum()), "events": int(E_arr[mask].sum())}
    return {"term_values": term_values, "main_support": main_support, "and_support": and_support}


def _phasec_simplify_drop_reason(reason):
    if reason is None:
        return ["unknown"]
    parts = []
    for item in str(reason).split(";"):
        item = item.strip()
        if not item:
            continue
        parts.append(item.split("::", 1)[0] if "::" in item else item)
    return parts or ["unknown"]


def phasec_stgp_filter_candidates(candidates, df_enc, E, term_meta=None, support_context=None, fold_label=""):
    term_meta = term_meta or build_phasec_term_metadata(df_enc)
    support_context = support_context or phasec_stgp_precompute_support(df_enc, E, term_meta, candidates)
    kept, dropped = [], []
    for cand in candidates:
        c = dict(cand)
        reasons = []
        main_n, main_ev = {}, {}
        leaf_terms = list(c.get("leaf_terms", c.get("main_effects", [])) or [])
        if len(leaf_terms) > int(PHASEC_STGP_MAX_TERMINALS_PER_RULE):
            reasons.append("semantic_invalid::too_many_terms")
        if c.get("operator") == "AND":
            pair = list(c.get("interaction_pair", []) or leaf_terms)
            if len(pair) != 2:
                reasons.append("semantic_invalid::too_many_terms")
            else:
                ok, reason = phasec_is_semantically_valid_and(pair[0], pair[1], term_meta)
                c["semantic_valid"] = bool(ok)
                c["semantic_reason"] = reason
                if not ok:
                    reasons.append(reason)
        elif c.get("operator") != "main":
            reasons.append("semantic_invalid::operator_not_allowed")
        for m in c.get("main_effects", []) or []:
            sup = support_context.get("main_support", {}).get(m, {"n": 0, "events": 0})
            n, ev = int(sup.get("n", 0)), int(sup.get("events", 0))
            if m not in term_meta:
                reasons.append(f"missing_main::{m}")
            if n < int(PHASEC_MIN_MAIN_N):
                reasons.append(f"low_main_n::{m}")
            if ev < int(PHASEC_MIN_MAIN_EVENTS):
                reasons.append(f"low_main_events::{m}")
            main_n[m], main_ev[m] = n, ev
        inter_n = inter_ev = np.nan
        if c.get("operator") == "AND" and c.get("interaction_pair"):
            a, b = _phasec_order_pair(*list(c.get("interaction_pair"))[:2])
            sup = support_context.get("and_support", {}).get((a, b), {"n": 0, "events": 0})
            inter_n, inter_ev = int(sup.get("n", 0)), int(sup.get("events", 0))
            if inter_n < int(PHASEC_MIN_INTER_N):
                reasons.append("low_inter_n")
            if inter_ev < int(PHASEC_MIN_INTER_EVENTS):
                reasons.append("low_inter_events")
        c.update({
            "fold": fold_label,
            "support_main_n": json.dumps(main_n, ensure_ascii=False),
            "support_main_events": json.dumps(main_ev, ensure_ascii=False),
            "support_inter_n": inter_n,
            "support_inter_events": inter_ev,
            "reason": ";".join(dict.fromkeys(reasons)) if reasons else "ok",
        })
        (dropped if reasons else kept).append(c)
    return kept, dropped


def phasec_seeded_fit_cox(X_tr, T_tr, E_tr):
    df = X_tr.replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(df) < 5 or int(np.asarray(E_tr, dtype=int).sum()) < 2:
        return None, "insufficient_events"
    df[TIME_COL] = np.asarray(T_tr, dtype=float)[df.index]
    df[EVENT_COL] = np.asarray(E_tr, dtype=int)[df.index]
    try:
        cph = CoxPHFitter(penalizer=float(PHASEC_PENALIZER), l1_ratio=float(PHASEC_L1_RATIO))
        cph.fit(df, duration_col=TIME_COL, event_col=EVENT_COL)
        return cph, "ok"
    except Exception as e:
        return None, f"fit_failed::{type(e).__name__}:{str(e)[:120]}"


def phasec_stgp_fit_base_once(df_enc, T, E, base_risk):
    eta = np.log(np.clip(np.asarray(base_risk, dtype=float), 1e-12, None))
    base_df = pd.DataFrame({"eta_ab": eta}, index=np.arange(len(df_enc)))
    base_model, base_reason = phasec_seeded_fit_cox(base_df, T, E)
    base_ll = float(getattr(base_model, "log_likelihood_", np.nan)) if base_model is not None else np.nan
    return {
        "base_model": base_model,
        "base_reason": base_reason,
        "base_ll": base_ll,
        "base_df": base_df,
        "T": np.asarray(T, dtype=float),
        "E": np.asarray(E, dtype=int),
        "df_enc": df_enc.reset_index(drop=True),
        "eta_col_name": "eta_ab",
        "base_fit_once": True,
        "candidate_fit_calls": 0,
        "cache_hits": 0,
    }


def phasec_stgp_add_candidate_to_design(base_df, candidate, support_context):
    X = base_df.copy()
    term_values = support_context.get("term_values", {})
    for col in candidate.get("main_effects", []) or []:
        v = term_values.get(col)
        if v is None:
            return X, f"missing_main::{col}"
        if col not in X.columns:
            X[col] = np.asarray(v, dtype=float)
    inter = candidate.get("interaction")
    pair = candidate.get("interaction_pair")
    if inter and pair:
        a, b = pair
        va = term_values.get(a)
        vb = term_values.get(b)
        if va is None or vb is None:
            return X, f"missing_inter_pair::{a},{b}"
        if inter not in X.columns:
            X[inter] = (np.asarray(va, dtype=bool) & np.asarray(vb, dtype=bool)).astype(float)
    return X, "ok"


def phasec_stgp_and_evaluate_candidate(candidate, base_context, fitness_cache, phasec_context=None):
    sig = phasec_candidate_signature(candidate)
    if sig in fitness_cache:
        base_context["cache_hits"] = int(base_context.get("cache_hits", 0)) + 1
        return dict(fitness_cache[sig])
    row = dict(candidate)
    base_ll = float(base_context.get("base_ll", np.nan))
    X, reason = phasec_stgp_add_candidate_to_design(base_context["base_df"], candidate, base_context["support_context"])
    zero_var = False
    if reason == "ok":
        zero_var = any(float(np.nanstd(pd.to_numeric(X[c], errors="coerce").fillna(0.0).values)) <= 0.0 for c in X.columns if c != base_context.get("eta_col_name", "eta_ab"))
    if zero_var:
        model, fit_reason = None, "zero_variance"
    elif reason == "ok":
        base_context["candidate_fit_calls"] = int(base_context.get("candidate_fit_calls", 0)) + 1
        model, fit_reason = phasec_seeded_fit_cox(X, base_context["T"], base_context["E"])
    else:
        model, fit_reason = None, reason
    cand_ll = float(getattr(model, "log_likelihood_", np.nan)) if model is not None else np.nan
    coefs = {str(k): float(v) for k, v in model.params_.to_dict().items()} if model is not None else {}
    if model is not None and (not all(np.isfinite(list(coefs.values()))) or any(abs(v) > 50 for v in coefs.values())):
        model = None
        fit_reason = "invalid_coefficients"
        cand_ll = np.nan
        coefs = {}
    imp = cand_ll - base_ll if np.isfinite(cand_ll) and np.isfinite(base_ll) else np.nan
    complexity = int(candidate.get("complexity", 1))
    row.update({
        "train_loglik_base": base_ll,
        "train_loglik_candidate": cand_ll,
        "train_loglik_improvement": imp,
        "complexity": complexity,
        "fitness": imp - float(PHASEC_COMPLEXITY_LAMBDA) * complexity if np.isfinite(imp) else -np.inf,
        "converged": bool(model is not None),
        "reason": fit_reason if model is None else "ok",
        "coefficient": json.dumps(coefs, ensure_ascii=False),
        "selected": False,
    })
    fitness_cache[sig] = dict(row)
    return row


def phasec_stgp_and_filter_candidates(candidates, df_enc, E, term_meta=None, support_context=None, fold_label=""):
    term_meta = term_meta or build_phasec_term_metadata(df_enc)
    support_context = support_context or phasec_stgp_precompute_support(df_enc, E, term_meta, candidates)
    return phasec_stgp_filter_candidates(candidates, df_enc, E, term_meta=term_meta, support_context=support_context, fold_label=fold_label)


def phasec_seeded_filter_candidates(candidates, df_enc, E, fold_label=""):
    # legacy only; not used by the STGP-AND main path
    if not _phasec_mode_is_stgp_and():
        return _phasec_legacy_seeded_filter_candidates(candidates, df_enc, E, fold_label=fold_label)
    return phasec_stgp_and_filter_candidates(candidates, df_enc, E, fold_label=fold_label)



def add_phasec_candidate_to_matrix(X, df_enc, candidate):
    if not _phasec_mode_is_stgp_and():
        return _phasec_legacy_add_candidate_to_matrix(X, df_enc, candidate)
    X = X.copy()
    for col in candidate.get("main_effects", []) or []:
        if col not in df_enc.columns:
            return X, f"missing_main::{col}"
        if col not in X.columns:
            X[col] = pd.to_numeric(df_enc[col], errors="coerce").fillna(0.0).astype(float).values
    inter = candidate.get("interaction")
    pair = candidate.get("interaction_pair")
    if inter and pair:
        a, b = pair
        if a not in df_enc.columns or b not in df_enc.columns:
            return X, f"missing_inter_pair::{a},{b}"
        if inter not in X.columns:
            xa = pd.to_numeric(df_enc[a], errors="coerce").fillna(0.0).astype(float).values.astype(bool)
            xb = pd.to_numeric(df_enc[b], errors="coerce").fillna(0.0).astype(float).values.astype(bool)
            X[inter] = (xa & xb).astype(float)
    return X, "ok"


def _phasec_make_candidate_from_parts(typ, a, b=None, term_meta=None):
    if not _phasec_mode_is_stgp_and():
        return _phasec_legacy_make_candidate_from_parts(typ, a, b)
    term_meta = term_meta or {}
    if typ in {"cat_main", "binary_main", "main"}:
        return _phasec_make_stgp_main_candidate(a, term_meta)
    if typ in {"and_rule", "binary_mul", "cat_binary_mul"} and b is not None:
        return _phasec_make_stgp_and_candidate(a, b, term_meta)
    return None



def phasec_candidate_signature(candidate):
    op = str(candidate.get("operator", ""))
    if op == "AND":
        pair = list(candidate.get("interaction_pair", []) or candidate.get("main_effects", []))[:2]
        return ("and_rule", "AND", tuple(_phasec_order_pair(pair[0], pair[1])) if len(pair) == 2 else tuple(pair))
    return ("main", "main", tuple(candidate.get("main_effects", []) or []))


def phasec_stgp_and_unique_by_signature(candidates):
    seen, out = set(), []
    for cand in candidates or []:
        if cand is None:
            continue
        sig = phasec_candidate_signature(cand)
        if sig in seen:
            continue
        seen.add(sig)
        out.append(dict(cand))
    return out


def phasec_is_legal_candidate(candidate, cat_cols=None, bin_cols=None, phasec_context=None):
    if not _phasec_mode_is_stgp_and():
        return _phasec_legacy_is_legal_candidate(candidate, cat_cols, bin_cols)
    phasec_context = phasec_context or {}
    cat_cols = set(cat_cols or phasec_context.get("cat_cols", []))
    bin_cols = set(bin_cols or phasec_context.get("bin_cols", []))
    term_meta = phasec_context.get("term_meta", {}) or {}
    mains = list(candidate.get("main_effects", []) or [])
    if candidate.get("operator") == "main":
        return len(mains) == 1 and (mains[0] in cat_cols or mains[0] in bin_cols)
    if candidate.get("operator") == "AND":
        if len(mains) != 2 or len(set(mains)) != 2 or not candidate.get("interaction"):
            return False
        ok, _ = phasec_is_semantically_valid_and(mains[0], mains[1], term_meta)
        return bool(ok)
    return False


def phasec_stgp_and_random_candidate(feasible=None, exclude_sigs=None, phasec_context=None):
    # Uniform/random sampling from the feasible STGP-AND rule space; no seed, warm-start, or fitness ranking.
    phasec_context = phasec_context or {}
    pool = list(feasible or phasec_context.get("feasible_rules", []) or [])
    if not pool:
        return None
    exclude_sigs = set(exclude_sigs or [])
    available = [c for c in pool if phasec_candidate_signature(c) not in exclude_sigs]
    if available:
        return dict(random.choice(available))
    return None


def phasec_stgp_random_legal_candidate(df_enc, E, cat_cols, bin_cols, term_meta, max_tries=100, phasec_context=None):
    phasec_context = phasec_context or {}
    sampled = phasec_stgp_and_random_candidate(phasec_context=phasec_context)
    if sampled is not None:
        return sampled
    all_terms = list(cat_cols) + list(bin_cols)
    support_context = phasec_context.get("support_context")
    for _ in range(int(max_tries)):
        make_and = bool(random.random() < 0.65 and (len(bin_cols) >= 2 or (cat_cols and bin_cols)))
        cand = None
        if not make_and and all_terms:
            cand = _phasec_make_stgp_main_candidate(random.choice(all_terms), term_meta)
        else:
            if cat_cols and bin_cols and random.random() < 0.6:
                cand = _phasec_make_stgp_and_candidate(random.choice(cat_cols), random.choice(bin_cols), term_meta)
            elif len(bin_cols) >= 2:
                a, b = random.sample(bin_cols, 2)
                cand = _phasec_make_stgp_and_candidate(a, b, term_meta)
        if cand is None:
            continue
        kept, _ = phasec_stgp_filter_candidates([cand], df_enc, E, term_meta=term_meta, support_context=support_context, fold_label="")
        if kept:
            return kept[0]
    return None


def phasec_random_legal_candidate(filtered_pool, df_enc=None, E=None, cat_cols=None, bin_cols=None):
    # legacy only; STGP-AND main path uses phasec_stgp_and_random_candidate(..., phasec_context=...).
    if not _phasec_mode_is_stgp_and():
        return _phasec_legacy_random_legal_candidate(filtered_pool, df_enc, E, cat_cols, bin_cols)
    return phasec_stgp_and_random_candidate(filtered_pool)


def _phasec_candidate_pass_support(candidate, df_enc, E, phasec_context=None):
    phasec_context = phasec_context or {}
    term_meta = phasec_context.get("term_meta") or build_phasec_term_metadata(df_enc)
    support_context = phasec_context.get("support_context")
    kept, _ = phasec_stgp_filter_candidates([candidate], df_enc, E, term_meta=term_meta, support_context=support_context, fold_label="")
    return kept[0] if kept else None


def phasec_stgp_and_mutate_candidate(candidate, feasible, df_enc, E, phasec_context):
    term_meta = phasec_context.get("term_meta") or build_phasec_term_metadata(df_enc)
    cat_cols = list(phasec_context.get("cat_cols", []))
    bin_cols = list(phasec_context.get("bin_cols", []))
    cand = dict(candidate)
    for _ in range(40):
        op = random.choice(["replace_terminal", "main_to_and", "and_to_main", "main_to_main", "random"])
        mains = list(cand.get("main_effects", []) or [])
        new = None
        if op == "random":
            new = phasec_stgp_and_random_candidate(feasible, phasec_context=phasec_context)
        elif op == "main_to_main" and cand.get("operator") == "main":
            pool = [x for x in (cat_cols + bin_cols) if x not in mains]
            if pool:
                new = _phasec_make_stgp_main_candidate(random.choice(pool), term_meta)
        elif op == "and_to_main" and cand.get("operator") == "AND" and mains:
            new = _phasec_make_stgp_main_candidate(random.choice(mains), term_meta)
        elif op == "main_to_and" and cand.get("operator") == "main" and mains:
            partners = [x for x in (cat_cols + bin_cols) if x != mains[0]]
            random.shuffle(partners)
            for b in partners[:20]:
                trial = _phasec_make_stgp_and_candidate(mains[0], b, term_meta)
                kept = _phasec_candidate_pass_support(trial, df_enc, E, phasec_context=phasec_context)
                if kept is not None:
                    new = kept
                    break
        elif op == "replace_terminal" and cand.get("operator") == "AND" and len(mains) == 2:
            keep = random.choice([0, 1])
            a = mains[keep]
            partners = [x for x in (cat_cols + bin_cols) if x != a and x not in mains]
            random.shuffle(partners)
            for b in partners[:20]:
                trial = _phasec_make_stgp_and_candidate(a, b, term_meta)
                kept = _phasec_candidate_pass_support(trial, df_enc, E, phasec_context=phasec_context)
                if kept is not None:
                    new = kept
                    break
        if new is not None:
            kept = _phasec_candidate_pass_support(new, df_enc, E, phasec_context=phasec_context)
            if kept is not None:
                return kept
    return dict(candidate)


def phasec_mutate_candidate(candidate, filtered_pool, df_enc, E, cat_cols, bin_cols):
    # legacy only; STGP-AND main path calls phasec_stgp_and_mutate_candidate.
    if not _phasec_mode_is_stgp_and():
        return _phasec_legacy_mutate_candidate(candidate, filtered_pool, df_enc, E, cat_cols, bin_cols)
    phasec_context = {"term_meta": build_phasec_term_metadata(df_enc), "feasible_rules": list(filtered_pool or []), "cat_cols": list(cat_cols or []), "bin_cols": list(bin_cols or [])}
    phasec_context["support_context"] = phasec_stgp_precompute_support(df_enc, E, phasec_context["term_meta"], filtered_pool)
    return phasec_stgp_and_mutate_candidate(candidate, filtered_pool, df_enc, E, phasec_context)


def phasec_stgp_and_crossover_candidates(parent1, parent2, feasible, df_enc, E, phasec_context):
    term_meta = phasec_context.get("term_meta") or build_phasec_term_metadata(df_enc)
    p1, p2 = dict(parent1), dict(parent2)
    m1 = list(p1.get("main_effects", []) or [])
    m2 = list(p2.get("main_effects", []) or [])
    if p1.get("operator") == "AND" and p2.get("operator") == "AND" and len(m1) == 2 and len(m2) == 2:
        trials = [_phasec_make_stgp_and_candidate(m1[0], m2[1], term_meta), _phasec_make_stgp_and_candidate(m2[0], m1[1], term_meta)]
    elif p1.get("operator") == "main" and p2.get("operator") == "AND" and len(m1) == 1 and len(m2) == 2:
        trials = [_phasec_make_stgp_and_candidate(m1[0], random.choice(m2), term_meta), p2]
    elif p2.get("operator") == "main" and p1.get("operator") == "AND" and len(m2) == 1 and len(m1) == 2:
        trials = [p1, _phasec_make_stgp_and_candidate(m2[0], random.choice(m1), term_meta)]
    else:
        r = phasec_stgp_and_random_candidate(feasible, phasec_context=phasec_context)
        trials = [r or p1, p2]
    out = []
    for new, old in zip(trials[:2], [p1, p2]):
        kept = _phasec_candidate_pass_support(new, df_enc, E, phasec_context=phasec_context) if new is not None else None
        out.append(kept if kept is not None else old)
    while len(out) < 2:
        out.append(p2 if len(out) else p1)
    return out[0], out[1]


def phasec_crossover_candidates(parent1, parent2, filtered_pool, df_enc, E, cat_cols, bin_cols):
    # legacy only; STGP-AND main path calls phasec_stgp_and_crossover_candidates.
    if not _phasec_mode_is_stgp_and():
        return _phasec_legacy_crossover_candidates(parent1, parent2, filtered_pool, df_enc, E, cat_cols, bin_cols)
    phasec_context = {"term_meta": build_phasec_term_metadata(df_enc), "feasible_rules": list(filtered_pool or []), "cat_cols": list(cat_cols or []), "bin_cols": list(bin_cols or [])}
    phasec_context["support_context"] = phasec_stgp_precompute_support(df_enc, E, phasec_context["term_meta"], filtered_pool)
    return phasec_stgp_and_crossover_candidates(parent1, parent2, filtered_pool, df_enc, E, phasec_context)



def phasec_seeded_score_candidates(candidates, df_tr, T_tr, E_tr, base_risk_tr, fold_label=""):
    if not _phasec_mode_is_stgp_and():
        return _phasec_legacy_score_candidates(candidates, df_tr, T_tr, E_tr, base_risk_tr, fold_label=fold_label)
    eta = np.log(np.clip(np.asarray(base_risk_tr, dtype=float), 1e-12, None))
    X_base = pd.DataFrame({"eta_ab": eta}, index=np.arange(len(df_tr)))
    base_model, _ = phasec_seeded_fit_cox(X_base, T_tr, E_tr)
    base_ll = float(getattr(base_model, "log_likelihood_", np.nan)) if base_model is not None else np.nan
    rows = []
    for cand in candidates:
        X, reason = add_phasec_candidate_to_matrix(X_base, df_tr.reset_index(drop=True), cand)
        zero_var = any(float(np.nanstd(pd.to_numeric(X[c], errors="coerce").fillna(0.0).values)) <= 0.0 for c in X.columns if c != "eta_ab")
        model, fit_reason = (None, "zero_variance") if zero_var else (phasec_seeded_fit_cox(X, T_tr, E_tr) if reason == "ok" else (None, reason))
        cand_ll = float(getattr(model, "log_likelihood_", np.nan)) if model is not None else np.nan
        coefs = {str(k): float(v) for k, v in model.params_.to_dict().items()} if model is not None else {}
        if model is not None and (not all(np.isfinite(list(coefs.values()))) or any(abs(v) > 50 for v in coefs.values())):
            model = None
            fit_reason = "invalid_coefficients"
            cand_ll = np.nan
            coefs = {}
        imp = cand_ll - base_ll if np.isfinite(cand_ll) and np.isfinite(base_ll) else np.nan
        complexity = int(cand.get("complexity", 1))
        row = dict(cand)
        row.update({
            "fold": fold_label, "train_loglik_base": base_ll,
            "train_loglik_candidate": cand_ll, "train_loglik_improvement": imp,
            "complexity": complexity,
            "fitness": imp - float(PHASEC_COMPLEXITY_LAMBDA) * complexity if np.isfinite(imp) else -np.inf,
            "converged": bool(model is not None),
            "reason": fit_reason if model is None else "ok",
            "coefficient": json.dumps(coefs, ensure_ascii=False),
            "selected": False,
        })
        rows.append(row)
    df = pd.DataFrame(rows)
    if len(df):
        df = df.sort_values(["fitness", "train_loglik_improvement", "complexity"], ascending=[False, False, True]).reset_index(drop=True)
    return df


def phasec_stgp_and_fit_selected(selected, df_tr, df_va, T_tr, E_tr, base_risk_tr, base_risk_va):
    eta_tr = np.log(np.clip(np.asarray(base_risk_tr, dtype=float), 1e-12, None))
    eta_va = np.log(np.clip(np.asarray(base_risk_va, dtype=float), 1e-12, None))
    X_tr = pd.DataFrame({"eta_ab": eta_tr}, index=np.arange(len(df_tr)))
    X_va = pd.DataFrame({"eta_ab": eta_va}, index=np.arange(len(df_va)))
    selected_clean = []
    for cand in selected or []:
        X_tr, r1 = add_phasec_candidate_to_matrix(X_tr, df_tr.reset_index(drop=True), cand)
        X_va, r2 = add_phasec_candidate_to_matrix(X_va, df_va.reset_index(drop=True), cand)
        if r1 == "ok" and r2 == "ok":
            selected_clean.append(dict(cand))
    X_tr = X_tr.loc[:, ~X_tr.columns.duplicated()].copy()
    X_va = X_va.reindex(columns=X_tr.columns, fill_value=0.0)
    if len(selected_clean) == 0 or X_tr.shape[1] <= 1:
        return {"risk_valid": np.asarray(base_risk_va, dtype=float), "coefficients": {}, "selected_candidates": [], "reason": "selected_k=0"}
    model, reason = phasec_seeded_fit_cox(X_tr, T_tr, E_tr)
    if model is None:
        return {"risk_valid": np.asarray(base_risk_va, dtype=float), "coefficients": {}, "selected_candidates": [], "reason": reason}
    coefs = {str(k): float(v) for k, v in model.params_.to_dict().items()}
    try:
        risk_va = model.predict_partial_hazard(X_va[X_tr.columns]).values.reshape(-1)
    except Exception:
        risk_va = np.asarray(base_risk_va, dtype=float)
        reason = "predict_failed"
    enriched = []
    for cand in selected_clean:
        cc = dict(cand)
        cc["coefficient"] = json.dumps({k: coefs.get(k, np.nan) for k in cc.get("cols_to_add", []) if k in coefs}, ensure_ascii=False)
        if cc.get("interaction") in coefs:
            cc["interaction_coefficient"] = coefs.get(cc.get("interaction"))
        enriched.append(cc)
    return {"risk_valid": risk_va, "coefficients": coefs, "selected_candidates": enriched, "reason": reason}


def phasec_seeded_fit_selected(selected, df_tr, df_va, T_tr, E_tr, base_risk_tr, base_risk_va):
    # legacy only; STGP-AND main path calls phasec_stgp_and_fit_selected.
    if not _phasec_mode_is_stgp_and():
        return _phasec_legacy_fit_selected(selected, df_tr, df_va, T_tr, E_tr, base_risk_tr, base_risk_va)
    return phasec_stgp_and_fit_selected(selected, df_tr, df_va, T_tr, E_tr, base_risk_tr, base_risk_va)



def phasec_stgp_and_prepare_fold(df_tr_enc, prep, T_tr, E_tr, base_risk_tr, fold_label=""):
    term_meta = build_phasec_term_metadata(df_tr_enc, prep)
    cat_cols, bin_cols = phasec_stgp_and_source_columns(df_tr_enc, prep)
    raw = phasec_stgp_and_build_rule_universe(df_tr_enc, prep)
    support_context = phasec_stgp_precompute_support(df_tr_enc, E_tr, term_meta, raw)
    filtered, dropped = phasec_stgp_and_filter_candidates(raw, df_tr_enc, E_tr, term_meta=term_meta, support_context=support_context, fold_label=fold_label)
    phasec_context = {
        "term_meta": term_meta,
        "support_context": support_context,
        "feasible_rules": list(filtered),
        "cat_cols": list(cat_cols),
        "bin_cols": list(bin_cols),
    }
    label = "Final full train" if str(fold_label) == "final_full_train" else f"Fold {fold_label}"
    reason_items = []
    for d in dropped:
        reason_items.extend(_phasec_simplify_drop_reason(d.get("reason", "")))
    phasec_stgp_and_log_policy_once()
    print(f"[PhaseC-STGP-AND][{label}] raw_rule_universe={len(raw)}")
    print(f"[PhaseC-STGP-AND][{label}] feasible_rule_universe={len(filtered)}")
    print(f"[PhaseC-STGP-AND][{label}] dropped_rules={len(dropped)}")
    print(f"[PhaseC-STGP-AND][{label}] top_drop_reasons={dict(Counter(reason_items).most_common(5))}")
    print(f"[PhaseC-STGP-AND][{label}] support precomputed")
    print(f"[PhaseC-STGP-AND][{label}] population_size={int(PHASEC_POP_SIZE)}")
    print(f"[PhaseC-STGP-AND][{label}] n_gen={int(PHASEC_N_GEN)}")
    return raw, filtered, dropped, pd.DataFrame(), phasec_context


def phasec_seeded_prepare_fold(df_tr_enc, prep, T_tr, E_tr, base_risk_tr, fold_label=""):
    # legacy only; STGP-AND main path calls phasec_stgp_and_prepare_fold.
    if not _phasec_mode_is_stgp_and():
        return _phasec_legacy_prepare_fold(df_tr_enc, prep, T_tr, E_tr, base_risk_tr, fold_label=fold_label)
    return phasec_stgp_and_prepare_fold(df_tr_enc, prep, T_tr, E_tr, base_risk_tr, fold_label=fold_label)


def phasec_stgp_and_run_evolution(raw, feasible, df_enc, T, E, base_risk, prep=None, fold_label="", phasec_context=None):
    phasec_context = dict(phasec_context or {})
    if not phasec_context:
        term_meta = build_phasec_term_metadata(df_enc, prep)
        cat_cols, bin_cols = phasec_stgp_and_source_columns(df_enc, prep or {})
        support_context = phasec_stgp_precompute_support(df_enc, E, term_meta, raw)
        phasec_context = {"term_meta": term_meta, "support_context": support_context, "feasible_rules": list(feasible or []), "cat_cols": list(cat_cols), "bin_cols": list(bin_cols)}
    term_meta = phasec_context.get("term_meta") or build_phasec_term_metadata(df_enc, prep)
    cat_cols = list(phasec_context.get("cat_cols", []))
    bin_cols = list(phasec_context.get("bin_cols", []))
    support_context = phasec_context.get("support_context") or phasec_stgp_precompute_support(df_enc, E, term_meta, raw)
    feasible = [c for c in (feasible or phasec_context.get("feasible_rules", []) or []) if phasec_is_legal_candidate(c, cat_cols, bin_cols, phasec_context=phasec_context)]
    feasible = phasec_stgp_and_unique_by_signature(feasible)
    phasec_context.update({"term_meta": term_meta, "support_context": support_context, "feasible_rules": list(feasible), "cat_cols": cat_cols, "bin_cols": bin_cols})
    if not feasible:
        return {"hof": [], "generation_log": pd.DataFrame(), "population_init": pd.DataFrame(), "candidate_scores": pd.DataFrame(), "hof_df": pd.DataFrame()}
    base_context = phasec_stgp_fit_base_once(df_enc, T, E, base_risk)
    base_context["support_context"] = support_context
    fitness_cache = {}

    population, population_seen = [], set()
    shuffled = list(feasible)
    random.shuffle(shuffled)
    for cand in shuffled:
        if len(population) >= int(PHASEC_POP_SIZE):
            break
        sig = phasec_candidate_signature(cand)
        if sig in population_seen:
            continue
        population_seen.add(sig)
        population.append(dict(cand, init_source="random_legal", fold=fold_label))
    while len(population) < int(PHASEC_POP_SIZE):
        cand = phasec_stgp_and_random_candidate(feasible, exclude_sigs=population_seen, phasec_context=phasec_context)
        if cand is None:
            break
        sig = phasec_candidate_signature(cand)
        if sig in population_seen:
            break
        population_seen.add(sig)
        population.append(dict(cand, init_source="random_legal", fold=fold_label))
        if len(population) >= len(feasible):
            break
    population = population[:int(PHASEC_POP_SIZE)]
    init_rows = [dict(c, fold=fold_label, initialization="random_legal", base_fit_once=True, support_precomputed=True, fitness_cache_enabled=True) for c in population]
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] raw_rule_universe={len(raw)}")
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] feasible_rule_universe={len(feasible)}")
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] init_population={len(population)} random_legal={len(population)}")
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] base Cox fitted once")
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] support precomputed")
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] fitness_cache enabled")
    hof, hof_seen, gen_rows = [], set(), []
    for gen in range(1, int(PHASEC_N_GEN) + 1):
        eval_rows = [phasec_stgp_and_evaluate_candidate(c, base_context, fitness_cache, phasec_context=phasec_context) for c in population]
        eval_rows = sorted(eval_rows, key=lambda x: float(x.get("fitness", -np.inf)), reverse=True)
        for row in eval_rows:
            sig = phasec_candidate_signature(row)
            if sig not in hof_seen:
                hof_seen.add(sig)
                hof.append(dict(row))
        hof = sorted(hof, key=lambda x: float(x.get("fitness", -np.inf)), reverse=True)[:int(PHASEC_HOF_SIZE)]
        hof_seen = {phasec_candidate_signature(c) for c in hof}
        fits = np.asarray([float(x.get("fitness", -np.inf)) for x in eval_rows if np.isfinite(float(x.get("fitness", -np.inf)))], dtype=float)
        avg_fit = float(np.mean(fits)) if len(fits) else -np.inf
        max_fit = float(np.max(fits)) if len(fits) else -np.inf
        hof_fit = float(hof[0].get("fitness", -np.inf)) if hof else -np.inf
        best_name = str(eval_rows[0].get("expr", eval_rows[0].get("name", ""))) if eval_rows else ""
        gen_rows.append({
            "fold": fold_label, "generation": int(gen), "avgFit": avg_fit, "maxFit": max_fit,
            "hofFit": hof_fit, "best": best_name, "base_fit_once": True, "support_precomputed": True,
            "fitness_cache_enabled": True, "fitness_cache_size": int(len(fitness_cache)),
            "candidate_fit_calls": int(base_context.get("candidate_fit_calls", 0)),
            "cache_hits": int(base_context.get("cache_hits", 0)), "initialization": "random_legal",
        })
        print(f"[PhaseC-STGP-AND][Fold {fold_label}] Gen {gen:02d} | avgFit={avg_fit:.4f} maxFit={max_fit:.4f} hofFit={hof_fit:.4f} best={best_name}")
        elites = [dict(x) for x in eval_rows[:int(PHASEC_ELITE_SIZE)]]
        next_pop, next_seen = [], set()
        for c in elites:
            sig = phasec_candidate_signature(c)
            if sig not in next_seen:
                next_seen.add(sig)
                next_pop.append(c)
        inject_n = int(round(int(PHASEC_POP_SIZE) * float(PHASEC_RANDOM_INJECTION_RATE)))
        guard = 0
        while len(next_pop) < int(PHASEC_POP_SIZE) - inject_n and eval_rows and guard < int(PHASEC_POP_SIZE) * 20:
            guard += 1
            parents = random.sample(eval_rows[:max(2, min(len(eval_rows), 20))], 2) if len(eval_rows) >= 2 else [eval_rows[0], eval_rows[0]]
            c1, c2 = dict(parents[0]), dict(parents[1])
            if random.random() < float(PHASEC_CROSSOVER_RATE):
                c1, c2 = phasec_stgp_and_crossover_candidates(c1, c2, feasible, df_enc, E, phasec_context=phasec_context)
            if random.random() < float(PHASEC_MUTATION_RATE):
                c1 = phasec_stgp_and_mutate_candidate(c1, feasible, df_enc, E, phasec_context=phasec_context)
            if random.random() < float(PHASEC_MUTATION_RATE):
                c2 = phasec_stgp_and_mutate_candidate(c2, feasible, df_enc, E, phasec_context=phasec_context)
            for child in (c1, c2):
                sig = phasec_candidate_signature(child)
                if sig not in next_seen:
                    next_seen.add(sig)
                    next_pop.append(dict(child))
                if len(next_pop) >= int(PHASEC_POP_SIZE) - inject_n:
                    break
        while len(next_pop) < int(PHASEC_POP_SIZE):
            r = phasec_stgp_and_random_candidate(feasible, exclude_sigs=next_seen, phasec_context=phasec_context)
            if r is None:
                break
            sig = phasec_candidate_signature(r)
            if sig in next_seen:
                break
            next_seen.add(sig)
            next_pop.append(dict(r, init_source="random_injection", fold=fold_label))
            if len(next_pop) >= len(feasible):
                break
        population = next_pop[:int(PHASEC_POP_SIZE)]
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] HOF top candidates:")
    for i, c in enumerate(hof[:min(10, len(hof))], 1):
        print(f"    {i}. {c.get('expr', c.get('name', ''))} | fitness={float(c.get('fitness', np.nan)):.4f}")
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] fitness_cache size={len(fitness_cache)}")
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] candidate_fit_calls={int(base_context.get('candidate_fit_calls', 0))}")
    print(f"[PhaseC-STGP-AND][Fold {fold_label}] cache_hits={int(base_context.get('cache_hits', 0))}")
    hof_df = pd.DataFrame([
        dict(c, fold=fold_label, hof_rank=i + 1, base_fit_once=True, support_precomputed=True,
             fitness_cache_enabled=True, fitness_cache_size=int(len(fitness_cache)),
             candidate_fit_calls=int(base_context.get("candidate_fit_calls", 0)),
             cache_hits=int(base_context.get("cache_hits", 0)), initialization="random_legal")
        for i, c in enumerate(hof)
    ])
    score_df = pd.DataFrame(list(fitness_cache.values()))
    return {"hof": hof, "generation_log": pd.DataFrame(gen_rows), "population_init": pd.DataFrame(init_rows), "candidate_scores": score_df, "hof_df": hof_df}


def phasec_stgp_and_evolution_dispatch(raw, filtered, df_enc, T, E, base_risk, prep=None, fold_label="", phasec_context=None):
    return phasec_stgp_and_run_evolution(raw, filtered, df_enc, T, E, base_risk, prep=prep, fold_label=fold_label, phasec_context=phasec_context)


def phasec_phasec_evolution_dispatch(raw, filtered, df_enc, T, E, base_risk, prep=None, fold_label=""):
    # legacy only; STGP-AND main path calls phasec_stgp_and_run_evolution directly.
    if not _phasec_mode_is_stgp_and():
        return _phasec_legacy_phasec_evolution(raw, filtered, df_enc, T, E, base_risk, prep=prep, fold_label=fold_label)
    return phasec_stgp_and_run_evolution(raw, filtered, df_enc, T, E, base_risk, prep=prep, fold_label=fold_label)



def build_phasec_candidate_terminals(df_enc: pd.DataFrame, prep: dict):
    cand_cols = []
    factor_map = {}
    cat_cols = prep.get("cat_cols", [])
    for c in df_enc.columns:
        if c in (TIME_COL, EVENT_COL):
            continue
        vals = pd.to_numeric(df_enc[c], errors="coerce").dropna().unique()
        if len(vals) == 0:
            continue
        if not set(np.asarray(vals, dtype=float)).issubset({0.0, 1.0}):
            continue
        cand_cols.append(c)
        fac = None
        for cat in cat_cols:
            if c.startswith(f"{cat}_"):
                fac = cat
                break
        if fac is None:
            if c.startswith("ethnicity_"):
                fac = "ethnicity"
            elif c.startswith("admission_type_"):
                fac = "admission_type"
            else:
                fac = c
        factor_map[c] = fac
    cand_cols = sorted(list(dict.fromkeys(cand_cols)))
    return df_enc[cand_cols].astype(float), factor_map


def build_gp_toolbox_c(n_features: int):
    pset = gp.PrimitiveSetTyped("MAIN_C", [np.ndarray] * n_features, np.ndarray)
    pset.addPrimitive(operator.add, [np.ndarray, np.ndarray], np.ndarray)
    pset.addPrimitive(operator.mul, [np.ndarray, np.ndarray], np.ndarray)
    for i in range(n_features):
        pset.renameArguments(**{f"ARG{i}": f"x{i}"})
    if "IndividualC" not in creator.__dict__:
        creator.create("IndividualC", gp.PrimitiveTree, fitness=creator.FitnessMax)
    tb = base.Toolbox()
    tb.register(
        "expr_c",
        gp.genHalfAndHalf,
        pset=pset,
        min_=1,
        max_=max(1, int(MAX_TREE_HEIGHT_C))
    )
    tb.register("individual", tools.initIterate, creator.IndividualC, tb.expr_c)
    tb.register("population", tools.initRepeat, list, tb.individual)
    tb.register("compile", gp.compile, pset=pset)
    def _limit_c(tree):
        if tree.height > MAX_TREE_HEIGHT_C:
            return creator.IndividualC(gp.PrimitiveTree(tb.expr_c()))
        return tree
    def mate_c(i1, i2):
        gp.cxOnePoint(i1, i2)
        return _limit_c(i1), _limit_c(i2)
    def mutate_c(ind):
        gp.mutUniform(ind, expr=tb.expr_c, pset=pset)
        return (_limit_c(ind),)
    tb.register("mate", mate_c)
    tb.register("mutate", mutate_c)
    tb.register("select", tools.selTournament, tournsize=TOURN_SIZE_C)
    return tb, pset


def _extract_x_tokens(tree):
    toks = []
    for node in tree:
        name = getattr(node, "name", str(node))
        if re.fullmatch(r"x\d+", name):
            toks.append(name)
    return toks


def _phasec_terms_from_tree(tree):
    arr = tree

    def parse(i):
        node = arr[i]
        name = getattr(node, "name", str(node))
        if node.arity == 0:
            tok = str(node)
            return i + 1, [set([tok])] if tok.startswith("x") else []
        j = i + 1
        children = []
        for _ in range(node.arity):
            j, terms = parse(j)
            children.append(terms)
        if name == "add":
            out = []
            for t in children:
                out.extend(t)
            return j, out
        if name == "mul" and len(children) == 2:
            out = []
            for a in children[0]:
                for b in children[1]:
                    out.append(set(a) | set(b))
            return j, out
        return j, []

    _, terms = parse(0)
    return terms


def validate_phasec_tree(tree, terminal_names, factor_map):
    toks = _extract_x_tokens(tree)
    if len(toks) == 0:
        return False, "no_terminal"
    if len(set(toks)) != len(toks):
        return False, "repeated_terminal"

    terms = _phasec_terms_from_tree(tree)
    if len(terms) == 0:
        return False, "invalid_terms"
    if len(terms) > 2:
        return False, "too_many_additive_terms"

    all_factors = set()
    for term in terms:
        fac_term = []
        for tok in term:
            if not tok.startswith("x"):
                continue
            idx = int(tok[1:])
            if idx < 0 or idx >= len(terminal_names):
                return False, "bad_terminal_index"
            col = terminal_names[idx]
            fac = factor_map.get(col, col)
            fac_term.append(fac)
            all_factors.add(fac)
        # Forbid two levels of the same original factor in one multiplicative term.
        if len(fac_term) != len(set(fac_term)):
            return False, "same_factor_levels_in_mul"
    if len(all_factors) > 3:
        return False, "too_many_factors"
    return True, "ok"


def normalize_phasec_tree(tree, terminal_names):
    return tree_to_named_expr(tree, terminal_names)


def compute_phasec_tree_feature(tree, toolbox_c, C: np.ndarray):
    func = toolbox_c.compile(expr=tree)
    cols = [C[:, j] for j in range(C.shape[1])]
    try:
        out = np.asarray(func(*cols), dtype=float).reshape(-1)
    except Exception:
        return None
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


def evaluate_individual_c(tree, toolbox_c, C_train_raw, base_ab_train, T_train, E_train, terminal_names, factor_map):
    ok, _ = validate_phasec_tree(tree, terminal_names, factor_map)
    if not ok:
        return (-1e6,)
    feat = compute_phasec_tree_feature(tree, toolbox_c, C_train_raw)
    if feat is None:
        return (-1e6,)
    kf = KFold(n_splits=max(2, FIT_FOLDS_T), shuffle=True, random_state=SEED)
    cidxs = []
    for tr, te in kf.split(C_train_raw):
        if int((feat[tr] > 0).sum()) < int(PHASEC_MIN_EVENT_SUPPORT):
            return (-1e6,)
        eta_tr = np.log(np.clip(base_ab_train[tr], 1e-12, None))
        eta_te = np.log(np.clip(base_ab_train[te], 1e-12, None))
        X_tr = pd.DataFrame({"eta_ab": eta_tr, "phasec_tree": feat[tr]})
        X_te = pd.DataFrame({"eta_ab": eta_te, "phasec_tree": feat[te]})
        df_tr = X_tr.copy()
        df_tr[TIME_COL] = T_train[tr]
        df_tr[EVENT_COL] = E_train[tr]
        try:
            cph = CoxPHFitter(penalizer=float(PHASEC_PENALIZER), l1_ratio=float(PHASEC_L1_RATIO))
            cph.fit(df_tr, duration_col=TIME_COL, event_col=EVENT_COL)
            risk = cph.predict_partial_hazard(X_te).values.reshape(-1)
            cidxs.append(float(concordance_index(T_train[te], -risk, E_train[te])))
        except Exception:
            return (-1e6,)
    return (float(np.mean(cidxs)) - float(COMPLEXITY_LAMBDA_C) * float(len(tree)),)


def search_phasec_trees(C_train_df, base_ab_train, T_train, E_train, phasec_params, factor_map=None):
    if C_train_df.shape[1] == 0:
        return [], {}, []
    old = _temporary_set_params(phasec_params)
    try:
        terminal_names = list(C_train_df.columns)
        if factor_map is None:
            factor_map = {c: c for c in terminal_names}
        else:
            factor_map = {c: factor_map.get(c, c) for c in terminal_names}
        C_train = C_train_df.values.astype(float)
        toolbox_c, _ = build_gp_toolbox_c(C_train.shape[1])
        toolbox_c.register(
            "evaluate", evaluate_individual_c,
            toolbox_c=toolbox_c,
            C_train_raw=C_train,
            base_ab_train=np.asarray(base_ab_train, dtype=float),
            T_train=np.asarray(T_train, dtype=float),
            E_train=np.asarray(E_train, dtype=int),
            terminal_names=terminal_names,
            factor_map=factor_map,
        )
        pop = toolbox_c.population(n=int(POP_SIZE_C))
        hof = tools.HallOfFame(max(1, int(ELITE_K_C)))
        for ind in [i for i in pop if not i.fitness.valid]:
            ind.fitness.values = toolbox_c.evaluate(ind)
        hof.update(pop)
        for _ in range(int(N_GEN_C)):
            offspring = list(map(toolbox_c.clone, toolbox_c.select(pop, len(pop))))
            for c1, c2 in zip(offspring[::2], offspring[1::2]):
                if random.random() < float(CX_PB_C):
                    toolbox_c.mate(c1, c2)
                    del c1.fitness.values
                    del c2.fitness.values
            for mut in offspring:
                if random.random() < float(MUT_PB_C):
                    toolbox_c.mutate(mut)
                    del mut.fitness.values
            for ind in [i for i in offspring if not i.fitness.valid]:
                ind.fitness.values = toolbox_c.evaluate(ind)
            pop[:] = offspring
            hof.update(pop)
        del toolbox_c.__dict__["evaluate"]
        k = int(min(int(phasec_params.get("K_interaction", K_INTERACTION)), int(K_INTERACTION_MAX)))
        if k <= 0:
            return [], factor_map, terminal_names
        selected, seen = [], set()
        for ind in hof:
            ok, _ = validate_phasec_tree(ind, terminal_names, factor_map)
            if not ok:
                continue
            expr = normalize_phasec_tree(ind, terminal_names)
            if expr in seen:
                continue
            seen.add(expr)
            selected.append(ind)
            if len(selected) >= k:
                break
        return selected, factor_map, terminal_names
    finally:
        _restore_params(old)


def apply_phasec_heredity(selected_trees, terminal_names, factor_map, ab_main_effects):
    ab_main_effects = set(ab_main_effects or [])
    must_add = set()
    for tr in selected_trees:
        for tok in _extract_x_tokens(tr):
            idx = int(tok[1:])
            if 0 <= idx < len(terminal_names):
                nm = terminal_names[idx]
                if nm not in ab_main_effects:
                    must_add.add(nm)
    return sorted(must_add)


def fit_phasec_model_with_ab_baseline(selected_trees, toolbox_c, C_train_df, C_valid_df,
                                      base_risk_train, base_risk_valid,
                                      T_train, E_train,
                                      terminal_names, factor_map,
                                      ab_main_effects=None):
    eta_tr = np.log(np.clip(np.asarray(base_risk_train, dtype=float), 1e-12, None))
    eta_va = np.log(np.clip(np.asarray(base_risk_valid, dtype=float), 1e-12, None))
    X_tr = pd.DataFrame({"eta_ab": eta_tr}, index=C_train_df.index)
    X_va = pd.DataFrame({"eta_ab": eta_va}, index=C_valid_df.index)
    selected_expr = []
    n_tree_input = int(len(selected_trees))
    n_tree_pass_support = 0
    for i, tr in enumerate(selected_trees):
        f_tr = compute_phasec_tree_feature(tr, toolbox_c, C_train_df.values.astype(float))
        f_va = compute_phasec_tree_feature(tr, toolbox_c, C_valid_df.values.astype(float))
        if f_tr is None or f_va is None:
            continue
        if int((f_tr > 0).sum()) < int(PHASEC_MIN_SUPPORT_TRAIN):
            continue
        if int((f_va > 0).sum()) < int(PHASEC_MIN_SUPPORT_VALID):
            continue
        c = f"phasec_tree_{i+1}"
        X_tr[c] = f_tr
        X_va[c] = f_va
        selected_expr.append(normalize_phasec_tree(tr, terminal_names))
        n_tree_pass_support += 1
    inherited = apply_phasec_heredity(selected_trees, terminal_names, factor_map, ab_main_effects)
    for c in inherited:
        if c in C_train_df.columns:
            X_tr[f"main_{c}"] = C_train_df[c].values.astype(float)
            X_va[f"main_{c}"] = C_valid_df[c].values.astype(float)
    if X_tr.shape[1] <= 1:
        return {
            "risk_valid": np.exp(eta_va),
            "selected_interaction_trees": [],
            "inherited_main_effects": [],
            "coefficients": {},
            "k_interaction": 0,
            "n_tree_input": n_tree_input,
            "n_tree_pass_support": n_tree_pass_support,
        }
    df_fit = X_tr.copy()
    df_fit[TIME_COL] = np.asarray(T_train, dtype=float)
    df_fit[EVENT_COL] = np.asarray(E_train, dtype=int)
    cph = CoxPHFitter(penalizer=float(PHASEC_PENALIZER), l1_ratio=float(PHASEC_L1_RATIO))
    cph.fit(df_fit, duration_col=TIME_COL, event_col=EVENT_COL)
    risk_va = cph.predict_partial_hazard(X_va).values.reshape(-1)
    return {
        "risk_valid": risk_va,
        "selected_interaction_trees": selected_expr,
        "inherited_main_effects": inherited,
        "coefficients": {k: float(v) for k, v in cph.params_.to_dict().items()},
        "k_interaction": len(selected_expr),
        "n_tree_input": n_tree_input,
        "n_tree_pass_support": n_tree_pass_support,
    }


def phasec_static_feature_frame(df_enc, selected_phasec_candidates):
    static = pd.DataFrame(index=np.arange(len(df_enc)))
    for cand in selected_phasec_candidates or []:
        static, reason = add_phasec_candidate_to_matrix(static, df_enc.reset_index(drop=True), cand)
        if reason != "ok":
            continue
    static = static.loc[:, ~static.columns.duplicated()].copy()
    for c in static.columns:
        static[c] = pd.to_numeric(static[c], errors="coerce").fillna(0.0).astype(float).values
    return static


def abc_unified_build_long_df(T, E, eta_a, feature_df, selected_phaseb_candidates, time_stats, phasec_static_df):
    long_df = phaseb_build_long_df(T, E, eta_a, feature_df, selected_phaseb_candidates or [], time_stats)
    if phasec_static_df is None or len(phasec_static_df.columns) == 0 or len(long_df) == 0:
        return long_df
    static = phasec_static_df.reset_index(drop=True).copy()
    for col in static.columns:
        vals = pd.to_numeric(static[col], errors="coerce").fillna(0.0).astype(float).values
        long_df[col] = long_df["id"].astype(int).map(lambda i, v=vals: float(v[i]) if 0 <= int(i) < len(v) else 0.0)
    return long_df


def time_dependent_cindex_abc(T, E, eta_a, feature_df, selected_phaseb_candidates=None, phasec_static_df=None, coefficients=None, time_stats=None):
    T = np.asarray(T, dtype=float)
    E = np.asarray(E, dtype=int)
    eta_a = np.asarray(eta_a, dtype=float)
    feature_df = feature_df.reset_index(drop=True).copy()
    static_df = phasec_static_df.reset_index(drop=True).copy() if phasec_static_df is not None else pd.DataFrame(index=np.arange(len(T)))
    coefficients = coefficients or {"eta_ab": 1.0}
    selected_phaseb_candidates = selected_phaseb_candidates or []
    event_idx = [i for i in range(len(T)) if E[i] == 1 and np.isfinite(T[i])]
    conc = ties = perm = 0.0
    for i in event_idx:
        ti = float(T[i])
        comparable = np.where(T > ti)[0]
        if len(comparable) == 0:
            continue
        risks = float(coefficients.get("eta_ab", 1.0)) * eta_a.copy()
        for cand in selected_phaseb_candidates:
            feat = cand.get("feature")
            if feat not in feature_df.columns:
                continue
            g = float(phaseb_eval_time_basis([ti], cand.get("basis"), time_stats)[0])
            risks = risks + float(coefficients.get(cand.get("interaction_col"), 0.0)) * pd.to_numeric(feature_df[feat], errors="coerce").fillna(0.0).values * g
        for col in static_df.columns:
            risks = risks + float(coefficients.get(col, 0.0)) * pd.to_numeric(static_df[col], errors="coerce").fillna(0.0).values
        ri = risks[i]
        rj = risks[comparable]
        conc += float(np.sum(ri > rj))
        ties += float(np.sum(np.isclose(ri, rj)))
        perm += float(len(comparable))
    return float((conc + 0.5 * ties) / perm) if perm > 0 else np.nan


def fit_eval_unified_abc_timevarying(
    df_train_enc,
    df_apply_enc,
    T_train,
    E_train,
    T_apply,
    E_apply,
    eta_A_train,
    eta_A_apply,
    feature_tr,
    feature_apply,
    selected_phaseB_candidates=None,
    time_stats=None,
    selected_phaseC_candidates=None,
    fold_label="",
):
    selected_phaseB_candidates = list(selected_phaseB_candidates or [])
    selected_phaseC_candidates = list(selected_phaseC_candidates or [])
    time_stats = time_stats or phaseb_make_time_stats(T_train)
    phasec_tr = phasec_static_feature_frame(df_train_enc, selected_phaseC_candidates)
    phasec_apply = phasec_static_feature_frame(df_apply_enc, selected_phaseC_candidates)
    for col in phasec_tr.columns:
        if col not in phasec_apply.columns:
            phasec_apply[col] = 0.0
    phasec_apply = phasec_apply.reindex(columns=phasec_tr.columns, fill_value=0.0)
    train_long = abc_unified_build_long_df(T_train, E_train, eta_A_train, feature_tr, selected_phaseB_candidates, time_stats, phasec_tr)
    phaseb_cols = [c.get("interaction_col") for c in selected_phaseB_candidates if c.get("interaction_col")]
    phasec_cols = list(phasec_tr.columns)
    design_cols = ["eta_ab"] + phaseb_cols + phasec_cols
    model, reason, usable_cols = fit_phaseb_ctv_model(train_long, design_cols, PHASEB_PENALIZER)
    coefficients = {"eta_ab": 1.0}
    if model is not None:
        coefficients.update({str(k): float(v) for k, v in model.params_.to_dict().items()})
    cindex = time_dependent_cindex_abc(
        T_apply, E_apply, eta_A_apply, feature_apply,
        selected_phaseb_candidates=selected_phaseB_candidates,
        phasec_static_df=phasec_apply,
        coefficients=coefficients,
        time_stats=time_stats,
    )
    if not np.isfinite(cindex):
        cindex = time_dependent_cindex_abc(
            T_apply, E_apply, eta_A_apply, feature_apply,
            selected_phaseb_candidates=[],
            phasec_static_df=pd.DataFrame(index=np.arange(len(T_apply))),
            coefficients={"eta_ab": 1.0},
            time_stats=time_stats,
        )
    return {
        "model": model,
        "reason": reason if model is None else "ok",
        "cindex": float(cindex) if np.isfinite(cindex) else np.nan,
        "design_cols": list(design_cols),
        "usable_cols": list(usable_cols or []),
        "coefficients": coefficients,
        "selected_phaseB_candidates": selected_phaseB_candidates,
        "selected_phaseC_candidates": selected_phaseC_candidates,
        "phasec_static_cols": phasec_cols,
        "fold": fold_label,
    }


def fit_eval_unified_abc_from_model(model_ab, va_pack, selected_phaseC_candidates=None, phaseB_allowed=True, fold_label=""):
    tr_pack = {"df_enc": model_ab["train_enc"], "T": model_ab["T_train"], "E": model_ab["E_train"], "F": model_ab["F_train"], "M": model_ab.get("M_train")}
    toolbox_t = model_ab["toolbox_t"]
    main_effect_names = model_ab.get("main_effect_vars", [])
    eta_tr = phaseb_static_eta_from_ab(tr_pack["F"], tr_pack["T"], tr_pack["E"], tr_pack["F"], tr_pack["T"], tr_pack["E"], toolbox_t, M_tr=tr_pack.get("M"), M_apply=tr_pack.get("M"), main_effect_names=main_effect_names)
    eta_va = phaseb_static_eta_from_ab(tr_pack["F"], tr_pack["T"], tr_pack["E"], va_pack.get("F"), va_pack["T"], va_pack["E"], toolbox_t, M_tr=tr_pack.get("M"), M_apply=va_pack.get("M"), main_effect_names=main_effect_names)
    feature_tr, feature_va, _, _ = phaseb_feature_frames(tr_pack.get("M"), va_pack.get("M"), main_effect_names, tr_pack.get("F"), va_pack.get("F"), n_tr=len(tr_pack["T"]), n_va=len(va_pack["T"]))
    tb = model_ab.get("phaseb_timebasis", {}) or {}
    selected_b = list(tb.get("selected_candidates", []) or []) if bool(phaseB_allowed) else []
    time_stats = tb.get("time_stats", None) or phaseb_make_time_stats(tr_pack["T"])
    return fit_eval_unified_abc_timevarying(
        tr_pack["df_enc"], va_pack["df_enc"],
        tr_pack["T"], tr_pack["E"], va_pack["T"], va_pack["E"],
        eta_tr, eta_va, feature_tr, feature_va,
        selected_phaseB_candidates=selected_b,
        time_stats=time_stats,
        selected_phaseC_candidates=selected_phaseC_candidates,
        fold_label=fold_label,
    )


def score_ab_plus_c_on_validation(T_valid, E_valid, risk_ab_valid, phasec_fit_res):
    cidx_ab = float(concordance_index(T_valid, -np.asarray(risk_ab_valid, dtype=float), E_valid))
    risk_abc = np.asarray(phasec_fit_res.get("risk_valid", risk_ab_valid), dtype=float)
    cidx_abc = float(concordance_index(T_valid, -risk_abc, E_valid))
    return cidx_ab, cidx_abc, float(cidx_abc - cidx_ab)


def run_internal_fixed_cv_select_phasec(
    df_train_raw,
    ab_params,
    k_grid=(0, 1, 2, 3),
    n_splits=4,
):
    df_train_raw = _coerce_time_event(df_train_raw).reset_index(drop=True)
    y = df_train_raw[EVENT_COL].values
    cv = SurvivalAwareKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    k_grid = tuple(sorted(set(int(k) for k in k_grid if 0 <= int(k) <= int(K_INTERACTION_MAX))))
    if len(k_grid) == 0:
        k_grid = (0,)

    fold_ab_gen_rows = []
    phasea_fold_traj_rows = []
    fold_cache = []
    oof_main_only_risk = np.full(len(df_train_raw), np.nan, dtype=float)
    oof_main_gp_risk = np.full(len(df_train_raw), np.nan, dtype=float)
    oof_ab_risk = np.full(len(df_train_raw), np.nan, dtype=float)
    oof_abc_risk_by_k = {int(k): np.full(len(df_train_raw), np.nan, dtype=float) for k in k_grid}
    phasec_search_rows = []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(np.arange(len(y)), y), 1):
        df_tr = df_train_raw.iloc[tr_idx].reset_index(drop=True)
        df_va = df_train_raw.iloc[va_idx].reset_index(drop=True)
        model_ab, gen_rows, phasea_rows = _fit_ab_backbone_on_train_with_valid_logging(
            df_tr,
            df_va,
            ab_params,
            fold_idx=fold,
            inner_folds_for_varsel=n_splits,
        )
        fold_ab_gen_rows.extend(gen_rows)
        phasea_fold_traj_rows.extend(phasea_rows)

        if model_ab is None:
            fold_cache.append({
                "fold": fold,
                "tr_idx": tr_idx,
                "va_idx": va_idx,
                "model_ab": None,
                "c_ab": np.nan,
                "risk_ab_va": None,
            })
            continue

        c_ab, va_pack, risk_ab_va = _score_ab_on_validation(model_ab, df_va)
        risk_main_va = predict_main_only_risk_on_test(model_ab.get("M_train"), model_ab.get("main_effect_vars", []), model_ab["T_train"], model_ab["E_train"], va_pack.get("M"))
        risk_maingp_va = predict_ab_risk_on_test(model_ab["F_train"], model_ab["T_train"], model_ab["E_train"], va_pack["F"], va_pack["T"], va_pack["E"], {}, model_ab["toolbox_t"], M_dev=model_ab.get("M_train"), M_te=va_pack.get("M"), main_effect_names=model_ab.get("main_effect_vars", [])) if va_pack.get("F") is not None else None
        if risk_main_va is not None and len(risk_main_va) == len(va_idx):
            oof_main_only_risk[va_idx] = np.asarray(risk_main_va, dtype=float)
        if risk_maingp_va is not None and len(risk_maingp_va) == len(va_idx):
            oof_main_gp_risk[va_idx] = np.asarray(risk_maingp_va, dtype=float)
        if risk_ab_va is not None and len(risk_ab_va) == len(va_idx):
            oof_ab_risk[va_idx] = np.asarray(risk_ab_va, dtype=float)

        tr_pack = {
            "df_enc": model_ab["train_enc"],
            "T": model_ab["T_train"],
            "E": model_ab["E_train"],
            "F": model_ab["F_train"],
            "M": model_ab.get("M_train")
        }
        base_risk_tr = predict_ab_risk_on_test(
            tr_pack["F"], tr_pack["T"], tr_pack["E"],
            tr_pack["F"], tr_pack["T"], tr_pack["E"],
            model_ab["g_fixed"], model_ab["toolbox_t"],
            M_dev=tr_pack.get("M"), M_te=tr_pack.get("M"), main_effect_names=model_ab.get("main_effect_vars", [])
        )
        C_tr_df, fmap = build_phasec_candidate_terminals(tr_pack["df_enc"], model_ab["prep"])
        C_va_df, _ = build_phasec_candidate_terminals(va_pack["df_enc"], model_ab["prep"])
        for col in C_tr_df.columns:
            if col not in C_va_df.columns:
                C_va_df[col] = 0.0
        C_va_df = C_va_df[C_tr_df.columns]

        fold_cache.append({
            "fold": fold,
            "tr_idx": tr_idx,
            "va_idx": va_idx,
            "model_ab": model_ab,
            "c_ab": c_ab,
            "risk_ab_va": risk_ab_va,
            "risk_main_va": risk_main_va,
            "risk_maingp_va": risk_maingp_va,
            "va_pack": va_pack,
            "tr_pack": tr_pack,
            "base_risk_tr": base_risk_tr,
            "C_tr_df": C_tr_df,
            "C_va_df": C_va_df,
            "fmap": fmap,
        })

    print("\n" + "-" * 80)
    print("Phase-C K-search based on internal 4-fold validation")
    print("-" * 80)

    for k in k_grid:
        k = int(k)
        print(f"[PhaseC-KSearch] Start K={k}")
        for fc in fold_cache:
            fold = int(fc["fold"])
            if fc["model_ab"] is None or fc.get("risk_ab_va") is None or not np.isfinite(fc.get("c_ab", np.nan)):
                phasec_search_rows.append({
                    "K_interaction": k,
                    "fold": fold,
                    "ab_cindex": np.nan,
                    "abc_cindex": np.nan,
                    "delta": np.nan,
                    "n_candidate_trees": 0,
                    "n_pass_support_trees": 0,
                    "actual_selected_trees_n": 0,
                    "selected_interaction_trees": json.dumps([], ensure_ascii=False),
                    "inherited_main_effects": json.dumps([], ensure_ascii=False),
                    "coefficients": json.dumps({}, ensure_ascii=False),
                    "n_gp": np.nan,
                    "time_interactions": np.nan,
                })
                continue

            c_ab = float(fc["c_ab"])
            risk_ab_va = np.asarray(fc["risk_ab_va"], dtype=float)
            va_idx = fc["va_idx"]
            if k == 0:
                c_abc = c_ab
                risk_abc_va = risk_ab_va.copy()
                n_cand = 0
                n_pass = 0
                selected_expr = []
                inherited = []
                coefs = {}
            else:
                c_params = get_fixed_phasec_params(k)
                old = _temporary_set_params(c_params)
                try:
                    trees, factor_map, terminal_names = search_phasec_trees(
                        fc["C_tr_df"],
                        fc["base_risk_tr"],
                        fc["tr_pack"]["T"],
                        fc["tr_pack"]["E"],
                        c_params,
                        factor_map=fc["fmap"]
                    )
                    n_cand = int(len(trees))
                    toolbox_c, _ = build_gp_toolbox_c(fc["C_tr_df"].shape[1])
                    fit_res = fit_phasec_model_with_ab_baseline(
                        trees, toolbox_c, fc["C_tr_df"], fc["C_va_df"],
                        fc["base_risk_tr"], risk_ab_va,
                        fc["tr_pack"]["T"], fc["tr_pack"]["E"],
                        terminal_names, factor_map, ab_main_effects=[]
                    )
                finally:
                    _restore_params(old)
                _, c_abc, _ = score_ab_plus_c_on_validation(fc["va_pack"]["T"], fc["va_pack"]["E"], risk_ab_va, fit_res)
                risk_abc_va = np.asarray(fit_res.get("risk_valid", []), dtype=float)
                n_pass = int(fit_res.get("n_tree_pass_support", 0))
                selected_expr = fit_res.get("selected_interaction_trees", [])
                inherited = fit_res.get("inherited_main_effects", [])
                coefs = fit_res.get("coefficients", {})

            if len(risk_abc_va) == len(va_idx):
                oof_abc_risk_by_k[k][va_idx] = risk_abc_va

            dlt = float(c_abc - c_ab) if np.isfinite(c_ab) and np.isfinite(c_abc) else np.nan
            print(
                f"[PhaseC-KSearch] fold={fold} K={k} | "
                f"n_candidate={int(n_cand)} | n_pass_support={int(n_pass)} | "
                f"actual_selected_trees={int(len(selected_expr))} | "
                f"AB={c_ab:.4f} | ABC={c_abc:.4f} | delta={dlt:.4f}"
            )
            phasec_search_rows.append({
                "K_interaction": k,
                "fold": fold,
                "ab_cindex": c_ab if np.isfinite(c_ab) else np.nan,
                "abc_cindex": c_abc if np.isfinite(c_abc) else np.nan,
                "delta": dlt,
                "n_candidate_trees": int(n_cand),
                "n_pass_support_trees": int(n_pass),
                "actual_selected_trees_n": int(len(selected_expr)),
                "selected_interaction_trees": json.dumps(selected_expr, ensure_ascii=False),
                "inherited_main_effects": json.dumps(inherited, ensure_ascii=False),
                "coefficients": json.dumps(coefs, ensure_ascii=False),
                "n_gp": int(fc["model_ab"].get("actual_n_gp", np.nan)) if fc["model_ab"] is not None else np.nan,
                "time_interactions": int(len(fc["model_ab"].get("g_fixed", {}))) if fc["model_ab"] is not None else np.nan,
                "main_effect_vars": json.dumps(fc["model_ab"].get("main_effect_vars", []), ensure_ascii=False) if fc["model_ab"] is not None else "[]",
                "gp_input_vars": json.dumps(fc["model_ab"].get("gp_input_vars", []), ensure_ascii=False) if fc["model_ab"] is not None else "[]",
                "main_effect_indices": json.dumps(fc["model_ab"].get("main_effect_indices", []), ensure_ascii=False) if fc["model_ab"] is not None else "[]",
                "kept_phaseB_interactions": json.dumps(fc["model_ab"].get("kept_phaseB_interactions", []), ensure_ascii=False) if fc["model_ab"] is not None else "[]",
                "ph_violating_main_effects": json.dumps(fc["model_ab"].get("ph_violating_main_effects", []), ensure_ascii=False) if fc["model_ab"] is not None else "[]",
                "ph_violating_gp_names": json.dumps(fc["model_ab"].get("ph_violating_gp_names", []), ensure_ascii=False) if fc["model_ab"] is not None else "[]",
                "main_only_cindex": fc["model_ab"].get("main_only_cindex", np.nan) if fc["model_ab"] is not None else np.nan,
                "main_gp_cindex": fc["model_ab"].get("main_gp_cindex", np.nan) if fc["model_ab"] is not None else np.nan,
                "main_gp_phaseB_cindex": fc["model_ab"].get("main_gp_phaseB_cindex", np.nan) if fc["model_ab"] is not None else np.nan,
            })

    df_phasec_search = pd.DataFrame(phasec_search_rows)
    summary_rows = []
    for k in k_grid:
        dfk = df_phasec_search[df_phasec_search["K_interaction"] == int(k)].copy()
        ab_vals = pd.to_numeric(dfk["ab_cindex"], errors="coerce").values if len(dfk) else np.array([])
        abc_vals = pd.to_numeric(dfk["abc_cindex"], errors="coerce").values if len(dfk) else np.array([])
        dlt_vals = pd.to_numeric(dfk["delta"], errors="coerce").values if len(dfk) else np.array([])
        tree_vals = pd.to_numeric(dfk["actual_selected_trees_n"], errors="coerce").values if len(dfk) else np.array([])

        mean_ab = float(np.nanmean(ab_vals)) if np.isfinite(ab_vals).any() else np.nan
        mean_abc = float(np.nanmean(abc_vals)) if np.isfinite(abc_vals).any() else np.nan
        mean_delta = float(np.nanmean(dlt_vals)) if np.isfinite(dlt_vals).any() else np.nan
        sd_abc = float(np.nanstd(abc_vals, ddof=0)) if np.isfinite(abc_vals).any() else np.nan
        positive_folds = int(np.sum(dlt_vals > 0)) if len(dlt_vals) else 0
        mean_actual_selected_trees = float(np.nanmean(tree_vals)) if np.isfinite(tree_vals).any() else 0.0
        score = (
            mean_abc
            + 0.25 * mean_delta
            - 0.15 * sd_abc
            - 0.001 * int(k)
        ) if np.isfinite(mean_abc) and np.isfinite(mean_delta) and np.isfinite(sd_abc) else np.nan
        eligible = bool(int(k) > 0 and np.isfinite(mean_delta) and mean_delta > float(PHASEC_MEAN_DELTA_MIN) and positive_folds >= int(PHASEC_POSITIVE_FOLDS_MIN))
        summary_rows.append({
            "K_interaction": int(k),
            "mean_ab": mean_ab,
            "mean_abc": mean_abc,
            "mean_delta": mean_delta,
            "sd_abc": sd_abc,
            "positive_folds": positive_folds,
            "mean_actual_selected_trees": mean_actual_selected_trees,
            "score": score,
            "eligible": int(eligible),
        })

    df_k_summary = pd.DataFrame(summary_rows).sort_values("K_interaction").reset_index(drop=True)
    eligible_df = df_k_summary[df_k_summary["eligible"] == 1].copy()
    if len(eligible_df) == 0:
        selected_k = 0
    else:
        selected_k = int(eligible_df.sort_values(["score", "mean_delta"], ascending=[False, False]).iloc[0]["K_interaction"])
    df_k_summary["selected"] = (df_k_summary["K_interaction"] == selected_k).astype(int)

    for _, r in df_k_summary.iterrows():
        print(
            f"[PhaseC-KSearch] Summary K={int(r['K_interaction'])} | "
            f"mean_ab={float(r['mean_ab']):.4f} | mean_abc={float(r['mean_abc']):.4f} | "
            f"mean_delta={float(r['mean_delta']):.4f} | positive_folds={int(r['positive_folds'])} | "
            f"selected={int(r['selected'])}"
        )
    print(f"[PhaseC-KSearch] Selected K = {selected_k}")

    oof_abc_risk = np.asarray(oof_abc_risk_by_k.get(selected_k, np.full(len(df_train_raw), np.nan)), dtype=float)
    if int(selected_k) == 0:
        oof_abc_risk = np.asarray(oof_ab_risk, dtype=float).copy()

    def _safe_cindex(T, E, risk):
        mask = np.isfinite(T) & np.isfinite(E) & np.isfinite(risk)
        if int(mask.sum()) < 3:
            return np.nan
        try:
            return float(concordance_index(T[mask], -risk[mask], E[mask].astype(int)))
        except Exception:
            return np.nan

    T_all = df_train_raw[TIME_COL].astype(float).values
    E_all = df_train_raw[EVENT_COL].astype(int).values
    oof_main_only_cindex = _safe_cindex(T_all, E_all, np.asarray(oof_main_only_risk, dtype=float))
    oof_main_gp_cindex = _safe_cindex(T_all, E_all, np.asarray(oof_main_gp_risk, dtype=float))
    oof_ab_cindex = _safe_cindex(T_all, E_all, np.asarray(oof_ab_risk, dtype=float))
    oof_abc_cindex = _safe_cindex(T_all, E_all, np.asarray(oof_abc_risk, dtype=float))
    oof_delta = oof_abc_cindex - oof_ab_cindex if np.isfinite(oof_ab_cindex) and np.isfinite(oof_abc_cindex) else np.nan

    df_sel = df_phasec_search[df_phasec_search["K_interaction"] == int(selected_k)].copy()
    ab_fold_cidx = []
    abc_fold_cidx = []
    for fold in range(1, int(n_splits) + 1):
        row0 = df_phasec_search[(df_phasec_search["K_interaction"] == 0) & (df_phasec_search["fold"] == fold)]
        rows = df_sel[df_sel["fold"] == fold]
        ab_fold_cidx.append(float(pd.to_numeric(row0["ab_cindex"], errors="coerce").iloc[0]) if len(row0) else np.nan)
        abc_fold_cidx.append(float(pd.to_numeric(rows["abc_cindex"], errors="coerce").iloc[0]) if len(rows) else np.nan)
    ab_mean_cindex = float(np.nanmean(ab_fold_cidx)) if np.isfinite(np.asarray(ab_fold_cidx, dtype=float)).any() else np.nan
    abc_mean_cindex = float(np.nanmean(abc_fold_cidx)) if np.isfinite(np.asarray(abc_fold_cidx, dtype=float)).any() else np.nan
    mean_delta = abc_mean_cindex - ab_mean_cindex if np.isfinite(ab_mean_cindex) and np.isfinite(abc_mean_cindex) else np.nan

    fold_summary_rows = []
    for fold in range(1, int(n_splits) + 1):
        rowk = df_sel[df_sel["fold"] == fold]
        if len(rowk):
            rr = rowk.iloc[0]
            ab_final = float(rr["ab_cindex"]) if np.isfinite(rr["ab_cindex"]) else np.nan
            abc_final = float(rr["abc_cindex"]) if np.isfinite(rr["abc_cindex"]) else np.nan
            dlt = float(rr["delta"]) if np.isfinite(rr["delta"]) else np.nan
            trees_n = int(rr["actual_selected_trees_n"])
        else:
            ab_final = np.nan
            abc_final = np.nan
            dlt = np.nan
            trees_n = 0
        fcm = [x for x in fold_cache if int(x["fold"]) == int(fold)]
        n_gp = int(fcm[0]["model_ab"].get("actual_n_gp", np.nan)) if fcm and fcm[0]["model_ab"] is not None else np.nan
        t_inter = int(len(fcm[0]["model_ab"].get("g_fixed", {}))) if fcm and fcm[0]["model_ab"] is not None else np.nan
        model_fold = fcm[0]["model_ab"] if fcm and fcm[0]["model_ab"] is not None else {}
        fold_summary_rows.append({
            "fold": int(fold),
            "main_only_cindex": model_fold.get("main_only_cindex", np.nan),
            "main_gp_cindex": model_fold.get("main_gp_cindex", np.nan),
            "main_gp_phaseB_cindex": model_fold.get("main_gp_phaseB_cindex", ab_final),
            "ab_final_cindex": ab_final,
            "abc_final_cindex": abc_final,
            "delta": dlt,
            "selected_k": int(selected_k),
            "n_gp": n_gp,
            "time_interactions": t_inter,
            "actual_selected_trees": trees_n,
            "main_effect_vars": json.dumps(model_fold.get("main_effect_vars", []), ensure_ascii=False),
            "gp_input_vars": json.dumps(model_fold.get("gp_input_vars", []), ensure_ascii=False),
            "main_effect_indices": json.dumps(model_fold.get("main_effect_indices", []), ensure_ascii=False),
            "kept_phaseB_interactions": json.dumps(model_fold.get("kept_phaseB_interactions", []), ensure_ascii=False),
            "ph_violating_main_effects": json.dumps(model_fold.get("ph_violating_main_effects", []), ensure_ascii=False),
            "ph_violating_gp_names": json.dumps(model_fold.get("ph_violating_gp_names", []), ensure_ascii=False),
            "enable_main_effects": model_fold.get("enable_main_effects", np.nan),
            "main_effect_k": model_fold.get("main_effect_k", np.nan),
        })

    fold_summary_df_tmp = pd.DataFrame(fold_summary_rows)
    main_only_fold_cidx = [float(x) if np.isfinite(x) else np.nan for x in pd.to_numeric(fold_summary_df_tmp.get("main_only_cindex", pd.Series(dtype=float)), errors="coerce").values]
    main_gp_fold_cidx = [float(x) if np.isfinite(x) else np.nan for x in pd.to_numeric(fold_summary_df_tmp.get("main_gp_cindex", pd.Series(dtype=float)), errors="coerce").values]
    main_gp_phaseB_fold_cidx = [float(x) if np.isfinite(x) else np.nan for x in pd.to_numeric(fold_summary_df_tmp.get("main_gp_phaseB_cindex", pd.Series(dtype=float)), errors="coerce").values]
    main_only_mean_cindex = float(np.nanmean(main_only_fold_cidx)) if np.isfinite(np.asarray(main_only_fold_cidx, dtype=float)).any() else np.nan
    main_gp_mean_cindex = float(np.nanmean(main_gp_fold_cidx)) if np.isfinite(np.asarray(main_gp_fold_cidx, dtype=float)).any() else np.nan
    main_gp_phaseB_mean_cindex = float(np.nanmean(main_gp_phaseB_fold_cidx)) if np.isfinite(np.asarray(main_gp_phaseB_fold_cidx, dtype=float)).any() else np.nan

    phaseb_timebasis_fold_summary = pd.concat([(fc.get("model_ab") or {}).get("phaseb_timebasis", {}).get("fold_summary", pd.DataFrame()) for fc in fold_cache], axis=0, ignore_index=True) if len(fold_cache) else pd.DataFrame()
    phaseb_timebasis_candidates_raw = pd.DataFrame([r for fc in fold_cache for r in ((fc.get("model_ab") or {}).get("phaseb_timebasis", {}).get("raw_candidates", []) if fc.get("model_ab") is not None else [])])
    phaseb_timebasis_candidate_scores = pd.concat([(fc.get("model_ab") or {}).get("phaseb_timebasis", {}).get("candidate_scores", pd.DataFrame()) for fc in fold_cache], axis=0, ignore_index=True) if len(fold_cache) else pd.DataFrame()
    phaseb_timebasis_selected_by_fold = pd.concat([(fc.get("model_ab") or {}).get("phaseb_timebasis", {}).get("selected_candidates_df", pd.DataFrame()) for fc in fold_cache], axis=0, ignore_index=True) if len(fold_cache) else pd.DataFrame()
    if str(globals().get("PHASEB_MODE", "time_basis_pool")).lower() == "time_basis_pool":
        tb_ab = pd.to_numeric(phaseb_timebasis_fold_summary.get("valid_ab_cindex", pd.Series(dtype=float)), errors="coerce").values
        tb_pb = pd.to_numeric(phaseb_timebasis_fold_summary.get("valid_phaseb_cindex", pd.Series(dtype=float)), errors="coerce").values
        tb_delta = pd.to_numeric(phaseb_timebasis_fold_summary.get("delta", pd.Series(dtype=float)), errors="coerce").values
        phaseB_oof_delta = float(np.nanmean(tb_delta)) if np.isfinite(tb_delta).any() else np.nan
        phaseB_positive_folds = int(np.sum(tb_delta > 0)) if len(tb_delta) else 0
        phaseB_allowed_final = (not bool(FINAL_PHASEB_GATE)) or (np.isfinite(phaseB_oof_delta) and phaseB_oof_delta > float(PHASEB_MEAN_DELTA_MIN) and phaseB_positive_folds >= int(PHASEB_POSITIVE_FOLDS_MIN))
        print("\n" + "=" * 80)
        print("[PhaseB-TimeBasis]")
        print(f"mean A C-index = {float(np.nanmean(tb_ab)) if np.isfinite(tb_ab).any() else np.nan:.4f}")
        print(f"mean AB C-index = {float(np.nanmean(tb_pb)) if np.isfinite(tb_pb).any() else np.nan:.4f}")
        print(f"[PhaseB-TimeBasis] gate rule: mean_delta > {PHASEB_MEAN_DELTA_MIN} and positive_folds >= {PHASEB_POSITIVE_FOLDS_MIN}")
        print(f"mean_delta = {phaseB_oof_delta:.4f}")
        print(f"positive_folds = {phaseB_positive_folds}")
        print(f"allowed = {bool(phaseB_allowed_final)}")
        print("=" * 80)
    else:
        phaseB_oof_delta = oof_ab_cindex - oof_main_gp_cindex if np.isfinite(oof_ab_cindex) and np.isfinite(oof_main_gp_cindex) else np.nan
        phaseB_positive_folds = int(np.sum(np.asarray(main_gp_phaseB_fold_cidx, dtype=float) > np.asarray(main_gp_fold_cidx, dtype=float))) if len(main_gp_fold_cidx) else 0
        phaseB_allowed_final = (not bool(FINAL_PHASEB_GATE)) or (np.isfinite(phaseB_oof_delta) and phaseB_oof_delta > float(PHASEB_MEAN_DELTA_MIN) and phaseB_positive_folds >= int(PHASEB_POSITIVE_FOLDS_MIN))
        print("\n" + "=" * 80)
        print("[Phase-B Gate]")
        print(f"A/main+GP OOF C-index = {oof_main_gp_cindex:.4f}")
        print(f"AB/main+GP+PhaseB OOF C-index = {oof_ab_cindex:.4f}")
        print(f"OOF delta = {phaseB_oof_delta:.4f}")
        print(f"positive folds = {phaseB_positive_folds}")
        print(f"Final full-train Phase-B allowed = {bool(phaseB_allowed_final)}")
        print("=" * 80)
    return {
        "selected_k": int(selected_k),
        "selected_c_params": get_fixed_phasec_params(selected_k),
        "k_summary": df_k_summary,
        "fold_records": df_phasec_search,
        "fold_ab_gen_trajectory": pd.DataFrame(fold_ab_gen_rows),
        "fold_summary": pd.DataFrame(fold_summary_rows),
        "phasea_fold_traj": pd.DataFrame(phasea_fold_traj_rows),
        "oof_main_only_risk": np.asarray(oof_main_only_risk, dtype=float),
        "oof_main_gp_risk": np.asarray(oof_main_gp_risk, dtype=float),
        "oof_ab_risk": np.asarray(oof_ab_risk, dtype=float),
        "oof_abc_risk": np.asarray(oof_abc_risk, dtype=float),
        "main_only_fold_cindexes": main_only_fold_cidx,
        "baseline_stable_main_only_mean_cindex": main_only_mean_cindex,
        "main_gp_fold_cindexes": main_gp_fold_cidx,
        "baseline_stable_main_gp_mean_cindex": main_gp_mean_cindex,
        "main_gp_phaseB_fold_cindexes": main_gp_phaseB_fold_cidx,
        "baseline_stable_main_gp_phaseB_mean_cindex": main_gp_phaseB_mean_cindex,
        "phaseB_oof_delta": phaseB_oof_delta,
        "phaseb_delta_AB_minus_A": phaseB_oof_delta,
        "phaseB_positive_folds": phaseB_positive_folds,
        "phaseB_allowed_final": bool(phaseB_allowed_final),
        "kept_phaseB_interactions_by_fold": [json.loads(x) if isinstance(x, str) else [] for x in fold_summary_df_tmp.get("kept_phaseB_interactions", pd.Series(dtype=object)).tolist()],
        "phaseb_timebasis_fold_summary": phaseb_timebasis_fold_summary,
        "phaseb_timebasis_candidates_raw": phaseb_timebasis_candidates_raw,
        "phaseb_timebasis_candidate_scores": phaseb_timebasis_candidate_scores,
        "phaseb_timebasis_selected_by_fold": phaseb_timebasis_selected_by_fold,
        "oof_main_only_cindex": oof_main_only_cindex,
        "oof_main_gp_cindex": oof_main_gp_cindex,
        "oof_ab_cindex": oof_ab_cindex,
        "oof_abc_cindex": oof_abc_cindex,
        "oof_delta": oof_delta,
        "ab_fold_cindexes": [float(x) if np.isfinite(x) else np.nan for x in ab_fold_cidx],
        "ab_mean_cindex": ab_mean_cindex,
        "abc_fold_cindexes": [float(x) if np.isfinite(x) else np.nan for x in abc_fold_cidx],
        "abc_mean_cindex": abc_mean_cindex,
        "abc_vs_ab_mean_delta": mean_delta,
        "abc_vs_ab_oof_delta": oof_delta,
        "ab_params": dict(ab_params),
    }



run_internal_fixed_cv_select_phasec_original = run_internal_fixed_cv_select_phasec


def run_internal_fixed_cv_select_phasec(
    df_train_raw,
    ab_params,
    k_grid=(0, 1, 2, 3),
    n_splits=4,
):
    if str(globals().get("PHASEC_MODE", "stgp_and")).lower() not in {"stgp_and", "off"}:
        return run_internal_fixed_cv_select_phasec_original(df_train_raw, ab_params, k_grid=k_grid, n_splits=n_splits)

    df_train_raw = _coerce_time_event(df_train_raw).reset_index(drop=True)
    y = df_train_raw[EVENT_COL].values
    cv = SurvivalAwareKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    mode = str(globals().get("PHASEC_MODE", "stgp_and")).lower()
    k_grid = tuple(sorted(set(int(k) for k in (resolve_phasec_k_values(globals().get("PHASEC_K_PROFILE", "legacy_k_0_1_2_3"), PHASEC_K_VALUES) if mode == "stgp_and" else [0]) if 0 <= int(k) <= 4)))
    if len(k_grid) == 0:
        k_grid = (0,)

    fold_ab_gen_rows, phasea_fold_traj_rows, fold_cache = [], [], []
    oof_main_only_risk = np.full(len(df_train_raw), np.nan, dtype=float)
    oof_main_gp_risk = np.full(len(df_train_raw), np.nan, dtype=float)
    oof_ab_risk = np.full(len(df_train_raw), np.nan, dtype=float)
    oof_abc_risk_by_k = {int(k): np.full(len(df_train_raw), np.nan, dtype=float) for k in k_grid}
    phasec_raw_all, phasec_feasible_all, phasec_dropped_all, phasec_scores_all, phasec_selected_all = [], [], [], [], []
    phasec_population_init_all, phasec_generation_log_all, phasec_hof_all = [], [], []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(np.arange(len(y)), y), 1):
        df_tr = df_train_raw.iloc[tr_idx].reset_index(drop=True)
        df_va = df_train_raw.iloc[va_idx].reset_index(drop=True)
        model_ab, gen_rows, phasea_rows = _fit_ab_backbone_on_train_with_valid_logging(
            df_tr, df_va, ab_params, fold_idx=fold, inner_folds_for_varsel=n_splits
        )
        fold_ab_gen_rows.extend(gen_rows)
        phasea_fold_traj_rows.extend(phasea_rows)
        if model_ab is None:
            fold_cache.append({"fold": fold, "tr_idx": tr_idx, "va_idx": va_idx, "model_ab": None})
            continue
        c_ab, va_pack, risk_ab_va = _score_ab_on_validation(model_ab, df_va)
        risk_main_va = predict_main_only_risk_on_test(model_ab.get("M_train"), model_ab.get("main_effect_vars", []), model_ab["T_train"], model_ab["E_train"], va_pack.get("M"))
        risk_maingp_va = predict_ab_risk_on_test(model_ab["F_train"], model_ab["T_train"], model_ab["E_train"], va_pack["F"], va_pack["T"], va_pack["E"], {}, model_ab["toolbox_t"], M_dev=model_ab.get("M_train"), M_te=va_pack.get("M"), main_effect_names=model_ab.get("main_effect_vars", [])) if va_pack.get("F") is not None else None
        if risk_main_va is not None and len(risk_main_va) == len(va_idx):
            oof_main_only_risk[va_idx] = np.asarray(risk_main_va, dtype=float)
        if risk_maingp_va is not None and len(risk_maingp_va) == len(va_idx):
            oof_main_gp_risk[va_idx] = np.asarray(risk_maingp_va, dtype=float)
        if risk_ab_va is not None and len(risk_ab_va) == len(va_idx):
            oof_ab_risk[va_idx] = np.asarray(risk_ab_va, dtype=float)

        tr_pack = {"df_enc": model_ab["train_enc"], "T": model_ab["T_train"], "E": model_ab["E_train"], "F": model_ab["F_train"], "M": model_ab.get("M_train")}
        base_risk_tr = predict_ab_risk_on_test(tr_pack["F"], tr_pack["T"], tr_pack["E"], tr_pack["F"], tr_pack["T"], tr_pack["E"], model_ab["g_fixed"], model_ab["toolbox_t"], M_dev=tr_pack.get("M"), M_te=tr_pack.get("M"), main_effect_names=model_ab.get("main_effect_vars", []))

        raw, filtered, dropped, score_df, phasec_context = phasec_stgp_and_prepare_fold(
            tr_pack["df_enc"], model_ab.get("prep", {}), tr_pack["T"], tr_pack["E"], base_risk_tr, fold_label=int(fold)
        ) if mode == "stgp_and" else ([], [], [], pd.DataFrame(), {})
        evo = {}
        if mode == "stgp_and":
            evo = phasec_stgp_and_run_evolution(raw, filtered, tr_pack["df_enc"], tr_pack["T"], tr_pack["E"], base_risk_tr, prep=model_ab.get("prep", {}), fold_label=int(fold), phasec_context=phasec_context)
            score_df = evo.get("candidate_scores", score_df)
            if len(evo.get("population_init", pd.DataFrame())): phasec_population_init_all.append(evo.get("population_init"))
            if len(evo.get("generation_log", pd.DataFrame())): phasec_generation_log_all.append(evo.get("generation_log"))
            if len(evo.get("hof_df", pd.DataFrame())): phasec_hof_all.append(evo.get("hof_df"))
        phasec_raw_all.extend(raw); phasec_feasible_all.extend(filtered); phasec_dropped_all.extend(dropped)
        if score_df is not None and len(score_df):
            phasec_scores_all.append(score_df)
        fold_cache.append({
            "fold": fold, "tr_idx": tr_idx, "va_idx": va_idx, "model_ab": model_ab,
            "c_ab": c_ab, "risk_ab_va": risk_ab_va, "risk_main_va": risk_main_va, "risk_maingp_va": risk_maingp_va,
            "va_pack": va_pack, "tr_pack": tr_pack, "base_risk_tr": base_risk_tr, "phasec_stgp_scores": score_df,
            "phasec_stgp_raw": raw, "phasec_stgp_feasible": filtered, "phasec_stgp_dropped": dropped, "phasec_stgp_context": phasec_context,
            "phasec_stgp_and_hof": evo.get("hof", []) if mode == "stgp_and" else [],
        })

    phasec_search_rows = []
    print("[PhaseC-STGP-AND] Phase C K-search base = AB_timevarying when PhaseB is active, otherwise A_static")
    print("[PhaseC-STGP-AND] K=0 uses AB_timevarying/A_static; K>0 uses unified ABC_timevarying when PhaseB is active, otherwise AC_static.")
    for k in k_grid:
        k = int(k)
        for fc in fold_cache:
            fold = int(fc["fold"])
            if fc.get("model_ab") is None or fc.get("va_pack") is None:
                phasec_search_rows.append({"K_interaction": k, "fold": fold, "ab_cindex": np.nan, "abc_cindex": np.nan, "delta": np.nan, "actual_selected_trees_n": 0, "selected_interaction_trees": "[]", "reason": "missing_ab"})
                continue
            selected = fc.get("phasec_stgp_and_hof", [])[:int(k)] if mode == "stgp_and" and int(k) > 0 else []
            tb = (fc.get("model_ab") or {}).get("phaseb_timebasis", {}) or {}
            selected_b_fold = list(tb.get("selected_candidates", []) or [])
            phaseB_active_fold = bool(len(selected_b_fold) > 0)
            phaseC_active_fold = bool(int(k) > 0 and len(selected) > 0)
            c_ab = float(tb.get("valid_phaseb_cindex", fc.get("c_ab", np.nan))) if phaseB_active_fold else float(fc["model_ab"].get("main_gp_cindex", fc.get("c_ab", np.nan)))
            unified = {"coefficients": {}, "reason": "not_called", "design_cols": [], "selected_phaseB_candidates": selected_b_fold}
            if phaseB_active_fold and phaseC_active_fold:
                unified = fit_eval_unified_abc_from_model(fc["model_ab"], fc["va_pack"], selected_phaseC_candidates=selected, phaseB_allowed=True, fold_label=int(fold))
                c_abc = float(unified.get("cindex", np.nan))
                fallback_mode = "unified_ABC"
            elif phaseB_active_fold and not phaseC_active_fold:
                c_abc = c_ab
                fallback_mode = "equals_AB"
            elif (not phaseB_active_fold) and phaseC_active_fold:
                risk_a_va = np.asarray(fc.get("risk_maingp_va", fc.get("risk_ab_va")), dtype=float)
                fit_res_static = phasec_stgp_and_fit_selected(selected, fc["tr_pack"]["df_enc"], fc["va_pack"]["df_enc"], fc["tr_pack"]["T"], fc["tr_pack"]["E"], fc["base_risk_tr"], risk_a_va)
                risk_ac_va = np.asarray(fit_res_static.get("risk_valid", risk_a_va), dtype=float)
                c_abc = float(concordance_index(fc["va_pack"]["T"], -risk_ac_va, fc["va_pack"]["E"])) if len(risk_ac_va) == len(fc["va_idx"]) else np.nan
                fallback_mode = "equals_AC"
                if np.isfinite(c_abc):
                    ac_check = c_abc
                    if abs(float(c_abc) - float(ac_check)) > 1e-8:
                        print("[ERROR] PhaseB inactive fold but ABC_timevarying != AC_static. This indicates evaluator fallback bug.")
            else:
                c_abc = c_ab
                fallback_mode = "equals_A"
            dlt = c_abc - c_ab if np.isfinite(c_abc) and np.isfinite(c_ab) else np.nan
            selected_names = [c.get("name", "") for c in selected]
            log_prefix = "PhaseC-STGP-AND"
            if int(k) == 0:
                base_label = "AB_timevarying" if phaseB_active_fold else "A_static"
                print(f"[{log_prefix}][Fold {fold}][K=0] phaseB_active={phaseB_active_fold}; using {base_label}")
                print(f"[{log_prefix}][Fold {fold}][K=0] {base_label} C-index = {c_abc:.4f}")
            else:
                method_label = "unified ABC_timevarying" if phaseB_active_fold else "AC_static"
                score_label = "ABC_timevarying" if phaseB_active_fold else "AC_static"
                print(f"[{log_prefix}][Fold {fold}][K={k}] phaseB_active={phaseB_active_fold}; using {method_label}")
                print(f"[{log_prefix}][Fold {fold}][K={k}] {score_label} C-index = {c_abc:.4f} delta_vs_base={dlt:.4f} selected_candidates={selected_names}")
            for cand in selected:
                rr = dict(cand); rr.update({"fold": fold, "selected_k": k, "valid_cindex": c_abc, "delta": dlt, "selected": True})
                phasec_selected_all.append(rr)
            phasec_search_rows.append({
                "K_interaction": k, "fold": fold, "ab_cindex": c_ab, "abc_cindex": c_abc, "delta": dlt,
                "AB_timevarying_cindex": c_ab, "ABC_timevarying_cindex": c_abc,
                "delta_ABC_minus_AB": dlt,
                "n_candidate_trees": int(len(fc.get("phasec_stgp_raw", []))),
                "n_pass_support_trees": int(len(fc.get("phasec_stgp_feasible", []))),
                "actual_selected_trees_n": int(len(selected)),
                "selected_interaction_trees": json.dumps(selected_names, ensure_ascii=False),
                "inherited_main_effects": json.dumps([], ensure_ascii=False),
                "coefficients": json.dumps(unified.get("coefficients", {}), ensure_ascii=False),
                "reason": unified.get("reason", "ok"),
                "phaseB_active_fold": bool(phaseB_active_fold),
                "phaseC_active_fold": bool(phaseC_active_fold),
                "unified_abc_fallback_mode": fallback_mode,
                "base_fit_once": True,
                "support_precomputed": True,
                "fitness_cache_enabled": True,
                "initialization": "random_legal",
                "abc_unified": True,
                "design_cols": json.dumps(unified.get("design_cols", []), ensure_ascii=False),
                "phaseb_selected_candidates": json.dumps([c.get("name", "") for c in unified.get("selected_phaseB_candidates", [])], ensure_ascii=False),
                "phasec_selected_candidates": json.dumps(selected_names, ensure_ascii=False),
                "main_only_cindex": fc["model_ab"].get("main_only_cindex", np.nan),
                "main_gp_cindex": fc["model_ab"].get("main_gp_cindex", np.nan),
                "main_gp_phaseB_cindex": fc["model_ab"].get("main_gp_phaseB_cindex", np.nan),
                "kept_phaseB_interactions": json.dumps(fc["model_ab"].get("kept_phaseB_interactions", []), ensure_ascii=False),
                "ph_violating_main_effects": json.dumps(fc["model_ab"].get("ph_violating_main_effects", []), ensure_ascii=False),
                "ph_violating_gp_names": json.dumps(fc["model_ab"].get("ph_violating_gp_names", []), ensure_ascii=False),
            })

    df_phasec_search = pd.DataFrame(phasec_search_rows)
    summary_rows = []
    for k in k_grid:
        dfk = df_phasec_search[df_phasec_search["K_interaction"] == int(k)].copy()
        ab_vals = pd.to_numeric(dfk["ab_cindex"], errors="coerce").values if len(dfk) else np.array([])
        abc_vals = pd.to_numeric(dfk["abc_cindex"], errors="coerce").values if len(dfk) else np.array([])
        dlt_vals = pd.to_numeric(dfk["delta"], errors="coerce").values if len(dfk) else np.array([])
        mean_ab = float(np.nanmean(ab_vals)) if np.isfinite(ab_vals).any() else np.nan
        mean_abc = float(np.nanmean(abc_vals)) if np.isfinite(abc_vals).any() else np.nan
        if int(k) == 0:
            base_mean = mean_abc
        mean_delta = mean_abc - base_mean if np.isfinite(mean_abc) and np.isfinite(base_mean) else np.nan
        positive_folds = int(np.sum(dlt_vals > 0)) if len(dlt_vals) else 0
        eligible = bool(int(k) > 0 and np.isfinite(mean_delta) and mean_delta > float(PHASEC_MEAN_DELTA_MIN) and positive_folds >= int(PHASEC_POSITIVE_FOLDS_MIN))
        summary_rows.append({"K_interaction": int(k), "mean_ab": mean_ab, "mean_abc": mean_abc, "mean_delta": mean_delta, "phasec_A_static_base_cindex": mean_ab, "phasec_AC_static_cindex": mean_abc, "phasec_delta_AC_minus_A": mean_delta, "sd_abc": float(np.nanstd(abc_vals, ddof=0)) if np.isfinite(abc_vals).any() else np.nan, "positive_folds": positive_folds, "score": mean_abc, "eligible": int(eligible)})
    df_k_summary = pd.DataFrame(summary_rows).sort_values("K_interaction").reset_index(drop=True)
    eligible_df = df_k_summary[df_k_summary["eligible"] == 1].copy()
    if len(eligible_df) == 0:
        selected_k = 0
    else:
        best_mean = float(eligible_df["mean_abc"].max())
        selected_k = int(eligible_df[np.abs(pd.to_numeric(eligible_df["mean_abc"], errors="coerce") - best_mean) < 0.002].sort_values("K_interaction").iloc[0]["K_interaction"])
    df_k_summary["selected"] = (df_k_summary["K_interaction"] == int(selected_k)).astype(int)
    for _, r in df_k_summary.iterrows():
        summary_prefix = "PhaseC-STGP-AND"
    print(f"[{summary_prefix}] gate rule: mean_delta > {PHASEC_MEAN_DELTA_MIN} and positive_folds >= {PHASEC_POSITIVE_FOLDS_MIN}")
    for _, r in df_k_summary.iterrows():
        print(f"[{summary_prefix}] K={int(r['K_interaction'])} mean={float(r['mean_abc']):.4f} delta={float(r['mean_delta']):.4f} positive_folds={int(r['positive_folds'])}")
    print(f"[{summary_prefix}] selected_k_cat = {selected_k}")

    oof_abc_risk = np.asarray(oof_abc_risk_by_k.get(int(selected_k), oof_ab_risk), dtype=float)
    if int(selected_k) == 0:
        oof_abc_risk = np.asarray(oof_ab_risk, dtype=float).copy()

    def _safe_cindex_local(T, E, risk):
        T = np.asarray(T, dtype=float); E = np.asarray(E, dtype=int); risk = np.asarray(risk, dtype=float)
        mask = np.isfinite(T) & np.isfinite(E) & np.isfinite(risk)
        if int(mask.sum()) < 3:
            return np.nan
        try:
            return float(concordance_index(T[mask], -risk[mask], E[mask]))
        except Exception:
            return np.nan

    T_all = df_train_raw[TIME_COL].astype(float).values
    E_all = df_train_raw[EVENT_COL].astype(int).values
    oof_main_only_cindex = _safe_cindex_local(T_all, E_all, oof_main_only_risk)
    oof_main_gp_cindex = _safe_cindex_local(T_all, E_all, oof_main_gp_risk)
    oof_ab_cindex = _safe_cindex_local(T_all, E_all, oof_ab_risk)
    oof_abc_cindex = _safe_cindex_local(T_all, E_all, oof_abc_risk)
    if not np.isfinite(oof_ab_cindex):
        row0_mean = df_k_summary[df_k_summary["K_interaction"] == 0]
        oof_ab_cindex = float(row0_mean["mean_abc"].iloc[0]) if len(row0_mean) else np.nan
    if not np.isfinite(oof_abc_cindex):
        row_sel_mean = df_k_summary[df_k_summary["K_interaction"] == int(selected_k)]
        oof_abc_cindex = float(row_sel_mean["mean_abc"].iloc[0]) if len(row_sel_mean) else np.nan
    oof_delta = oof_abc_cindex - oof_ab_cindex if np.isfinite(oof_ab_cindex) and np.isfinite(oof_abc_cindex) else np.nan

    df_sel = df_phasec_search[df_phasec_search["K_interaction"] == int(selected_k)].copy()
    ab_fold_cidx, abc_fold_cidx, fold_summary_rows = [], [], []
    for fold in range(1, int(n_splits) + 1):
        row0 = df_phasec_search[(df_phasec_search["K_interaction"] == 0) & (df_phasec_search["fold"] == fold)]
        rowk = df_sel[df_sel["fold"] == fold]
        ab_final = float(pd.to_numeric(row0["ab_cindex"], errors="coerce").iloc[0]) if len(row0) else np.nan
        abc_final = float(pd.to_numeric(rowk["abc_cindex"], errors="coerce").iloc[0]) if len(rowk) else ab_final
        dlt = abc_final - ab_final if np.isfinite(abc_final) and np.isfinite(ab_final) else np.nan
        ab_fold_cidx.append(ab_final); abc_fold_cidx.append(abc_final)
        fcm = [x for x in fold_cache if int(x["fold"]) == int(fold)]
        model_fold = fcm[0]["model_ab"] if fcm and fcm[0].get("model_ab") is not None else {}
        fold_summary_rows.append({
            "fold": int(fold), "main_only_cindex": model_fold.get("main_only_cindex", np.nan),
            "main_gp_cindex": model_fold.get("main_gp_cindex", np.nan), "main_gp_phaseB_cindex": model_fold.get("main_gp_phaseB_cindex", ab_final),
            "ab_final_cindex": ab_final, "abc_final_cindex": abc_final, "delta": dlt, "selected_k": int(selected_k),
            "n_gp": model_fold.get("actual_n_gp", np.nan), "time_interactions": int(len(model_fold.get("kept_phaseB_interactions", []))) if model_fold else np.nan,
            "actual_selected_trees": int(rowk["actual_selected_trees_n"].iloc[0]) if len(rowk) else 0,
            "main_effect_vars": json.dumps(model_fold.get("main_effect_vars", []), ensure_ascii=False),
            "gp_input_vars": json.dumps(model_fold.get("gp_input_vars", []), ensure_ascii=False),
            "kept_phaseB_interactions": json.dumps(model_fold.get("kept_phaseB_interactions", []), ensure_ascii=False),
            "ph_violating_main_effects": json.dumps(model_fold.get("ph_violating_main_effects", []), ensure_ascii=False),
            "ph_violating_gp_names": json.dumps(model_fold.get("ph_violating_gp_names", []), ensure_ascii=False),
        })
    fold_summary_df_tmp = pd.DataFrame(fold_summary_rows)
    main_only_fold_cidx = [float(x) if np.isfinite(x) else np.nan for x in pd.to_numeric(fold_summary_df_tmp.get("main_only_cindex", pd.Series(dtype=float)), errors="coerce").values]
    main_gp_fold_cidx = [float(x) if np.isfinite(x) else np.nan for x in pd.to_numeric(fold_summary_df_tmp.get("main_gp_cindex", pd.Series(dtype=float)), errors="coerce").values]
    main_gp_phaseB_fold_cidx = [float(x) if np.isfinite(x) else np.nan for x in pd.to_numeric(fold_summary_df_tmp.get("main_gp_phaseB_cindex", pd.Series(dtype=float)), errors="coerce").values]
    main_only_mean_cindex = float(np.nanmean(main_only_fold_cidx)) if np.isfinite(np.asarray(main_only_fold_cidx, dtype=float)).any() else np.nan
    main_gp_mean_cindex = float(np.nanmean(main_gp_fold_cidx)) if np.isfinite(np.asarray(main_gp_fold_cidx, dtype=float)).any() else np.nan
    main_gp_phaseB_mean_cindex = float(np.nanmean(main_gp_phaseB_fold_cidx)) if np.isfinite(np.asarray(main_gp_phaseB_fold_cidx, dtype=float)).any() else np.nan
    ab_mean_cindex = float(np.nanmean(ab_fold_cidx)) if np.isfinite(np.asarray(ab_fold_cidx, dtype=float)).any() else np.nan
    abc_mean_cindex = float(np.nanmean(abc_fold_cidx)) if np.isfinite(np.asarray(abc_fold_cidx, dtype=float)).any() else np.nan
    mean_delta = abc_mean_cindex - ab_mean_cindex if np.isfinite(ab_mean_cindex) and np.isfinite(abc_mean_cindex) else np.nan

    phaseb_timebasis_fold_summary = pd.concat([(fc.get("model_ab") or {}).get("phaseb_timebasis", {}).get("fold_summary", pd.DataFrame()) for fc in fold_cache], axis=0, ignore_index=True) if len(fold_cache) else pd.DataFrame()
    phaseb_timebasis_candidates_raw = pd.DataFrame([r for fc in fold_cache for r in ((fc.get("model_ab") or {}).get("phaseb_timebasis", {}).get("raw_candidates", []) if fc.get("model_ab") is not None else [])])
    phaseb_timebasis_candidate_scores = pd.concat([(fc.get("model_ab") or {}).get("phaseb_timebasis", {}).get("candidate_scores", pd.DataFrame()) for fc in fold_cache], axis=0, ignore_index=True) if len(fold_cache) else pd.DataFrame()
    phaseb_timebasis_selected_by_fold = pd.concat([(fc.get("model_ab") or {}).get("phaseb_timebasis", {}).get("selected_candidates_df", pd.DataFrame()) for fc in fold_cache], axis=0, ignore_index=True) if len(fold_cache) else pd.DataFrame()
    if str(globals().get("PHASEB_MODE", "time_basis_pool")).lower() == "time_basis_pool":
        tb_delta = pd.to_numeric(phaseb_timebasis_fold_summary.get("delta", pd.Series(dtype=float)), errors="coerce").values
        phaseB_oof_delta = float(np.nanmean(tb_delta)) if np.isfinite(tb_delta).any() else np.nan
        phaseB_positive_folds = int(np.sum(tb_delta > 0)) if len(tb_delta) else 0
        phaseB_allowed_final = (not bool(FINAL_PHASEB_GATE)) or (np.isfinite(phaseB_oof_delta) and phaseB_oof_delta > float(PHASEB_MEAN_DELTA_MIN) and phaseB_positive_folds >= int(PHASEB_POSITIVE_FOLDS_MIN))
    else:
        phaseB_oof_delta = oof_ab_cindex - oof_main_gp_cindex if np.isfinite(oof_ab_cindex) and np.isfinite(oof_main_gp_cindex) else np.nan
        phaseB_positive_folds = int(np.sum(np.asarray(main_gp_phaseB_fold_cidx, dtype=float) > np.asarray(main_gp_fold_cidx, dtype=float))) if len(main_gp_fold_cidx) else 0
        phaseB_allowed_final = (not bool(FINAL_PHASEB_GATE)) or (np.isfinite(phaseB_oof_delta) and phaseB_oof_delta > float(PHASEB_MEAN_DELTA_MIN) and phaseB_positive_folds >= int(PHASEB_POSITIVE_FOLDS_MIN))
    print(f"[PhaseB-TimeBasis] gate rule: mean_delta > {PHASEB_MEAN_DELTA_MIN} and positive_folds >= {PHASEB_POSITIVE_FOLDS_MIN}")
    print(f"[PhaseB-TimeBasis] mean_delta = {phaseB_oof_delta:.4f}")
    print(f"[PhaseB-TimeBasis] positive_folds = {phaseB_positive_folds}")
    print(f"[PhaseB-TimeBasis] allowed = {bool(phaseB_allowed_final)}")

    return {
        "selected_k": int(selected_k), "selected_c_params": get_fixed_phasec_params(selected_k), "k_summary": df_k_summary,
        "fold_records": df_phasec_search, "fold_ab_gen_trajectory": pd.DataFrame(fold_ab_gen_rows), "fold_summary": pd.DataFrame(fold_summary_rows),
        "phasea_fold_traj": pd.DataFrame(phasea_fold_traj_rows), "oof_main_only_risk": np.asarray(oof_main_only_risk, dtype=float),
        "oof_main_gp_risk": np.asarray(oof_main_gp_risk, dtype=float), "oof_ab_risk": np.asarray(oof_ab_risk, dtype=float), "oof_abc_risk": np.asarray(oof_abc_risk, dtype=float),
        "main_only_fold_cindexes": main_only_fold_cidx, "baseline_stable_main_only_mean_cindex": main_only_mean_cindex,
        "main_gp_fold_cindexes": main_gp_fold_cidx, "baseline_stable_main_gp_mean_cindex": main_gp_mean_cindex,
        "main_gp_phaseB_fold_cindexes": main_gp_phaseB_fold_cidx, "baseline_stable_main_gp_phaseB_mean_cindex": main_gp_phaseB_mean_cindex,
        "phaseB_oof_delta": phaseB_oof_delta, "phaseB_positive_folds": phaseB_positive_folds, "phaseB_allowed_final": bool(phaseB_allowed_final),
        "kept_phaseB_interactions_by_fold": [json.loads(x) if isinstance(x, str) else [] for x in fold_summary_df_tmp.get("kept_phaseB_interactions", pd.Series(dtype=object)).tolist()],
        "phaseb_timebasis_fold_summary": phaseb_timebasis_fold_summary, "phaseb_timebasis_candidates_raw": phaseb_timebasis_candidates_raw,
        "phaseb_timebasis_candidate_scores": phaseb_timebasis_candidate_scores, "phaseb_timebasis_selected_by_fold": phaseb_timebasis_selected_by_fold,
        "oof_main_only_cindex": oof_main_only_cindex, "oof_main_gp_cindex": oof_main_gp_cindex, "oof_ab_cindex": oof_ab_cindex, "oof_abc_cindex": oof_abc_cindex,
        "oof_delta": oof_delta, "ab_fold_cindexes": [float(x) if np.isfinite(x) else np.nan for x in ab_fold_cidx],
        "ab_mean_cindex": ab_mean_cindex, "abc_fold_cindexes": [float(x) if np.isfinite(x) else np.nan for x in abc_fold_cidx],
        "abc_mean_cindex": abc_mean_cindex, "abc_vs_ab_mean_delta": mean_delta, "abc_vs_ab_oof_delta": oof_delta, "ab_params": dict(ab_params),
        "phasec_stgp_and_rule_universe_raw": pd.DataFrame(phasec_raw_all),
        "phasec_stgp_and_rule_universe_feasible": pd.DataFrame(phasec_feasible_all),
        "phasec_stgp_and_semantic_dropped": pd.DataFrame(phasec_dropped_all),
        "phasec_stgp_and_population_init": pd.concat(phasec_population_init_all, axis=0, ignore_index=True) if len(phasec_population_init_all) else pd.DataFrame(),
        "phasec_stgp_and_generation_log": pd.concat(phasec_generation_log_all, axis=0, ignore_index=True) if len(phasec_generation_log_all) else pd.DataFrame(),
        "phasec_stgp_and_hof_by_fold": pd.concat(phasec_hof_all, axis=0, ignore_index=True) if len(phasec_hof_all) else pd.DataFrame(),
        "phasec_stgp_and_k_summary": df_k_summary,
        "phasec_abc_unified_k_summary": df_k_summary,
        "phasec_stgp_and_selected_by_fold": pd.DataFrame(phasec_selected_all),
        "phasec_abc_unified_selected_by_fold": pd.DataFrame(phasec_selected_all),
    }

def train_final_model_on_full_train(df_train_raw, final_params):
    allow_phaseB = bool(final_params.get("phaseB_allowed_final", True))
    model_ab = _fit_ab_backbone_on_train(
        df_train_raw,
        final_params["ab_params"],
        seed_offset=909,
        inner_folds_for_varsel=FINAL_HPO_FOLDS,
        collect_trajectory=True,
        allow_phaseB=allow_phaseB,
    )
    if model_ab is None:
        return None
    tr_pack = {"df_enc": model_ab["train_enc"], "T": model_ab["T_train"], "E": model_ab["E_train"], "F": model_ab["F_train"], "M": model_ab.get("M_train")}
    base_risk_tr = predict_ab_risk_on_test(tr_pack["F"], tr_pack["T"], tr_pack["E"], tr_pack["F"], tr_pack["T"], tr_pack["E"], model_ab["g_fixed"], model_ab["toolbox_t"], M_dev=tr_pack.get("M"), M_te=tr_pack.get("M"), main_effect_names=model_ab.get("main_effect_vars", []))
    if str(globals().get("PHASEC_MODE", "stgp_and")).lower() in {"stgp_and", "off"}:
        selected_k = int(final_params.get("c_params", {}).get("K_interaction", 0))
        raw, filtered, dropped, score_df, phasec_context = phasec_stgp_and_prepare_fold(
            tr_pack["df_enc"], model_ab.get("prep", {}), tr_pack["T"], tr_pack["E"], base_risk_tr, fold_label="final_full_train"
        ) if str(globals().get("PHASEC_MODE", "stgp_and")).lower() == "stgp_and" else ([], [], [], pd.DataFrame(), {})
        mode_c = str(globals().get("PHASEC_MODE", "stgp_and")).lower()
        evo = {}
        if mode_c == "stgp_and":
            evo = phasec_stgp_and_run_evolution(raw, filtered, tr_pack["df_enc"], tr_pack["T"], tr_pack["E"], base_risk_tr, prep=model_ab.get("prep", {}), fold_label="final_full_train", phasec_context=phasec_context)
            score_df = evo.get("candidate_scores", score_df)
            selected = evo.get("hof", [])[:selected_k]
        else:
            selected = []
        fit_train = phasec_stgp_and_fit_selected(selected, tr_pack["df_enc"], tr_pack["df_enc"], tr_pack["T"], tr_pack["E"], base_risk_tr, base_risk_tr)
        cidx_ab_tr, cidx_abc_tr, _ = score_ab_plus_c_on_validation(tr_pack["T"], tr_pack["E"], base_risk_tr, fit_train)
        if selected_k <= 0 or len(selected) == 0:
            print("[Final-PhaseC-STGP-AND] selected_k=0, PhaseC disabled")
        else:
            final_prefix = "Final-PhaseC-STGP-AND"
            print(f"[{final_prefix}] raw_candidates={len(raw)}")
            print(f"[{final_prefix}] filtered_candidates={len(filtered)}")
            print(f"[{final_prefix}] selected_candidates={[c.get('name', '') for c in selected]}")
        final_traj = {
            "phaseA": list(model_ab.get("phaseA_history", [])),
            "phaseB": list(model_ab.get("phaseB_history", [])),
            "phaseC": float(cidx_abc_tr) if np.isfinite(cidx_abc_tr) else np.nan,
            "phaseAB_train_cindex": float(cidx_ab_tr) if np.isfinite(cidx_ab_tr) else np.nan,
        }
        return {
            "ab_model": model_ab,
            "phasec_selected_trees": [],
            "phasec_terminal_names": [],
            "phasec_factor_map": {},
            "phasec_fit_train": fit_train,
            "phasec_params": final_params["c_params"],
            "final_trajectory": final_traj,
            "phasec_stgp_and_selected_candidates": selected,
            "phasec_stgp_and_rule_universe_raw": pd.DataFrame(raw),
            "phasec_stgp_and_rule_universe_feasible": pd.DataFrame(filtered),
            "phasec_stgp_and_semantic_dropped": pd.DataFrame(dropped),
            "phasec_stgp_and_population_init": evo.get("population_init", pd.DataFrame()) if mode_c == "stgp_and" else pd.DataFrame(),
            "phasec_stgp_and_generation_log": evo.get("generation_log", pd.DataFrame()) if mode_c == "stgp_and" else pd.DataFrame(),
            "phasec_stgp_and_hof_by_fold": evo.get("hof_df", pd.DataFrame()) if mode_c == "stgp_and" else pd.DataFrame(),
            "phasec_stgp_and_final_selected": pd.DataFrame(selected),
        }
    C_tr_df, fmap = build_phasec_candidate_terminals(tr_pack["df_enc"], model_ab["prep"])
    old = _temporary_set_params(final_params["c_params"])
    try:
        trees, factor_map, terminal_names = search_phasec_trees(C_tr_df, base_risk_tr, tr_pack["T"], tr_pack["E"], final_params["c_params"], factor_map=fmap)
        toolbox_c, _ = build_gp_toolbox_c(C_tr_df.shape[1])
        fit_train = fit_phasec_model_with_ab_baseline(
            trees, toolbox_c, C_tr_df, C_tr_df,
            base_risk_tr, base_risk_tr,
            tr_pack["T"], tr_pack["E"], terminal_names, factor_map, ab_main_effects=[]
        )
    finally:
        _restore_params(old)

    cidx_ab_tr, cidx_abc_tr, _ = score_ab_plus_c_on_validation(tr_pack["T"], tr_pack["E"], base_risk_tr, fit_train)
    final_traj = {
        "phaseA": list(model_ab.get("phaseA_history", [])),
        "phaseB": list(model_ab.get("phaseB_history", [])),
        "phaseC": float(cidx_abc_tr) if np.isfinite(cidx_abc_tr) else np.nan,
        "phaseAB_train_cindex": float(cidx_ab_tr) if np.isfinite(cidx_ab_tr) else np.nan,
    }

    return {
        "ab_model": model_ab,
        "phasec_selected_trees": trees,
        "phasec_terminal_names": terminal_names,
        "phasec_factor_map": fmap,
        "phasec_fit_train": fit_train,
        "phasec_params": final_params["c_params"],
        "final_trajectory": final_traj,
    }


def run_full_train_external_validation(df_train_raw, df_valid_raw, final_params=None):
    if final_params is None:
        final_params = {
            "ab_params": get_fixed_ab_params(),
            "c_params": get_fixed_phasec_params(0),
            "phasec_selection": {
                "selected_k": 0,
                "selection_rule": "K>0 allowed only if mean_delta > 0 and positive_folds >= 2; Phase C uses A_static base risk",
            },
            "phaseB_allowed_final": True,
        }
    final_model = train_final_model_on_full_train(df_train_raw, final_params)
    if final_model is None:
        return {
            "external_cindex": np.nan,
            "external_final_ab_cindex": np.nan,
            "external_risk": np.array([]),
            "phasec": {},
            "selected_cont_vars": [],
            "final_params": final_params,
            "final_train_trajectory": {},
            "actual_n_gp": np.nan,
            "actual_tree_heights": [],
            "actual_tree_lengths": [],
            "gp_filter_diag": {},
            "ab_cox_coefficients": {},
            "cont_standardize_stats": {},
            "continuous_preprocessing": "MICE + training-only 1%/99% winsorized z-score standardization",
            "gp_variable_scale": "standardized continuous variables",
        }
    model_ab = final_model["ab_model"]
    va_pack = _ab_prepare_apply(model_ab, df_valid_raw)

    stable_main_only = score_main_only_on_test(model_ab.get("M_train"), model_ab.get("main_effect_vars", []), model_ab["T_train"], model_ab["E_train"], va_pack.get("M"), va_pack["T"], va_pack["E"])
    stable_main_gp = _score_ab_with_gdict_on_valid(model_ab["F_train"], model_ab["T_train"], model_ab["E_train"], va_pack.get("F"), va_pack["T"], va_pack["E"], {}, model_ab["toolbox_t"], M_tr=model_ab.get("M_train"), M_va=va_pack.get("M"), main_effect_names=model_ab.get("main_effect_vars", []))
    cidx_ab, _, risk_ab_va = _score_ab_on_validation(model_ab, df_valid_raw)
    phaseB_allowed_final = bool(final_params.get("phaseB_allowed_final", True))
    print(f"[Final-PhaseB-TimeBasis] allowed = {bool(phaseB_allowed_final)}")
    final_feature_set_mode = str(model_ab.get("final_feature_set_mode", model_ab.get("feature_set_mode", "baseline_stable_main_gp")))
    phaseb_timebasis_final_selected = pd.DataFrame()
    phaseb_timebasis_external_ab = np.nan
    phaseb_timebasis_external_cindex = np.nan
    if str(globals().get("PHASEB_MODE", "time_basis_pool")).lower() == "time_basis_pool" and phaseB_allowed_final:
        tb = model_ab.get("phaseb_timebasis", {}) or {}
        selected_tb = tb.get("selected_candidates", []) or []
        phaseb_timebasis_final_selected = pd.DataFrame(selected_tb)
        if len(selected_tb):
            eta_va_tb = phaseb_static_eta_from_ab(model_ab["F_train"], model_ab["T_train"], model_ab["E_train"], va_pack.get("F"), va_pack["T"], va_pack["E"], model_ab["toolbox_t"], M_tr=model_ab.get("M_train"), M_apply=va_pack.get("M"), main_effect_names=model_ab.get("main_effect_vars", []))
            _, feature_va_tb, _, _ = phaseb_feature_frames(model_ab.get("M_train"), va_pack.get("M"), model_ab.get("main_effect_vars", []), model_ab.get("F_train"), va_pack.get("F"), n_tr=len(model_ab["T_train"]), n_va=len(va_pack["T"]))
            phaseb_timebasis_external_ab = time_dependent_cindex_phaseb(va_pack["T"], va_pack["E"], eta_va_tb, feature_va_tb, [], {"eta_ab": 1.0}, tb.get("time_stats", {}))
            phaseb_timebasis_external_cindex = time_dependent_cindex_phaseb(va_pack["T"], va_pack["E"], eta_va_tb, feature_va_tb, selected_tb, tb.get("coefficients", {"eta_ab": 1.0}), tb.get("time_stats", {}))
            print(f"[External] A_static C-index = {phaseb_timebasis_external_ab:.4f} (PhaseB time-dependent approximation baseline)")
            print(f"[External] AB_timevarying C-index = {phaseb_timebasis_external_cindex:.4f} (time-dependent approximation)")
            print(f"[Final-PhaseB-TimeBasis] selected_candidates = {[c.get('name', '') for c in selected_tb]}")
    if not phaseB_allowed_final:
        stable_main_gp_phaseB = stable_main_gp
        phaseB_status = "disabled_by_internal_oof_gate"
        final_ab_model_type = final_feature_set_mode
        external_final_ab = stable_main_gp
        if str(globals().get("PHASEB_MODE", "time_basis_pool")).lower() == "time_basis_pool":
            print("[Final-PhaseB-TimeBasis] disabled by internal gate")
    elif str(globals().get("PHASEB_MODE", "time_basis_pool")).lower() == "time_basis_pool" and np.isfinite(phaseb_timebasis_external_cindex):
        stable_main_gp_phaseB = phaseb_timebasis_external_cindex
        phaseB_status = "enabled_time_basis_pool_by_internal_gate"
        final_ab_model_type = f"{final_feature_set_mode}_phaseB_timebasis"
        external_final_ab = phaseb_timebasis_external_cindex
    else:
        stable_main_gp_phaseB = cidx_ab
        phaseB_status = "enabled_by_internal_oof_gate"
        final_ab_model_type = f"{final_feature_set_mode}_phaseB"
        external_final_ab = cidx_ab
    external_baseline_stable_main_only = stable_main_only if final_feature_set_mode == "baseline_stable_main_gp" else np.nan
    external_baseline_stable_main_gp = stable_main_gp if final_feature_set_mode == "baseline_stable_main_gp" else np.nan
    external_baseline_stable_main_gp_phaseB = stable_main_gp_phaseB if final_feature_set_mode == "baseline_stable_main_gp" else np.nan
    external_top10_gp_only = stable_main_gp if final_feature_set_mode == "top10_gp_only" else np.nan
    external_top10_gp_only_phaseB = stable_main_gp_phaseB if final_feature_set_mode == "top10_gp_only" else np.nan

    tr_pack = {"df_enc": model_ab["train_enc"], "T": model_ab["T_train"], "E": model_ab["E_train"], "F": model_ab["F_train"], "M": model_ab.get("M_train")}
    base_risk_tr = predict_ab_risk_on_test(tr_pack["F"], tr_pack["T"], tr_pack["E"], tr_pack["F"], tr_pack["T"], tr_pack["E"], model_ab["g_fixed"], model_ab["toolbox_t"], M_dev=tr_pack.get("M"), M_te=tr_pack.get("M"), main_effect_names=model_ab.get("main_effect_vars", []))
    if str(globals().get("PHASEC_MODE", "stgp_and")).lower() in {"stgp_and", "off"}:
        selected = final_model.get("phasec_stgp_and_selected_candidates", []) or []
        fit_res = phasec_stgp_and_fit_selected(
            selected,
            tr_pack["df_enc"],
            va_pack["df_enc"],
            tr_pack["T"],
            tr_pack["E"],
            base_risk_tr,
            risk_ab_va,
        )
        _, c_abc, _ = score_ab_plus_c_on_validation(va_pack["T"], va_pack["E"], risk_ab_va, fit_res)
        print(f"[External] A_static_for_C cindex={float(concordance_index(va_pack['T'], -np.asarray(risk_ab_va, dtype=float), va_pack['E'])):.4f}")
        print(f"[External] AC_static cindex={c_abc:.4f}")
    else:
        C_tr_df, fmap = build_phasec_candidate_terminals(tr_pack["df_enc"], model_ab["prep"])
        C_va_df, _ = build_phasec_candidate_terminals(va_pack["df_enc"], model_ab["prep"])
        for col in C_tr_df.columns:
            if col not in C_va_df.columns:
                C_va_df[col] = 0.0
        C_va_df = C_va_df[C_tr_df.columns]
        old = _temporary_set_params(final_model["phasec_params"])
        try:
            toolbox_c, _ = build_gp_toolbox_c(C_tr_df.shape[1])
            fit_res = fit_phasec_model_with_ab_baseline(
                final_model["phasec_selected_trees"], toolbox_c,
                C_tr_df, C_va_df,
                base_risk_tr, risk_ab_va,
                tr_pack["T"], tr_pack["E"],
                final_model["phasec_terminal_names"], fmap, ab_main_effects=[]
            )
        finally:
            _restore_params(old)
        _, c_abc, _ = score_ab_plus_c_on_validation(va_pack["T"], va_pack["E"], risk_ab_va, fit_res)
    ac_static_cindex = c_abc
    selected_k_final = int(final_params.get("c_params", {}).get("K_interaction", 0))
    A_static_cindex = stable_main_gp
    AB_timevarying_cindex = external_final_ab
    AC_static_cindex = ac_static_cindex
    selected_phasec_final = final_model.get("phasec_stgp_and_selected_candidates", []) or []
    selected_phaseb_final = list((model_ab.get("phaseb_timebasis", {}) or {}).get("selected_candidates", []) or [])
    phaseB_active_final = bool(phaseB_allowed_final and len(selected_phaseb_final) > 0)
    phaseC_active_final = bool(selected_k_final > 0 and len(selected_phasec_final) > 0)
    unified_external = {
        "coefficients": {},
        "reason": "not_called",
        "design_cols": [],
        "selected_phaseB_candidates": selected_phaseb_final,
        "selected_phaseC_candidates": selected_phasec_final,
    }
    if phaseB_active_final and phaseC_active_final:
        unified_external = fit_eval_unified_abc_from_model(
            model_ab, va_pack,
            selected_phaseC_candidates=selected_phasec_final,
            phaseB_allowed=True,
            fold_label="external",
        )
        ABC_timevarying_cindex = float(unified_external.get("cindex", np.nan))
        unified_abc_fallback_mode = "unified_ABC"
    elif phaseB_active_final and not phaseC_active_final:
        print("[Unified-ABC] PhaseC disabled; ABC_timevarying equals AB_timevarying.")
        ABC_timevarying_cindex = AB_timevarying_cindex
        unified_abc_fallback_mode = "equals_AB"
    elif (not phaseB_active_final) and phaseC_active_final:
        print("[Unified-ABC] PhaseB disabled; ABC_timevarying equals AC_static.")
        ABC_timevarying_cindex = AC_static_cindex
        unified_abc_fallback_mode = "equals_AC"
        if abs(float(ABC_timevarying_cindex) - float(AC_static_cindex)) > 1e-8:
            print("[ERROR] PhaseB disabled but ABC_timevarying != AC_static. This indicates evaluator fallback bug.")
    else:
        print("[Unified-ABC] PhaseB and PhaseC disabled; ABC_timevarying equals A_static.")
        ABC_timevarying_cindex = A_static_cindex
        unified_abc_fallback_mode = "equals_A"
    ABC_final_cindex = ABC_timevarying_cindex
    print(f"[Unified-ABC][External] phaseB_active = {phaseB_active_final}")
    print(f"[Unified-ABC][External] phaseC_active = {phaseC_active_final}")
    print(f"[Unified-ABC][External] fallback_mode = {unified_abc_fallback_mode}")
    c_abc = ABC_final_cindex
    print(f"[External] A_static C-index = {A_static_cindex:.4f}")
    print(f"[External] AB_timevarying C-index = {AB_timevarying_cindex:.4f}")
    print(f"[External] AC_static C-index = {AC_static_cindex:.4f}")
    print(f"[External] ABC_timevarying C-index = {ABC_timevarying_cindex:.4f}")
    print("[External] Final model = ABC_timevarying")
    return {
        "external_cindex": c_abc,
        "A_static_cindex": A_static_cindex,
        "AB_timevarying_cindex": AB_timevarying_cindex,
        "AC_static_cindex": AC_static_cindex,
        "AC_static_cindex_optional": AC_static_cindex,
        "ABC_timevarying_cindex": ABC_timevarying_cindex,
        "ABC_final_cindex": ABC_final_cindex,
        "abc_unified_external": unified_external,
        "selected_k_cat": selected_k_final,
        "phaseB_active_final": bool(phaseB_active_final),
        "phaseC_active_final": bool(phaseC_active_final),
        "unified_abc_fallback_mode": unified_abc_fallback_mode,
        "phaseB_selected_candidates": selected_phaseb_final,
        "phaseC_selected_candidates": selected_phasec_final,
        "final_model": "ABC_timevarying",
        "external_ab_cindex": external_final_ab,
        "external_final_ab_cindex": external_final_ab,
        "external_baseline_stable_main_only_cindex": external_baseline_stable_main_only,
        "external_baseline_stable_main_gp_cindex": external_baseline_stable_main_gp,
        "external_baseline_stable_main_gp_phaseB_cindex": external_baseline_stable_main_gp_phaseB,
        "external_top10_gp_only_cindex": external_top10_gp_only,
        "external_top10_gp_only_phaseB_cindex": external_top10_gp_only_phaseB,
        "phaseB_final_status": phaseB_status,
        "final_ab_model_type": final_ab_model_type,
        "external_risk": np.asarray(fit_res.get("risk_valid", []), dtype=float),
        "phasec": fit_res,
        "selected_cont_vars": model_ab.get("final_vars", []),
        "feature_set_mode": model_ab.get("feature_set_mode", ""),
        "final_feature_set_mode": model_ab.get("final_feature_set_mode", model_ab.get("feature_set_mode", "")),
        "feature_set_decision": model_ab.get("feature_set_decision", ""),
        "use_main_effects_for_this_run": model_ab.get("use_main_effects_for_this_run", False),
        "gp_input_vars_source": model_ab.get("gp_input_vars_source", ""),
        "stable_selection_mode": model_ab.get("stable_selection_mode", np.nan),
        "stable_vars": model_ab.get("stable_vars", []),
        "stable_var_frequency": model_ab.get("stable_var_frequency", []),
        "stable_min_freq_used": model_ab.get("stable_min_freq_used", np.nan),
        "stable_min_vars": model_ab.get("stable_min_vars", np.nan),
        "stable_fallback_freq": model_ab.get("stable_fallback_freq", np.nan),
        "main_effect_vars": model_ab.get("main_effect_vars", []),
        "gp_input_vars": model_ab.get("gp_input_vars", []),
        "main_effect_indices": model_ab.get("main_effect_indices", []),
        "enable_main_effects": model_ab.get("enable_main_effects", np.nan),
        "main_effect_k": model_ab.get("main_effect_k", np.nan),
        "kept_phaseB_interactions": model_ab.get("kept_phaseB_interactions", []),
        "ph_violating_main_effects": model_ab.get("ph_violating_main_effects", []),
        "ph_violating_gp_names": model_ab.get("ph_violating_gp_names", []),
        "phaseb_timebasis_final_selected": phaseb_timebasis_final_selected,
        "phasec_stgp_and_final_selected": final_model.get("phasec_stgp_and_final_selected", pd.DataFrame()),
        "phaseb_timebasis_external_ab_cindex": phaseb_timebasis_external_ab,
        "phaseb_timebasis_external_cindex": phaseb_timebasis_external_cindex,
        "final_params": final_params,
        "final_train_trajectory": final_model.get("final_trajectory", {}),
        "gp_exprs": model_ab.get("gp_exprs", []),
        "actual_n_gp": model_ab.get("actual_n_gp", np.nan),
        "actual_tree_heights": model_ab.get("actual_tree_heights", []),
        "actual_tree_lengths": model_ab.get("actual_tree_lengths", []),
        "gp_filter_diag": model_ab.get("gp_filter_diag", {}),
        "ab_cox_coefficients": fit_ab_cox_coefficients_for_reporting(model_ab),
        "cont_standardize_stats": model_ab.get("prep", {}).get("cont_standardize_stats", {}),
        "continuous_preprocessing": "MICE + training-only 1%/99% winsorized z-score standardization",
        "gp_variable_scale": "standardized continuous variables",
    }


def run_once(overrides=None):
    _apply_overrides(overrides)
    set_all_seeds(SEED)
    print("=" * 80)
    print("Load dataset")
    print("=" * 80)
    df_train_raw_local = globals().get("df_train_raw")
    df_valid_raw_local = globals().get("df_valid_raw")
    if df_train_raw_local is None or df_valid_raw_local is None:
        df_train_raw_local = _coerce_time_event(pd.read_excel(TRAIN_DATA_PATH))
        df_valid_raw_local = _coerce_time_event(pd.read_excel(EXTERNAL_VALID_DATA_PATH))
    df_train_raw = df_train_raw_local.copy()
    df_valid_raw = df_valid_raw_local.copy()
    train_n = int(len(df_train_raw))
    train_events = int(df_train_raw[EVENT_COL].sum())
    external_n = int(len(df_valid_raw))
    external_events = int(df_valid_raw[EVENT_COL].sum())
    print(f"Train n={train_n} events={train_events}")
    print(f"External valid n={external_n} events={external_events}")

    reference_top10 = reproduce_cell2_topk_main_only(df_train_raw, df_valid_raw, top_k=TOP_K)
    print(f"[ReferenceBaseline] top10 variables = {reference_top10.get('selected_topk_vars', [])}")
    print(f"[ReferenceBaseline] top10 train C-index = {reference_top10.get('train_cindex', np.nan):.4f}")
    print(f"[ReferenceBaseline] top10 external C-index = {reference_top10.get('external_cindex', np.nan):.4f}")

    baseline_stable_vars, baseline_stable_freq_df, baseline_stable_min_freq_used = extract_baseline_stable_vars_from_cell2_results(df_train_raw, top_k=TOP_K)
    globals()["BASELINE_STABLE_VARS"] = list(baseline_stable_vars)
    globals()["BASELINE_STABLE_VAR_FREQUENCY"] = baseline_stable_freq_df.to_dict(orient="records")
    globals()["BASELINE_STABLE_MIN_FREQ_USED"] = baseline_stable_min_freq_used
    globals()["REFERENCE_TOP10_VARS"] = list(reference_top10.get("selected_topk_vars", []))[:int(TOP_K)]
    feature_decision = decide_feature_set_from_baseline_stable(baseline_stable_vars, globals()["REFERENCE_TOP10_VARS"])
    globals()["FINAL_FEATURE_SET_MODE"] = feature_decision["final_feature_set_mode"]
    globals()["USE_MAIN_EFFECTS_FOR_THIS_RUN"] = feature_decision["use_main_effects_for_this_run"]
    globals()["MAIN_EFFECT_VARS"] = list(feature_decision["main_effect_vars"])
    globals()["GP_INPUT_VARS"] = list(feature_decision["gp_input_vars"])
    globals()["FEATURE_SET_DECISION"] = feature_decision["feature_set_decision"]

    ab_params = apply_gp_param_profile(get_fixed_ab_params(), feature_decision["final_feature_set_mode"])
    cv_res = run_internal_fixed_cv_select_phasec(
        df_train_raw,
        ab_params,
        k_grid=(0, 1, 2, 3),
        n_splits=4
    )
    plot_phasea_4fold_valid_cindex_trajectory(cv_res.get("phasea_fold_traj", pd.DataFrame()), OUT_PNG_PHASEA_4FOLD)

    final_ab_params = dict(ab_params)
    final_ab_params["N_GEN_X"] = int(FINAL_FULL_TRAIN_N_GEN_X)

    final_params = {
        "ab_params": final_ab_params,
        "c_params": cv_res["selected_c_params"],
        "phaseB_allowed_final": bool(cv_res.get("phaseB_allowed_final", True)),
        "phasec_selection": {
            "selected_k": cv_res["selected_k"],
            "selection_rule": "K>0 allowed only if mean_delta > 0 and positive_folds >= 2; Phase C K-search uses unified ABC_timevarying over AB_timevarying",
        },
    }
    external_res = run_full_train_external_validation(df_train_raw, df_valid_raw, final_params=final_params)
    plot_final_fulltrain_trajectory(external_res.get("final_train_trajectory", {}), OUT_PNG_TRAJ)

    selected_k = int(cv_res.get("selected_k", 0))
    external_abc = float(external_res.get("ABC_timevarying_cindex", external_res.get("ABC_final_cindex", external_res.get("external_cindex", np.nan))))
    external_final_ab = float(external_res.get("AB_timevarying_cindex", external_res.get("external_final_ab_cindex", np.nan)))
    external_delta = external_abc - external_final_ab if np.isfinite(external_abc) and np.isfinite(external_final_ab) else np.nan
    final_ab_model_type = str(external_res.get("final_ab_model_type", "baseline_stable_main_gp"))
    phaseB_allowed_final = bool(cv_res.get("phaseB_allowed_final", True))
    phaseB_oof_delta = float(cv_res.get("phaseB_oof_delta", np.nan))
    phaseB_positive_folds = int(cv_res.get("phaseB_positive_folds", 0))

    baseline_stable_main_only_oof = float(cv_res.get("oof_main_only_cindex", np.nan))
    baseline_stable_main_gp_oof = float(cv_res.get("oof_main_gp_cindex", np.nan))
    baseline_stable_main_gp_phaseB_oof = float(cv_res.get("oof_ab_cindex", np.nan))
    external_main_only = float(external_res.get("external_baseline_stable_main_only_cindex", np.nan))
    external_main_gp = float(external_res.get("external_baseline_stable_main_gp_cindex", np.nan))
    external_main_gp_phaseB = float(external_res.get("external_baseline_stable_main_gp_phaseB_cindex", np.nan))
    external_top10_gp_only = float(external_res.get("external_top10_gp_only_cindex", np.nan))
    external_top10_gp_only_phaseB = float(external_res.get("external_top10_gp_only_phaseB_cindex", np.nan))
    final_feature_set_mode = str(feature_decision.get("final_feature_set_mode", globals().get("FINAL_FEATURE_SET_MODE", "")))
    use_main_effects_for_this_run = bool(feature_decision.get("use_main_effects_for_this_run", False))
    main_effect_vars_for_summary = list(external_res.get("main_effect_vars", feature_decision.get("main_effect_vars", [])))
    gp_input_vars_for_summary = list(external_res.get("gp_input_vars", feature_decision.get("gp_input_vars", [])))
    actual_gp_params = dict(globals().get("ACTUAL_GP_PARAMS", {}))

    df_fold_summary = cv_res.get("fold_summary", pd.DataFrame()).copy()
    fold_results_cols = [
        "fold", "main_only_cindex", "main_gp_cindex", "main_gp_phaseB_cindex",
        "ab_final_cindex", "abc_final_cindex", "kept_phaseB_interactions",
        "ph_violating_main_effects", "ph_violating_gp_names",
        "main_effect_vars", "gp_input_vars", "actual_n_gp",
    ]
    fold_results = df_fold_summary[[c for c in fold_results_cols if c in df_fold_summary.columns]].copy()

    external_results = pd.DataFrame([
        {
            "model_name": "reference_top10_cox",
            "external_cindex": reference_top10.get("external_cindex", np.nan),
            "selected_variables": json.dumps(reference_top10.get("selected_topk_vars", []), ensure_ascii=False),
            "phaseB_enabled": False,
            "phaseC_selected_k": np.nan,
        },
        {
            "model_name": "baseline_stable_main_only",
            "external_cindex": external_main_only,
            "selected_variables": json.dumps(baseline_stable_vars, ensure_ascii=False),
            "phaseB_enabled": False,
            "phaseC_selected_k": np.nan,
        },
        {
            "model_name": "baseline_stable_main_gp",
            "external_cindex": external_main_gp,
            "selected_variables": json.dumps(baseline_stable_vars, ensure_ascii=False),
            "phaseB_enabled": False,
            "phaseC_selected_k": np.nan,
        },
        {
            "model_name": "baseline_stable_main_gp_phaseB",
            "external_cindex": external_main_gp_phaseB,
            "selected_variables": json.dumps(baseline_stable_vars, ensure_ascii=False),
            "phaseB_enabled": phaseB_allowed_final,
            "phaseC_selected_k": np.nan,
        },
        {
            "model_name": "top10_gp_only",
            "external_cindex": external_top10_gp_only,
            "selected_variables": json.dumps(gp_input_vars_for_summary, ensure_ascii=False),
            "phaseB_enabled": False,
            "phaseC_selected_k": np.nan,
        },
        {
            "model_name": "top10_gp_only_phaseB",
            "external_cindex": external_top10_gp_only_phaseB,
            "selected_variables": json.dumps(gp_input_vars_for_summary, ensure_ascii=False),
            "phaseB_enabled": phaseB_allowed_final,
            "phaseC_selected_k": np.nan,
        },
        {
            "model_name": "final_ab",
            "external_cindex": external_final_ab,
            "selected_variables": json.dumps(gp_input_vars_for_summary, ensure_ascii=False),
            "phaseB_enabled": phaseB_allowed_final,
            "phaseC_selected_k": np.nan,
        },
        {
            "model_name": "abc",
            "external_cindex": external_abc,
            "selected_variables": json.dumps(gp_input_vars_for_summary, ensure_ascii=False),
            "phaseB_enabled": phaseB_allowed_final,
            "phaseC_selected_k": selected_k,
        },
    ])
    external_results["A_static_cindex"] = external_res.get("A_static_cindex", np.nan)
    external_results["AB_timevarying_cindex"] = external_res.get("AB_timevarying_cindex", np.nan)
    external_results["AC_static_cindex"] = external_res.get("AC_static_cindex", np.nan)
    external_results["ABC_final_cindex"] = external_abc
    external_results["ABC_timevarying_cindex"] = external_res.get("ABC_timevarying_cindex", np.nan)

    ab_coefs = external_res.get("ab_cox_coefficients", {}) or {}
    gp_rows = []
    gp_exprs = external_res.get("gp_exprs", [])
    heights = external_res.get("actual_tree_heights", [])
    lengths = external_res.get("actual_tree_lengths", [])
    for i, expr in enumerate(gp_exprs, 1):
        gp_rows.append({
            "gp_index": int(i),
            "expression": expr,
            "coefficient": ab_coefs.get(f"gp{i}", np.nan),
            "tree_height": heights[i - 1] if i - 1 < len(heights) else np.nan,
            "tree_length": lengths[i - 1] if i - 1 < len(lengths) else np.nan,
        })
    final_gp_expressions = pd.DataFrame(gp_rows)

    phaseB_selected_terms = external_res.get("kept_phaseB_interactions", []) or []
    phaseB_candidate_table = cv_res.get("phaseb_timebasis_candidates_raw", pd.DataFrame())
    if phaseB_candidate_table is None:
        phaseB_candidate_table = pd.DataFrame()
    phaseB_candidate_n = int(len(phaseB_candidate_table)) if hasattr(phaseB_candidate_table, "__len__") else None
    phaseB_selected_n = int(len(phaseB_selected_terms)) if phaseB_selected_terms is not None else None
    phaseB_gate_reason = "passed" if bool(phaseB_allowed_final) else f"failed: mean_delta <= {PHASEB_MEAN_DELTA_MIN} or positive_folds < {PHASEB_POSITIVE_FOLDS_MIN}"

    phaseB_summary = pd.DataFrame([{
        "phaseB_mode": globals().get("PHASEB_MODE", "time_basis_pool"),
        "phaseB_selection_mode": PHASEB_SELECTION_MODE,
        "phaseB_mean_delta_min": PHASEB_MEAN_DELTA_MIN,
        "phaseB_positive_folds_min": PHASEB_POSITIVE_FOLDS_MIN,
        "phaseB_complexity_lambda": PHASEB_COMPLEXITY_LAMBDA,
        "timebasis_complexity_lambda": TIMEBASIS_COMPLEXITY_LAMBDA,
        "phaseB_candidate_n": phaseB_candidate_n,
        "phaseB_selected_n": phaseB_selected_n,
        "phaseB_allowed": bool(phaseB_allowed_final),
        "phaseB_gate_reason": phaseB_gate_reason,
        "phaseB_selected_terms": json.dumps(phaseB_selected_terms, ensure_ascii=False),
        "phaseB_allowed_final": phaseB_allowed_final,
        "main_gp_oof_cindex": baseline_stable_main_gp_oof,
        "main_gp_phaseB_oof_cindex": baseline_stable_main_gp_phaseB_oof,
        "phaseB_oof_delta": phaseB_oof_delta,
        "phaseb_delta_AB_minus_A": phaseB_oof_delta,
        "phaseB_positive_folds": phaseB_positive_folds,
        "A_internal_cindex": baseline_stable_main_gp_oof,
        "AB_internal_cindex": baseline_stable_main_gp_phaseB_oof,
        "delta_B": phaseB_oof_delta,
        "final_kept_phaseB_interactions": json.dumps(external_res.get("kept_phaseB_interactions", []), ensure_ascii=False),
        "phaseb_timebasis_external_ab_cindex": external_res.get("phaseb_timebasis_external_ab_cindex", np.nan),
        "phaseb_timebasis_external_cindex": external_res.get("phaseb_timebasis_external_cindex", np.nan),
    }])

    df_phasec_k_summary = cv_res.get("k_summary", pd.DataFrame()).copy()
    phasec_selected = df_phasec_k_summary[df_phasec_k_summary.get("selected", pd.Series(dtype=int)) == 1].copy() if len(df_phasec_k_summary) else pd.DataFrame()
    if len(phasec_selected):
        pr = phasec_selected.iloc[0].to_dict()
    else:
        pr = {}
    phasec_final_rule_rows = external_res.get("phasec_stgp_and_final_selected", pd.DataFrame())
    phaseC_candidate_rule_n = int(len(cv_res.get("phasec_stgp_and_rule_universe_feasible", pd.DataFrame()))) if hasattr(cv_res.get("phasec_stgp_and_rule_universe_feasible", pd.DataFrame()), "__len__") else None
    phaseC_selected_rule_n = int(len(phasec_final_rule_rows)) if hasattr(phasec_final_rule_rows, "__len__") else None
    phaseC_gate_reason = "passed" if int(selected_k) > 0 else f"failed: no K passed mean_delta > {PHASEC_MEAN_DELTA_MIN} and positive_folds >= {PHASEC_POSITIVE_FOLDS_MIN}"
    phaseC_summary = pd.DataFrame([{
        "selected_k": selected_k,
        "selected_k_cat": selected_k,
        "phasec_mode": globals().get("PHASEC_MODE", "stgp_and"),
        "phaseC_selection_mode": PHASEC_SELECTION_MODE,
        "phaseC_k_profile": PHASEC_K_PROFILE,
        "phaseC_k_values": json.dumps(resolve_phasec_k_values(PHASEC_K_PROFILE, PHASEC_K_VALUES), ensure_ascii=False),
        "phaseC_mean_delta_min": PHASEC_MEAN_DELTA_MIN,
        "phaseC_positive_folds_min": PHASEC_POSITIVE_FOLDS_MIN,
        "phaseC_complexity_lambda": PHASEC_SELECTION_COMPLEXITY_LAMBDA,
        "phaseC_algorithm_fitness_complexity_lambda": PHASEC_COMPLEXITY_LAMBDA,
        "phaseC_candidate_rule_n": phaseC_candidate_rule_n,
        "phaseC_selected_rule_n": phaseC_selected_rule_n,
        "phaseC_active_final": external_res.get("phaseC_active_final", np.nan),
        "phaseC_gate_reason": phaseC_gate_reason,
        "mean_ab": pr.get("mean_ab", np.nan),
        "mean_abc": pr.get("mean_abc", np.nan),
        "mean_delta": pr.get("mean_delta", np.nan),
        "positive_folds": pr.get("positive_folds", np.nan),
        "AC_internal_cindex": pr.get("mean_abc", np.nan),
        "ABC_internal_cindex": locals().get("oof_abc_cindex", pr.get("mean_abc", np.nan)),
        "delta_C_static": pr.get("mean_delta", np.nan),
        "delta_C_afterB": pr.get("mean_delta", np.nan),
        "external_abc_cindex": external_abc,
        "external_ABC_timevarying_cindex": external_res.get("ABC_timevarying_cindex", np.nan),
        "external_delta": external_delta,
        "initialization": "random_legal",
        "base_fit_once": True,
        "support_precomputed": True,
        "fitness_cache_enabled": True,
    }])

    summary = pd.DataFrame([{
        "train_n": train_n,
        "train_events": train_events,
        "external_n": external_n,
        "external_events": external_events,
        "TOP_K": int(TOP_K),
        "feature_set_decision": feature_decision.get("feature_set_decision", ""),
        "final_feature_set_mode": final_feature_set_mode,
        "baseline_stable_min_strict_n": int(BASELINE_STABLE_MIN_STRICT_N),
        "strict_stable_vars": json.dumps(baseline_stable_vars, ensure_ascii=False),
        "strict_stable_n": int(len(baseline_stable_vars)),
        "use_main_effects_for_this_run": use_main_effects_for_this_run,
        "actual_gp_param_profile": globals().get("ACTUAL_GP_PARAM_PROFILE"),
        "actual_N_TREES": actual_gp_params.get("N_TREES", N_TREES),
        "actual_MAX_TREE_HEIGHT_X": actual_gp_params.get("MAX_TREE_HEIGHT_X", MAX_TREE_HEIGHT_X),
        "actual_N_GEN_X": actual_gp_params.get("N_GEN_X", N_GEN_X),
        "actual_POP_SIZE_X": actual_gp_params.get("POP_SIZE_X", POP_SIZE_X),
        "actual_COMPLEXITY_LAMBDA_X": actual_gp_params.get("COMPLEXITY_LAMBDA_X", COMPLEXITY_LAMBDA_X),
        "actual_DUPLICATE_TERMINAL_LAMBDA_X": actual_gp_params.get("DUPLICATE_TERMINAL_LAMBDA_X", DUPLICATE_TERMINAL_LAMBDA_X),
        "reference_top10_vars": json.dumps(reference_top10.get("selected_topk_vars", []), ensure_ascii=False),
        "reference_top10_external_cindex": reference_top10.get("external_cindex", np.nan),
        "baseline_stable_var_frequency": json.dumps(baseline_stable_freq_df.to_dict(orient="records"), ensure_ascii=False),
        "main_effect_vars": json.dumps(main_effect_vars_for_summary, ensure_ascii=False),
        "gp_input_vars": json.dumps(gp_input_vars_for_summary, ensure_ascii=False),
        "final_ab_model_type": final_ab_model_type,
        "phaseB_allowed_final": phaseB_allowed_final,
        "selected_k_phaseC": selected_k,
        "selected_k_cat": selected_k,
        "baseline_stable_main_only_oof_cindex": baseline_stable_main_only_oof,
        "baseline_stable_main_gp_oof_cindex": baseline_stable_main_gp_oof,
        "baseline_stable_main_gp_phaseB_oof_cindex": baseline_stable_main_gp_phaseB_oof,
        "external_baseline_stable_main_only_cindex": external_main_only,
        "external_baseline_stable_main_gp_cindex": external_main_gp,
        "external_baseline_stable_main_gp_phaseB_cindex": external_main_gp_phaseB,
        "external_top10_gp_only_cindex": external_top10_gp_only,
        "external_top10_gp_only_phaseB_cindex": external_top10_gp_only_phaseB,
        "external_final_ab_cindex": external_final_ab,
        "A_static_cindex": external_res.get("A_static_cindex", np.nan),
        "AB_timevarying_cindex": external_res.get("AB_timevarying_cindex", np.nan),
        "AC_static_cindex": external_res.get("AC_static_cindex", np.nan),
        "ABC_final_cindex": external_abc,
        "ABC_timevarying_cindex": external_res.get("ABC_timevarying_cindex", np.nan),
        "AC_static_cindex_optional": external_res.get("AC_static_cindex_optional", external_res.get("AC_static_cindex", np.nan)),
        "external_abc_cindex": external_abc,
        "external_ABC_timevarying_cindex": external_res.get("ABC_timevarying_cindex", np.nan),
        "phaseB_active_final": external_res.get("phaseB_active_final", np.nan),
        "phaseC_active_final": external_res.get("phaseC_active_final", np.nan),
        "phaseB_selection_mode": PHASEB_SELECTION_MODE,
        "phaseB_mean_delta_min": PHASEB_MEAN_DELTA_MIN,
        "phaseB_positive_folds_min": PHASEB_POSITIVE_FOLDS_MIN,
        "phaseB_mean_delta": phaseB_oof_delta,
        "phaseB_positive_folds": phaseB_positive_folds,
        "phaseB_gate_reason": phaseB_gate_reason,
        "phaseC_selection_mode": PHASEC_SELECTION_MODE,
        "phaseC_k_profile": PHASEC_K_PROFILE,
        "phaseC_k_values": json.dumps(resolve_phasec_k_values(PHASEC_K_PROFILE, PHASEC_K_VALUES), ensure_ascii=False),
        "phaseC_mean_delta_min": PHASEC_MEAN_DELTA_MIN,
        "phaseC_positive_folds_min": PHASEC_POSITIVE_FOLDS_MIN,
        "phaseC_mean_delta": pr.get("mean_delta", np.nan),
        "phaseC_positive_folds": pr.get("positive_folds", np.nan),
        "phaseC_gate_reason": phaseC_gate_reason,
        "algorithm_logic_changed": False,
        "phaseB_logic_preserved": True,
        "phaseC_logic_preserved": True,
        "unified_abc_fallback_mode": external_res.get("unified_abc_fallback_mode", ""),
    }])

    phaseb_timebasis_fold_summary = cv_res.get("phaseb_timebasis_fold_summary", pd.DataFrame()).copy()
    phaseb_timebasis_candidates_raw = cv_res.get("phaseb_timebasis_candidates_raw", pd.DataFrame()).copy()
    phaseb_timebasis_candidate_scores = cv_res.get("phaseb_timebasis_candidate_scores", pd.DataFrame()).copy()
    phaseb_timebasis_selected_by_fold = cv_res.get("phaseb_timebasis_selected_by_fold", pd.DataFrame()).copy()
    phaseb_timebasis_final_selected = external_res.get("phaseb_timebasis_final_selected", pd.DataFrame()).copy()

    phasec_stgp_and_rule_universe_raw = cv_res.get("phasec_stgp_and_rule_universe_raw", pd.DataFrame()).copy()
    phasec_stgp_and_rule_universe_feasible = cv_res.get("phasec_stgp_and_rule_universe_feasible", pd.DataFrame()).copy()
    phasec_stgp_and_semantic_dropped = cv_res.get("phasec_stgp_and_semantic_dropped", pd.DataFrame()).copy()
    phasec_stgp_and_population_init = cv_res.get("phasec_stgp_and_population_init", pd.DataFrame()).copy()
    phasec_stgp_and_generation_log = cv_res.get("phasec_stgp_and_generation_log", pd.DataFrame()).copy()
    phasec_stgp_and_hof_by_fold = cv_res.get("phasec_stgp_and_hof_by_fold", pd.DataFrame()).copy()
    phasec_stgp_and_k_summary = cv_res.get("phasec_stgp_and_k_summary", cv_res.get("k_summary", pd.DataFrame())).copy()
    phasec_stgp_and_selected_by_fold = cv_res.get("phasec_stgp_and_selected_by_fold", pd.DataFrame()).copy()
    phasec_abc_unified_k_summary = cv_res.get("phasec_abc_unified_k_summary", phasec_stgp_and_k_summary).copy()
    phasec_abc_unified_selected_by_fold = cv_res.get("phasec_abc_unified_selected_by_fold", phasec_stgp_and_selected_by_fold).copy()
    phasec_stgp_and_final_selected = external_res.get("phasec_stgp_and_final_selected", pd.DataFrame()).copy()

    result_payload = {
        "summary": summary.to_dict(orient="records"),
        "baseline_stable_var_frequency": baseline_stable_freq_df.to_dict(orient="records"),
        "fold_results": fold_results.to_dict(orient="records"),
        "external_results": external_results.to_dict(orient="records"),
        "final_gp_expressions": final_gp_expressions.to_dict(orient="records"),
        "phaseB_summary": phaseB_summary.to_dict(orient="records"),
        "phaseC_summary": phaseC_summary.to_dict(orient="records"),
        "phasec_mode": "stgp_and",
        "phasec_pop_size": PHASEC_POP_SIZE,
        "phasec_n_gen": PHASEC_N_GEN,
        "phasec_selected_k": selected_k,
        "phasec_k_summary": phasec_stgp_and_k_summary.to_dict(orient="records"),
        "phasec_selected_candidates_by_fold": phasec_stgp_and_selected_by_fold.to_dict(orient="records"),
        "phasec_final_selected_candidates": phasec_stgp_and_final_selected.to_dict(orient="records"),
        "phasec_stgp_and_primitives": ["AND"],
        "phasec_stgp_and_or_enabled": bool(PHASEC_STGP_ALLOW_OR),
        "phasec_stgp_and_not_enabled": bool(PHASEC_STGP_ALLOW_NOT),
        "phasec_stgp_and_heredity": bool(PHASEC_STGP_HEREDITY),
        "phasec_stgp_and_initialization": "random_legal",
        "phasec_stgp_and_base_fit_once": True,
        "phasec_stgp_and_support_precomputed": True,
        "phasec_stgp_and_fitness_cache_enabled": True,
        "phasec_stgp_and_generation_log": phasec_stgp_and_generation_log.to_dict(orient="records"),
        "phasec_stgp_and_hof_by_fold": phasec_stgp_and_hof_by_fold.to_dict(orient="records"),
        "phasec_stgp_and_population_init": phasec_stgp_and_population_init.to_dict(orient="records"),
        "phasec_stgp_and_rule_universe_raw": phasec_stgp_and_rule_universe_raw.to_dict(orient="records"),
        "phasec_stgp_and_rule_universe_feasible": phasec_stgp_and_rule_universe_feasible.to_dict(orient="records"),
        "phasec_stgp_and_semantic_dropped": phasec_stgp_and_semantic_dropped.to_dict(orient="records"),
        "phasec_stgp_and_semantic_rules": {
            "allowed_rules": ["cat_binary_AND", "binary_binary_AND"],
            "disallowed_rules": ["cat_cat_AND", "same_factor_AND", "three_way_AND"],
            "max_terminals_per_rule": int(PHASEC_STGP_MAX_TERMINALS_PER_RULE),
        },
        "phasec_stgp_and_selected_k": selected_k,
        "phasec_stgp_and_k_summary": phasec_stgp_and_k_summary.to_dict(orient="records"),
        "phasec_abc_unified_k_summary": phasec_abc_unified_k_summary.to_dict(orient="records"),
        "phasec_abc_unified_selected_by_fold": phasec_abc_unified_selected_by_fold.to_dict(orient="records"),
        "selected_k_cat": selected_k,
        "internal_cv_abc_k_summary": phasec_abc_unified_k_summary.to_dict(orient="records"),
        "phaseB_selection_mode": PHASEB_SELECTION_MODE,
        "phaseB_mean_delta_min": PHASEB_MEAN_DELTA_MIN,
        "phaseB_positive_folds_min": PHASEB_POSITIVE_FOLDS_MIN,
        "phaseB_complexity_lambda": PHASEB_COMPLEXITY_LAMBDA,
        "timebasis_complexity_lambda": TIMEBASIS_COMPLEXITY_LAMBDA,
        "phaseB_candidate_n": phaseB_candidate_n,
        "phaseB_selected_n": phaseB_selected_n,
        "phaseB_allowed": bool(phaseB_allowed_final),
        "phaseB_mean_delta": phaseB_oof_delta,
        "phaseB_positive_folds": phaseB_positive_folds,
        "phaseB_selected_terms": phaseB_selected_terms,
        "phaseB_gate_reason": phaseB_gate_reason,
        "A_internal_cindex": baseline_stable_main_gp_oof,
        "AB_internal_cindex": baseline_stable_main_gp_phaseB_oof,
        "delta_B": phaseB_oof_delta,
        "phaseC_selection_mode": PHASEC_SELECTION_MODE,
        "phaseC_k_profile": PHASEC_K_PROFILE,
        "phaseC_k_values": resolve_phasec_k_values(PHASEC_K_PROFILE, PHASEC_K_VALUES),
        "phaseC_mean_delta_min": PHASEC_MEAN_DELTA_MIN,
        "phaseC_positive_folds_min": PHASEC_POSITIVE_FOLDS_MIN,
        "phaseC_complexity_lambda": PHASEC_SELECTION_COMPLEXITY_LAMBDA,
        "phaseC_algorithm_fitness_complexity_lambda": PHASEC_COMPLEXITY_LAMBDA,
        "phaseC_candidate_rule_n": phaseC_candidate_rule_n,
        "phaseC_selected_rule_n": phaseC_selected_rule_n,
        "phaseC_k_table": phasec_abc_unified_k_summary.rename(columns={"K_interaction": "k", "mean_abc": "mean_cindex"}).to_dict(orient="records"),
        "phaseC_mean_delta": pr.get("mean_delta", None),
        "phaseC_positive_folds": pr.get("positive_folds", None),
        "phaseC_gate_reason": phaseC_gate_reason,
        "phaseC_selected_rules": phasec_stgp_and_final_selected.to_dict(orient="records"),
        "AC_internal_cindex": pr.get("mean_abc", None),
        "ABC_internal_cindex": locals().get("oof_abc_cindex", pr.get("mean_abc", np.nan)),
        "delta_C_static": pr.get("mean_delta", None),
        "delta_C_afterB": pr.get("mean_delta", None),
        "legacy_template_name": "newcox+gpab_route1_phaseB_timebasis_phaseC_STGP_AND",
        "template_role": "legacy_single_endpoint_gpcox_template",
        "algorithm_logic_changed": False,
        "phaseB_phaseC_selection_style_aligned": True,
        "internal_cv_selection_note": "Legacy internal CV may use fold-validation performance for phase gating/selection depending on the original notebook logic; this task documents but does not change that behavior.",
        "external_validation_note": "MIMIC external validation is separate from eICU training in the notebook. If MIMIC C-index is later used for profile/Optuna selection, that downstream run should be labeled external-guided sensitivity optimization rather than pristine external validation.",
        "train_data_path": str(TRAIN_DATA_PATH),
        "external_data_path": str(EXTERNAL_VALID_DATA_PATH),
        "endpoint": globals().get("ENDPOINT", globals().get("TARGET_NAME", None)),
        "random_seed": int(SEED),
        "cv_fold_n": int(OUTER_FOLDS),
        "phaseB_logic_preserved": True,
        "phaseC_logic_preserved": True,
        "external_A_static_cindex": external_res.get("A_static_cindex", np.nan),
        "external_AB_timevarying_cindex": external_res.get("AB_timevarying_cindex", np.nan),
        "external_AC_static_cindex": external_res.get("AC_static_cindex", np.nan),
        "external_AC_static_cindex_optional": external_res.get("AC_static_cindex_optional", np.nan),
        "external_ABC_timevarying_cindex": external_res.get("ABC_timevarying_cindex", np.nan),
        "phaseB_active_final": external_res.get("phaseB_active_final", np.nan),
        "phaseC_active_final": external_res.get("phaseC_active_final", np.nan),
        "unified_abc_fallback_mode": external_res.get("unified_abc_fallback_mode", ""),
        "final_model": "ABC_timevarying",
        "phaseB_selected_candidates": external_res.get("phaseB_selected_candidates", []),
        "phaseC_selected_candidates": external_res.get("phaseC_selected_candidates", []),
        "phasec_stgp_and_selected_candidates_by_fold": phasec_stgp_and_selected_by_fold.to_dict(orient="records"),
        "phasec_stgp_and_final_selected_candidates": phasec_stgp_and_final_selected.to_dict(orient="records"),
        "phaseb_timebasis_enabled": str(globals().get("PHASEB_MODE", "time_basis_pool")).lower() == "time_basis_pool",
        "phaseb_timebasis_allowed_by_gate": bool(phaseB_allowed_final),
        "phaseb_timebasis_mean_delta": phaseB_oof_delta,
        "phaseb_timebasis_positive_folds": phaseB_positive_folds,
        "phaseb_timebasis_selected_candidates_by_fold": phaseb_timebasis_selected_by_fold.to_dict(orient="records"),
        "phaseb_timebasis_final_selected_candidates": phaseb_timebasis_final_selected.to_dict(orient="records"),
        "phaseb_timebasis_fold_summary": phaseb_timebasis_fold_summary.to_dict(orient="records"),
        "phaseb_timebasis_candidates_raw": phaseb_timebasis_candidates_raw.to_dict(orient="records"),
        "phaseb_timebasis_candidate_scores": phaseb_timebasis_candidate_scores.to_dict(orient="records"),
    }

    Path(OUT_XLSX).parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name="summary", index=False)
        baseline_stable_freq_df.to_excel(writer, sheet_name="baseline_stable_vars", index=False)
        fold_results.to_excel(writer, sheet_name="fold_results", index=False)
        external_results.to_excel(writer, sheet_name="external_results", index=False)
        final_gp_expressions.to_excel(writer, sheet_name="final_gp_expressions", index=False)
        phaseB_summary.to_excel(writer, sheet_name="phaseB_summary", index=False)
        phaseC_summary.to_excel(writer, sheet_name="phaseC_summary", index=False)
        phaseb_timebasis_fold_summary.to_excel(writer, sheet_name="phaseb_timebasis_fold_summary", index=False)
        phaseb_timebasis_candidates_raw.to_excel(writer, sheet_name="phaseb_timebasis_candidates_raw", index=False)
        phaseb_timebasis_candidate_scores.to_excel(writer, sheet_name="phaseb_timebasis_candidate_scores", index=False)
        phaseb_timebasis_selected_by_fold.to_excel(writer, sheet_name="phaseb_timebasis_selected_by_fold", index=False)
        phaseb_timebasis_final_selected.to_excel(writer, sheet_name="phaseb_timebasis_final_selected", index=False)
        phasec_stgp_and_rule_universe_raw.to_excel(writer, sheet_name="phasec_stgp_and_rule_universe_raw", index=False)
        phasec_stgp_and_rule_universe_feasible.to_excel(writer, sheet_name="phasec_stgp_and_rule_universe_feasible", index=False)
        phasec_stgp_and_semantic_dropped.to_excel(writer, sheet_name="phasec_stgp_and_semantic_dropped", index=False)
        phasec_stgp_and_population_init.to_excel(writer, sheet_name="phasec_stgp_and_population_init", index=False)
        phasec_stgp_and_generation_log.to_excel(writer, sheet_name="phasec_stgp_and_generation_log", index=False)
        phasec_stgp_and_hof_by_fold.to_excel(writer, sheet_name="phasec_stgp_and_hof_by_fold", index=False)
        phasec_stgp_and_k_summary.to_excel(writer, sheet_name="phasec_stgp_and_k_summary", index=False)
        phasec_stgp_and_selected_by_fold.to_excel(writer, sheet_name="phasec_stgp_and_selected_by_fold", index=False)
        phasec_abc_unified_k_summary.to_excel(writer, sheet_name="phasec_abc_unified_k_summary", index=False)
        phasec_abc_unified_selected_by_fold.to_excel(writer, sheet_name="phasec_abc_unified_selected_by_fold", index=False)
        external_results.to_excel(writer, sheet_name="final_external", index=False)
        phasec_stgp_and_final_selected.to_excel(writer, sheet_name="phasec_stgp_and_final_selected", index=False)
    Path(RESULT_JSON).parent.mkdir(parents=True, exist_ok=True)
    Path(RESULT_JSON).write_text(json.dumps(_jsonable(result_payload), ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"Saved Excel: {OUT_XLSX}")
    print(f"Saved JSON: {RESULT_JSON}")
    print(f"Saved Figure: {OUT_PNG_PHASEA_4FOLD}")
    print(f"Final feature set mode: {final_feature_set_mode}")
    print(f"Final AB model type: {final_ab_model_type}")
    print(f"Final AB_timevarying external C-index: {external_final_ab:.4f}")
    print(f"External ABC_timevarying C-index: {external_abc:.4f}")
    return result_payload

algorithm_result = run_once()
